# 可再生能源制化学品优化模型集合

本notebook包含了多个基于Gurobi优化的可再生能源制化学品系统模型，主要涵盖：

## 模型概述
1. **FT液体燃料生产系统** - 风光发电+电解制氢+FT合成+DAC捕碳的集成系统
2. **甲醇生产系统** - 风光发电+电解制氢+甲醇合成+DAC捕碳的集成系统  
3. **合成氨生产系统** - 风光发电+电解制氢+合成氨的集成系统

## 技术特点
- 区分AE和PEM电解槽效率和成本
- 拆分电池功率和储能容量成本
- 支持多种时间聚合模式（小时/日/周/月/年）
- 包含完整的储能系统（电池、氢气、产品储存）
- 考虑设备爬坡约束和最小负荷约束

## 数据源
- 风光资源数据：内蒙古2023年风光容量因子
- 技术经济参数：基于DEA 2024、IRENA等权威机构数据

# 全局参数配置

## 参数说明
本部分定义了所有优化模型共用的技术经济参数，包括：
- 可再生能源技术参数（风电、光伏）
- 储能技术参数（锂电池）
- 电解槽技术参数（AE、PEM）
- 化工合成技术参数（FT、甲醇、合成氨）
- DAC和储存系统参数
- 经济参数（折现率、年化成本）

所有参数基于权威机构数据（DEA 2024、IRENA等），技术水平对应2040年预期成本。

In [ ]:
# =============================================================================
# 全局参数配置 - 所有优化模型共用参数
# =============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB
import os

# Gurobi许可证配置
os.environ["GRB_WLSACCESSID"] = "e53ba8c8-d918-4dbc-8b70-719fdccb385c"
os.environ["GRB_WLSSECRET"] = "0ee54d8d-f1c2-4642-b8de-f37edffd7059"
os.environ["GRB_LICENSEID"] = "2570583"

# =============================================================================
# 数据加载
# =============================================================================
wind_solar_data = pd.read_csv('CF_NM_2023.csv')
pv_output   = wind_solar_data.iloc[:, 0].values  # 光伏容量因子
wind_output = wind_solar_data.iloc[:, 1].values  # 风电容量因子

# =============================================================================
# 1. 可再生能源技术参数
# =============================================================================
# 风电参数
wind_cost = 800            # 风电安装成本 ($/kW) Source: IRENA Renewable Power Generation Costs in 2023 Figure 2.1 for 5th Global
#wind_cost = 600            # 风电安装成本 ($/kW) Source: http://cnste.org/uploads/2025/0614/a2f95d8c6b7b7a0e397cccb4a7f24a25.pdf
wind_lifetime = 30         # 风电设备寿命 (年) Source: DEA 2024 technology_data_for_el_and_dh 20 Onshore turbines for 2035 estimated
wind_om_rate = 0.03        # 运维成本比例 Source: DEA 2024 technology_data_for_el_and_dh 20 Onshore turbines

# 光伏参数
pv_cost = 350              # 光伏安装成本 ($/kW) Source: DEA 2024 technology_data_for_el_and_dh 22 Utility-scale PV for 2035 estimated
#pv_cost = 480              # 光伏安装成本 ($/kW) Source: http://cnste.org/uploads/2025/0614/a2f95d8c6b7b7a0e397cccb4a7f24a25.pdf
pv_lifetime = 40           # 光伏设备寿命 (年) Source: DEA 2024 technology_data_for_el_and_dh 22 Utility-scale PV for 2035 estimated
pv_om_rate = 0.03          # 运维成本比例 Source: DEA 2024 technology_data_for_el_and_dh 22 Utility-scale PV

# 计算风光运维成本
wind_om = wind_cost * wind_om_rate
pv_om = pv_cost * pv_om_rate

# =============================================================================
# 2. 储能技术参数
# =============================================================================
# 锂电池参数（功率/容量分离建模）
battery_energy_cost = 100         # 储能安装成本 ($/kW) Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery 2040
battery_storage_cost = 100        # 储能安装成本 ($/kWh) Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery 2040
#battery_energy_cost = 110         # 储能安装成本 ($/kW) China Source: https://finance.sina.com.cn/roll/2025-01-15/doc-ineeznmf7350490.shtml
#battery_storage_cost = 110        # 储能安装成本 ($/kWh) China Source: https://finance.sina.com.cn/roll/2025-01-15/doc-ineeznmf7350490.shtml
#battery_energy_cost = 200         # 储能安装成本 ($/kW) Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery 2030
#battery_storage_cost = 200        # 储能安装成本 ($/kWh) Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery 2030
#battery_energy_cost = 50         # 储能安装成本 ($/kW) Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery 2050
#battery_storage_cost = 50        # 储能安装成本 ($/kWh) Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery 2050

battery_charge_penalty = 0.002    # ($/kWh) Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery
battery_discharge_penalty = 0.000 # $/kWh
battery_charge_efficiency = 0.985 # Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery
battery_discharge_efficiency = 0.975  # Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery
battery_lifetime = 20              # Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery for 2020
battery_energy_om = battery_energy_cost * 0.03
battery_storage_om = battery_storage_cost * 0.03

# =============================================================================
# 3. 电解槽技术参数
# =============================================================================
# AE电解槽参数
electrolyzer_cost_AE = 350         # ($/kW) Source: DEA 2024 data_sheets_for_renewable_fuels 1.1 AEC 1 GW for 2040 lower
#electrolyzer_cost_AE = 430         # ($/kW) Source: https://apps.epri.com/lcri-electrolysis-tea/en/capex-rates.html
#electrolyzer_cost_AE = 250         # ($/kW) Source: 实际项目 in China and DEA 2024 data_sheets_for_renewable_fuels 1.1 AEC 1 GW for 2050 lower

# PEM电解槽参数
electrolyzer_cost_PEM = 500        # ($/kW) Source: DEA 2024 data_sheets_for_renewable_fuels 1.1 PEMEC 1 GW for 2040 Central
#electrolyzer_cost_PEM = 350        # ($/kW) Source: DEA 2024 data_sheets_for_renewable_fuels 1.1 PEMEC 1 GW for 2050 Central
#electrolyzer_cost_PEM = 650        # ($/kW) Source: DEA 2024 data_sheets_for_renewable_fuels 1.1 PEMEC 1 GW for 2030 Central

electrolyzer_om_AE = electrolyzer_cost_AE * 0.03
electrolyzer_om_PEM = electrolyzer_cost_PEM * 0.03

# 区分电解效率
electrolyzer_eff_AE = 0.65         # Source: DEA 2024 data_sheets_for_renewable_fuels 1.1 AEC 1 GW for 2040 estimated
electrolyzer_eff_PEM = 0.70        # Source: IEA 2024 Global Hydrogen Review: Assumptions annex

electrolyzer_lifetime_AE = 25      # (年) Source: DEA 2024 data_sheets_for_renewable_fuels 1.1 AEC 1 GW for 2040 estimated
#electrolyzer_lifetime_AE = 15      # (年) Source: 目前中国实际项目基本上只有15年
electrolyzer_lifetime_PEM = 25     # (年) Source: DEA 2024 data_sheets_for_renewable_fuels 1.1 PEMEC 1 GW for 2040 estimated
#electrolyzer_lifetime_PEM = 15     # (年) Source: 目前中国实际项目基本上只有15年

# 氢气转换系数
conversion_factor_AE = electrolyzer_eff_AE * 3.6 / 120    # kW -> kg H2/h
conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120  # kW -> kg H2/h

# =============================================================================
# 4. 氢气储存参数
# =============================================================================
hydrogen_storage_cost       = 500    # $/kgH2 NEED TO CHECK:Source: D. D. Papadias and R. K. Ahluwalia, Int J Hydrogen Energy, 2021, 46, 34527–34541, and DEA 2024 technology_datasheet_for_energy_storage 151a Hydrogen Storage - Tanks for 2040 estimated
#hydrogen_storage_cost       = 900    # Current $/kgH2 NEED TO CHECK:Source: D. D. Papadias and R. K. Ahluwalia, Int J Hydrogen Energy, 2021, 46, 34527–34541, and DEA 2024 technology_datasheet_for_energy_storage 151a Hydrogen Storage - Tanks for 2030 estimated
#hydrogen_storage_cost       = 300    # 2050 $/kgH2 NEED TO CHECK:Source: D. D. Papadias and R. K. Ahluwalia, Int J Hydrogen Energy, 2021, 46, 34527–34541, and DEA 2024 technology_datasheet_for_energy_storage 151a Hydrogen Storage - Tanks for 2050 estimated

hydrogen_charge_penalty = 0.00001 # $/kgH2
hydrogen_storage_efficiency = 1.0 # Source: Fasihi EES 2024
hydrogen_storage_elec = 0          # kWh/kg H2（充氢所需电耗）
hydrogen_storage_lifetime = 30    # Source: Fasihi EES 2024

# =============================================================================
# 5. FT液体燃料参数
# =============================================================================
# FT合成参数
#FTL_synthesis_cost = 15400        # ($/kg·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.6 Hydrogen to Jet for 2040 estimated
FTL_synthesis_cost = 9400          # Low ($/kg·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.6 Hydrogen to Jet for 2050 Low estimated
#FTL_synthesis_cost = 22000        # High ($/kg·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.6 Hydrogen to Jet for 2030 estimated

FTL_synthesis_om = FTL_synthesis_cost * 0.06  # Source: DEA 2024 data_sheets_for_renewable_fuels 5.6 Hydrogen to Jet for 2040 estimated
FTL_synthesis_lifetime = 25        # Source: DEA 2024 data_sheets_for_renewable_fuels 5.6 Hydrogen to Jet for 2040 estimated
target_FTL_production = 10 * 1000  # kg/年，即10吨/年

FTL_hydrogen_ratio = 0.49          # kg H2/kg FTL Source: DEA 2024 data_sheets_for_renewable_fuels 5.6 Hydrogen to Jet for 2040 estimated
#FTL_hydrogen_ratio = 0.51         # kg H2/kg FTL Source: DEA 2024 data_sheets_for_renewable_fuels 5.6 Hydrogen to Jet for average 2030 estimated
#FTL_hydrogen_ratio = 0.48         # kg H2/kg FTL Source: DEA 2024 data_sheets_for_renewable_fuels 5.6 Hydrogen to Jet for average 2050 estimated

FTL_co2_ratio = 3.6               # kg CO2/kg FTL Source: DEA 2024 data_sheets_for_renewable_fuels 5.6 Hydrogen to Jet for 2040 estimated
#FTL_co2_ratio = 3.9               # High kg CO2/kg FTL Source: DEA 2024 data_sheets_for_renewable_fuels 5.6 Hydrogen to Jet for 2030 estimated
#FTL_co2_ratio = 3.3               # Low kg CO2/kg FTL Source: DEA 2024 data_sheets_for_renewable_fuels 5.6 Hydrogen to Jet for 2050 estimated

FTL_electricity_ratio = 0.3       # kWh/kg FTL Source: DEA 2024 data_sheets_for_renewable_fuels 5.6 Hydrogen to Jet for 2040 estimated
target_hydrogen_for_FTL = target_FTL_production * FTL_hydrogen_ratio

# FT储存参数
FTL_storage_cost_param = 0.056     # ($/kg) Source: Fasihi EES 2024 Table S4. 2040
FTL_storage_om = FTL_storage_cost_param * 0.03
FTL_storage_lifetime = 30          # (年) Source: Fasihi EES 2024 Table S4. 2040
FTL_charge_penalty = 0.00001       # $/kg

# 每小时FT需求（均匀分布假设）
hourly_FTL_demand = target_FTL_production / 8760

# =============================================================================
# 6. 甲醇合成参数
# =============================================================================
# 甲醇合成参数
methanol_synthesis_cost = 5700     # ($/kg·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.2 Methanol from hydrogen for 2040 estimated
#methanol_synthesis_cost = 2600    # Low ($/kg·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.2 Methanol from hydrogen for 2050 Low
#methanol_synthesis_cost = 5200    # 2050 ($/kg·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.2 Methanol from hydrogen for 2050 central
#methanol_synthesis_cost = 8000    # High ($/kg·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.2 Methanol from hydrogen for 2025 estimated
#methanol_synthesis_cost = 6500    # 2030 ($/kg·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.2 Methanol from hydrogen for 2030 estimated

methanol_synthesis_om = methanol_synthesis_cost * 0.03
methanol_synthesis_lifetime = 30   # Source: Fasihi EES 2024
target_methanol_production = 10 * 1000  # kg/年，即10吨/年
methanol_hydrogen_ratio = 0.19     # kg H2/kg CH3OH Source: DEA 2024 data_sheets_for_renewable_fuels 5.2 Methanol from hydrogen for 2040 estimated
methanol_co2_ratio = 1.4           # kg CO2/kg CH3OH Source: DEA 2024 data_sheets_for_renewable_fuels 5.2 Methanol from hydrogen for 2040 estimated
methanol_electricity_ratio = 0.2   # kWh/kg CH3OH Source: DEA 2024 data_sheets_for_renewable_fuels 5.2 Methanol from hydrogen for 2040 estimated
target_hydrogen_for_methanol = target_methanol_production * methanol_hydrogen_ratio

# 甲醇储存参数
methanol_storage_cost_param = 0.056  # ($/kg) Source: Fasihi EES 2024 Table S4. 2040
methanol_storage_om = methanol_storage_cost_param * 0.03
methanol_storage_lifetime = 30      # (年) Source: Fasihi EES 2024 Table S4. 2040
methanol_charge_penalty = 0.00001   # $/kg

# 每小时甲醇需求（均匀分布假设）
hourly_methanol_demand = target_methanol_production / 8760

# =============================================================================
# 7. 合成氨参数
# =============================================================================
# 合成氨参数
ammonia_synthesis_cost = 6500      # $/(kgNH3·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.4 Hydrogen to Ammonia for 2040 estimated
#ammonia_synthesis_cost = 3500     # Low $/(kgNH3·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.4 Hydrogen to Ammonia for 2050 Low estimated
#ammonia_synthesis_cost = 5000     # 2050 $/(kgNH3·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.4 Hydrogen to Ammonia for 2050 central estimated
#ammonia_synthesis_cost = 9500     # High $/(kgNH3·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.4 Hydrogen to Ammonia for 2020 estimated
#ammonia_synthesis_cost = 8000     # 2030 $/(kgNH3·h) Source: DEA 2024 data_sheets_for_renewable_fuels 5.4 Hydrogen to Ammonia for 2030 estimated

ammonia_synthesis_om = ammonia_synthesis_cost * 0.03
ammonia_synthesis_lifetime = 30    # Source: DEA 2024 data_sheets_for_renewable_fuels 5.4 Hydrogen to Ammonia for 2040 estimated
target_ammonia_production = 10e3   # kg/年
ammonia_hydrogen_ratio = 0.18      # kg H2/kg NH3 Source: DEA 2024 data_sheets_for_renewable_fuels 5.4 Hydrogen to Ammonia for 2040 estimated
ammonia_electricity_ratio = 0.315  # kWh/kg NH3 Source: DEA 2024 data_sheets_for_renewable_fuels 5.4 Hydrogen to Ammonia for 2040 estimated

# 合成氨储存参数
ammonia_storage_cost_param = 0.59  # $/kgNH3 Source: M. Fasihi, R. Weiss, J. Savolainen and C. Breyer, Appl Energy, 2021, 294, 116170
ammonia_storage_om = ammonia_storage_cost_param * 0.03
ammonia_storage_lifetime = 30      # Source: M. Fasihi, R. Weiss, J. Savolainen and C. Breyer, Appl Energy, 2021, 294, 116170

# 每小时合成氨需求（均匀分布假设）
hourly_ammonia_demand = target_ammonia_production / 8760

# =============================================================================
# 8. DAC系统参数
# =============================================================================
dac_cost = 4000                    # ($/(kgCO2/h)) Source: Fasihi EES 2024 Table S4. 2040
#dac_cost = 3000                   # Low and 2050 ($/(kgCO2/h)) Source: Fasihi EES 2024 Table S4. 2050
#dac_cost = 7800                   # High and 2030($/(kgCO2/h)) Source: Fasihi EES 2024 Table S4. 2030

dac_om = dac_cost * 0.03
dac_lifetime = 30                  # Source: Fasihi EES 2024
dac_elec_per_kg = 0.25             # 2040 kWh/kgCO2 Source: Fasihi EES 2024 Table S4. 2040
#dac_elec_per_kg = 0.31             # 2030 kWh/kgCO2 Source: Fasihi EES 2024 Table S4. 2030
#dac_elec_per_kg = 0.22             # 2050 kWh/kgCO2 Source: Fasihi EES 2024 Table S4. 2050

dac_heat_per_kg = 1.0              # kWh/kgCO2 Source: Fasihi EES 2024 Table S4. 2040 and 2050
#dac_heat_per_kg = 1.3             # kWh/kgCO2 Source: Fasihi EES 2024 Table S4. 2030

sorbent_cost = 0.012               # $/kgCO2 Source: Fasihi EES 2024 Table S4. 2040
#sorbent_cost = 0.020               # $/kgCO2 Source: Fasihi EES 2024 Table S4. 2030
#sorbent_cost = 0.009               # $/kgCO2 Source: Fasihi EES 2024 Table S4. 2050

# =============================================================================
# 9. CO2处理系统参数
# =============================================================================
# CO2压缩与储存参数
co2_compressor_cost = 365          # ($/(kgCO2/h)) Source: Fasihi EES 2024 Table S4. 2040 and G. Towler and R. Sinnott, Chemical Engineering Design: Principles, Practice and Economics of Plant and Process Design, Butterworth-Heinemann, Oxford, 3rd Edition., 2021
co2_compressor_om = co2_compressor_cost * 0.03
co2_compressor_lifetime = 20       # Source: Fasihi EES 2024 Table S4. 2040
co2_compression_energy = 0.1       # kWh/kg CO2 Source: Fasihi EES 2024 Table S4. 2040 and Aspen Technology, Aspen Plus, Aspen Technology, Inc., Bedford, Massachusetts, 2024

# Source: Sweco Finland Holding Oy, Cost Estimates for Green Field Power to Food Plant Utilities and Storage Systems - internal report for Solar Foods Oy, 2019
co2_storage_cost = 24              # ($/kg CO2储存容量) Source: Fasihi EES 2024 Table S4. 2040
co2_storage_om = co2_storage_cost * 0.03
co2_storage_lifetime = 30          # Source: Fasihi EES 2024 Table S4. 2040

# =============================================================================
# 10. 热管理系统参数
# =============================================================================
# 热泵与热储参数
heat_pump_cost = 950               # ($/kW_th) Source: 2040 DEA 2024 technology_data_for_industrial_process_heat_0 302.b High temp. hp Up to 150 for 2040 estimated
#heat_pump_cost = 1000             # ($/kW_th) Source: 2030 DEA 2024 technology_data_for_industrial_process_heat_0 302.b High temp. hp Up to 150 for 2030 estimated
#heat_pump_cost = 900             # ($/kW_th) Source: 2050 DEA 2024 technology_data_for_industrial_process_heat_0 302.b High temp. hp Up to 150 for 2050 estimated

heat_pump_om = heat_pump_cost * 0.03
heat_pump_cop = 3.2                # COP Source: DEA 2024 technology_data_for_industrial_process_heat_0 302.b High temp. hp Up to 150 for 2040 estimated
heat_pump_lifetime = 20            # Source: DEA 2024 technology_data_for_industrial_process_heat_0 302.b High temp. hp Up to 150 for 2040 estimated
thermal_storage_cost = 30          # ($/kWh_th) Source: A. C. Christidis, PhD Dissertation, Technical University of Berlin, 2019
thermal_storage_om = thermal_storage_cost * 0.03
thermal_storage_efficiency = 0.98  # Source: Fasihi EES 2024
thermal_storage_lifetime = 30      # Source: Fasihi EES 2024

# =============================================================================
# 11. 系统运行参数
# =============================================================================
# 逐时爬坡上限
Flex_up = 0.02
Flex_mid = 0.1
Flex_down = 0.2
surplus_penalty = 0.00001

# 电力市场参数
electricity_cost = 0.08            # 电价 ($/kWh)
elec_demand_penalty = -electricity_cost  # 向外输电收益

# 供需平衡时间聚合尺度："hourly"、"3hourly"、"daily"、"weekly"、"monthly"、"annual"
aggregation_mode = "daily"

# 时间参数
time_steps = 8760
start_index = 168                  # 避开初始不稳定阶段

# =============================================================================
# 12. 经济参数
# =============================================================================
discount_rate = 0.07

# 年化成本计算函数
def calculate_annualized_cost(capex, lifetime, om_cost):
    """计算年化成本"""
    return capex * discount_rate / (1 - (1 + discount_rate)**-lifetime) + om_cost

# 计算所有设备的年化成本
wind_annual_cost = calculate_annualized_cost(wind_cost, wind_lifetime, wind_om)
pv_annual_cost = calculate_annualized_cost(pv_cost, pv_lifetime, pv_om)
battery_energy_annual_cost = calculate_annualized_cost(battery_energy_cost, battery_lifetime, battery_energy_om)
battery_storage_annual_cost = calculate_annualized_cost(battery_storage_cost, battery_lifetime, battery_storage_om)
electrolyzer_annual_cost_AE = calculate_annualized_cost(electrolyzer_cost_AE, electrolyzer_lifetime_AE, electrolyzer_om_AE)
electrolyzer_annual_cost_PEM = calculate_annualized_cost(electrolyzer_cost_PEM, electrolyzer_lifetime_PEM, electrolyzer_om_PEM)
hydrogen_storage_annual_cost = calculate_annualized_cost(hydrogen_storage_cost, hydrogen_storage_lifetime, hydrogen_storage_om)

# FT液体燃料相关年化成本
FTL_synthesis_annual_cost = calculate_annualized_cost(FTL_synthesis_cost, FTL_synthesis_lifetime, FTL_synthesis_om)
FTL_storage_annual_cost = calculate_annualized_cost(FTL_storage_cost_param, FTL_storage_lifetime, FTL_storage_om)

# 甲醇相关年化成本
methanol_synthesis_annual_cost = calculate_annualized_cost(methanol_synthesis_cost, methanol_synthesis_lifetime, methanol_synthesis_om)
methanol_storage_annual_cost = calculate_annualized_cost(methanol_storage_cost_param, methanol_storage_lifetime, methanol_storage_om)

# 合成氨相关年化成本
ammonia_synthesis_annual_cost = calculate_annualized_cost(ammonia_synthesis_cost, ammonia_synthesis_lifetime, ammonia_synthesis_om)
ammonia_storage_annual_cost = calculate_annualized_cost(ammonia_storage_cost_param, ammonia_storage_lifetime, ammonia_storage_om)

# DAC和CO2处理年化成本
dac_annual_cost = calculate_annualized_cost(dac_cost, dac_lifetime, dac_om)
co2_compressor_annual_cost = calculate_annualized_cost(co2_compressor_cost, co2_compressor_lifetime, co2_compressor_om)
co2_storage_annual_cost = calculate_annualized_cost(co2_storage_cost, co2_storage_lifetime, co2_storage_om)

# 热管理系统年化成本
heat_pump_annual_cost = calculate_annualized_cost(heat_pump_cost, heat_pump_lifetime, heat_pump_om)
thermal_storage_annual_cost = calculate_annualized_cost(thermal_storage_cost, thermal_storage_lifetime, thermal_storage_om)

print("✅ 全局参数配置完成")
print(f"风电年化成本: ${wind_annual_cost:.2f}/kW")
print(f"光伏年化成本: ${pv_annual_cost:.2f}/kW") 
print(f"AE电解年化成本: ${electrolyzer_annual_cost_AE:.2f}/kW")
print(f"PEM电解年化成本: ${electrolyzer_annual_cost_PEM:.2f}/kW")
print(f"FT合成年化成本: ${FTL_synthesis_annual_cost:.2f}/(kg/h)")
print(f"甲醇合成年化成本: ${methanol_synthesis_annual_cost:.2f}/(kg/h)")
print(f"合成氨年化成本: ${ammonia_synthesis_annual_cost:.2f}/(kg/h)")
print(f"DAC年化成本: ${dac_annual_cost:.2f}/(kgCO2/h)")

# 1. FT液体燃料生产优化模型

## 1.1 模型概述
基于Fisher-Tropsch合成技术的液体燃料生产系统优化模型，集成以下组件：

### 主要设备
- **可再生能源发电**：风电、光伏
- **储能系统**：锂电池（拆分功率和容量）
- **制氢系统**：AE电解槽、PEM电解槽
- **FT合成装置**：氢气+CO2→液体燃料
- **DAC系统**：直接空气捕获CO2
- **热泵系统**：为DAC提供低温热源
- **储存系统**：氢气储存、CO2储存、产品储存

### 技术参数基准年：2040年

### 目标
最小化系统总成本，满足年产10吨FT液体燃料需求

In [ ]:
# =============================================================================
# FT Liquid + DAC 优化（拆分电池电力/储能，区分 AE/PEM 效率）
# =============================================================================

# =============================================================================
# 1. 导入库与数据加载
# =============================================================================
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB

# =============================================================================
# 1. 导入库与数据加载
# =============================================================================
wind_solar_data = pd.read_csv('CF_NM_2023.csv')
pv_output   = wind_solar_data.iloc[:, 0].values  # 假设第一列为 'pv'
wind_output = wind_solar_data.iloc[:, 1].values  # 假设第二列为 'wind'

# =============================================================================
# 2. 使用全局参数
# =============================================================================
# 所有参数已在全局配置单元格中定义，这里直接使用全局变量

# =============================================================================
# 3. 模型创建与变量定义
# =============================================================================

# 创建模型
model = Model("HydrogenFTLOptimization")
model.setParam('OutputFlag', 0)
model.setParam('TimeLimit', 1000)
model.setParam('MIPFocus', 1)

# 强制用 Barrier，跳过 Crossover，把收敛精度放宽到 1e-6 ，并行 8 线程
model.setParam('Method', 2)        # 2 = Barrier
model.setParam('Crossover', 0)     # 关闭回退
model.setParam('BarConvTol', 1e-6) # 默认 1e-8 → 1e-6
model.setParam('Threads', 1)

# 选用 Homogeneous KKT 因式分解策略，通常更稳定
model.setParam('BarHomogeneous', 1)

# 关闭过度 Presolve（有时 Aggressive 反而慢）
model.setParam('Presolve', 1)      # 1 = 默认

time_steps = 8760

# 3.1 装机容量决策变量
# —— 可再生发电、储能、工艺设备及CO2处理装置
wind_capacity = model.addVar(lb=0, name="wind_capacity")
pv_capacity   = model.addVar(lb=0, name="pv_capacity")
battery_energy_capacity = model.addVar(lb=0, name="battery_energy_capacity")
battery_storage_capacity = model.addVar(lb=0, name="battery_storage_capacity")
electrolyzer_capacity_AE = model.addVar(lb=0, name="electrolyzer_capacity_AE")
electrolyzer_capacity_PEM = model.addVar(lb=0, name="electrolyzer_capacity_PEM")
FTL_synthesis_capacity = model.addVar(lb=0, name="FTL_capacity")
heat_pump_capacity = model.addVar(lb=0, name="heat_pump_capacity")
thermal_storage_capacity = model.addVar(lb=0, name="thermal_storage_capacity")
FTL_storage_capacity = model.addVar(lb=0, name="FTL_storage_capacity")
hydrogen_storage_capacity = model.addVar(lb=0, name="hydrogen_storage_capacity")
dac_capacity = model.addVar(lb=0, name="dac_capacity")
co2_compressor_capacity = model.addVar(lb=0, name="co2_compressor_capacity")
co2_storage_capacity = model.addVar(lb=0, name="co2_storage_capacity")

# 3.2 时间步变量（每小时变量，共 time_steps 个）
# 电池相关变量
energy_balance = model.addVars(time_steps, lb=0, name="energy_balance")
battery_charge = model.addVars(time_steps, lb=0, name="battery_charge")
battery_discharge = model.addVars(time_steps, lb=0, name="battery_discharge")

# 电解槽产氢变量
electrolyzer_power_AE = model.addVars(time_steps, lb=0, name="electrolyzer_power_AE")
electrolyzer_power_PEM = model.addVars(time_steps, lb=0, name="electrolyzer_power_PEM")

# 甲醇合成与甲醇储存变量
FTL_synthesis_power = model.addVars(time_steps, lb=0, name="FTL_synthesis_power")
FTL_storage_balance = model.addVars(time_steps, lb=0, name="FTL_storage_balance")
FTL_storage_charge = model.addVars(time_steps, lb=0, name="FTL_storage_charge")
FTL_storage_discharge = model.addVars(time_steps, lb=0, name="FTL_storage_discharge")

# 氢气存储变量
hydrogen_storage_balance = model.addVars(time_steps, lb=0, name="hydrogen_storage_balance")
hydrogen_charge = model.addVars(time_steps, lb=0, name="hydrogen_charge")
hydrogen_discharge = model.addVars(time_steps, lb=0, name="hydrogen_discharge")

# DAC（CO2捕集）变量
dac_CO2 = model.addVars(time_steps, lb=0, name="dac_CO2")
dac_electricity = model.addVars(time_steps, lb=0, name="dac_electricity")
dac_heat_input = model.addVars(time_steps, lb=0, name="dac_heat_input")

# 热泵与热储变量
heat_pump_output = model.addVars(time_steps, lb=0, name="heat_pump_output")
thermal_charge = model.addVars(time_steps, lb=0, name="thermal_charge")
thermal_discharge = model.addVars(time_steps, lb=0, name="thermal_discharge")
thermal_storage_level = model.addVars(time_steps, lb=0, name="thermal_storage_level")

# CO2压缩及储存变量
co2_comp_elec = model.addVars(time_steps, lb=0, name="co2_comp_elec")
co2_storage_level = model.addVars(time_steps, lb=0, name="co2_storage_level")
co2_charge = model.addVars(time_steps, lb=0, name="co2_charge")
co2_discharge = model.addVars(time_steps, lb=0, name="co2_discharge")
direct_co2 = model.addVars(time_steps, lb=0, name="direct_co2")
stored_co2 = model.addVars(time_steps, lb=0, name="stored_co2")

# 3.3 其它变量（例如多余电量）
surplus = model.addVars(time_steps, lb=0, name="surplus")

# =============================================================================
# 4. 目标函数
# =============================================================================
model.setObjective(
    (wind_annual_cost * wind_capacity +
     pv_annual_cost * pv_capacity +
     battery_energy_annual_cost * battery_energy_capacity +
     battery_storage_annual_cost * battery_storage_capacity +
     electrolyzer_annual_cost_AE * electrolyzer_capacity_AE +
     electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM +
     FTL_synthesis_annual_cost * FTL_synthesis_capacity +
     heat_pump_annual_cost * heat_pump_capacity +
     thermal_storage_annual_cost * thermal_storage_capacity +
     hydrogen_storage_annual_cost * hydrogen_storage_capacity +
     FTL_storage_annual_cost * FTL_storage_capacity +
     dac_annual_cost * dac_capacity +
     co2_compressor_annual_cost * co2_compressor_capacity +
     co2_storage_annual_cost * co2_storage_capacity +
     surplus_penalty * gp.quicksum(surplus[t] for t in range(time_steps)) +
     battery_charge_penalty * gp.quicksum(battery_charge[t] for t in range(time_steps)) +
     battery_discharge_penalty * gp.quicksum(battery_discharge[t] for t in range(time_steps)) +
     sorbent_cost * gp.quicksum(dac_CO2[t] for t in range(time_steps))+
     hydrogen_charge_penalty * gp.quicksum(hydrogen_charge[t] for t in range(time_steps))+
     FTL_charge_penalty * gp.quicksum(FTL_storage_charge[t] for t in range(time_steps))
    ),
    GRB.MINIMIZE
)

# =============================================================================
# 5. 约束条件
# =============================================================================

# 5.1 逐时设备灵活性（爬坡）约束
for t in range(1, time_steps):
    # 电解槽 AE 爬坡限制
    model.addConstr(electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] <= Flex_down * electrolyzer_capacity_AE,
                    name=f"electrolyzer_AE_ramp_up_{t}")
    model.addConstr(electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] >= -Flex_down * electrolyzer_capacity_AE,
                    name=f"electrolyzer_AE_ramp_down_{t}")
    # 甲醇合成爬坡限制
    model.addConstr(FTL_synthesis_power[t] - FTL_synthesis_power[t-1] <= Flex_up * FTL_synthesis_capacity,
                    name=f"FTL_ramp_up_{t}")
    model.addConstr(FTL_synthesis_power[t] - FTL_synthesis_power[t-1] >= -Flex_down * FTL_synthesis_capacity,
                    name=f"FTL_ramp_down_{t}")
    # DAC爬坡限制
    model.addConstr(dac_CO2[t] - dac_CO2[t-1] <= Flex_up * dac_capacity,
                    name=f"dac_ramp_up_{t}")
    model.addConstr(dac_CO2[t] - dac_CO2[t-1] >= -Flex_down * dac_capacity,
                    name=f"dac_ramp_down_{t}")

# 5.2 甲醇供需平衡约束（含储存充放，支持多种聚合尺度）
start_index = 168  # 避开初始不稳定阶段
if aggregation_mode == "hourly":
    for t in range(start_index, time_steps):
        model.addConstr(
            FTL_synthesis_power[t] + FTL_storage_discharge[t] >=
            hourly_FTL_demand + FTL_storage_charge[t],
            name=f"FTL_balance_hourly_{t}"
        )
elif aggregation_mode == "3hourly":
    interval = 3
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(FTL_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(FTL_storage_discharge[t] for t in range(t_start, t_end))
            ==
            hourly_FTL_demand * interval +
            gp.quicksum(FTL_storage_charge[t] for t in range(t_start, t_end)),
            name=f"FTL_balance_3hourly_{i}"
        )
elif aggregation_mode == "daily":
    interval = 24
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(FTL_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(FTL_storage_discharge[t] for t in range(t_start, t_end))
            >=
            0.8 * hourly_FTL_demand * interval +
            gp.quicksum(FTL_storage_charge[t] for t in range(t_start, t_end)),
            name=f"FTL_balance_daily_{i}"
        )
elif aggregation_mode == "weekly":
    interval = 168
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(FTL_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(FTL_storage_discharge[t] for t in range(t_start, t_end))
            ==
            hourly_FTL_demand * interval +
            gp.quicksum(FTL_storage_charge[t] for t in range(t_start, t_end)),
            name=f"FTL_balance_weekly_{i}"
        )
elif aggregation_mode == "monthly":
    interval = 730
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(FTL_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(FTL_storage_discharge[t] for t in range(t_start, t_end))
            ==
            hourly_FTL_demand * interval +
            gp.quicksum(FTL_storage_charge[t] for t in range(t_start, t_end)),
            name=f"FTL_balance_monthly_{i}"
        )
elif aggregation_mode == "annual":
    model.addConstr(
        gp.quicksum(FTL_synthesis_power[t] for t in range(start_index, time_steps)) +
        gp.quicksum(FTL_storage_discharge[t] for t in range(start_index, time_steps))
        ==
        hourly_FTL_demand * (time_steps - start_index) +
        gp.quicksum(FTL_storage_charge[t] for t in range(start_index, time_steps)),
        name="FTL_balance_annual"
    )
else:
    raise ValueError("Unsupported aggregation_mode value!")

# 附加：年生产总量约束
model.addConstr(
    gp.quicksum(FTL_synthesis_power[t] for t in range(time_steps)) >= hourly_FTL_demand * 8760,
    name="annual_FTL_production"
)

# 5.3 甲醇储存动态与容量约束
model.addConstr(FTL_storage_balance[0] ==
                FTL_storage_charge[0] - FTL_storage_discharge[0],
                name="FTL_storage_balance_0")
for t in range(1, time_steps):
    model.addConstr(
        FTL_storage_balance[t] == FTL_storage_balance[t-1] + FTL_storage_charge[t] - FTL_storage_discharge[t],
        name=f"FTL_storage_balance_{t}"
    )
    model.addConstr(FTL_storage_balance[t] <= FTL_storage_capacity,
                    name=f"FTL_storage_capacity_limit_{t}")

# 5.4 能量平衡与储能动态约束
conversion_factor_AE  = electrolyzer_eff_AE  * 3.6 / 120
conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

for t in range(time_steps):
    # 计算当期风电与光伏发电
    wind_power = wind_output[t] * wind_capacity
    pv_power = pv_output[t] * pv_capacity
    
    # 甲醇储存：要求合成功率与储存放出至少满足充入量
    model.addConstr(FTL_synthesis_power[t] + FTL_storage_discharge[t] >=
                    FTL_storage_charge[t],
                    name=f"FTL_min_balance_{t}")
                    
    # 电池储能及充放电限额
    model.addConstr(energy_balance[t] <= battery_storage_capacity,
                    name=f"energy_balance_limit_{t}")
    model.addConstr(battery_charge[t] + battery_discharge[t] <= battery_energy_capacity,
                    name=f"battery_charge_discharge_limit_{t}")
    # 氢气存储上限
    model.addConstr(hydrogen_storage_balance[t] <= hydrogen_storage_capacity,
                    name=f"hydrogen_storage_capacity_limit_{t}")
    model.addConstr(hydrogen_charge[t] + hydrogen_discharge[t] <= hydrogen_storage_capacity,
                    name=f"hydrogen_charge_limit_{t}")
    
    # 电力平衡（含电解槽、甲醇合成、储能、热泵、DAC与CO2压缩电耗等）
    model.addConstr(
        wind_power + pv_power + battery_discharge[t] ==
        (electrolyzer_power_AE[t] + electrolyzer_power_PEM[t]) +
        FTL_synthesis_power[t] * FTL_electricity_ratio +
        battery_charge[t] + hydrogen_charge[t] * hydrogen_storage_elec +
        (heat_pump_output[t] / heat_pump_cop) + dac_electricity[t] + surplus[t] + co2_comp_elec[t],
        name=f"power_balance_{t}"
    )
    # 电池动态平衡
    if t == 0:
        model.addConstr(
            energy_balance[t] == battery_charge[t] * battery_charge_efficiency
                                - battery_discharge[t] / battery_discharge_efficiency,
            name=f"battery_balance_{t}" )
    else:
        model.addConstr(
            energy_balance[t] == energy_balance[t-1] +
                                battery_charge[t] * battery_charge_efficiency
                              - battery_discharge[t] / battery_discharge_efficiency,
            name=f"battery_balance_{t}" )
    # 氢气存储动态平衡
    if t == 0:
        model.addConstr(
            hydrogen_storage_balance[t] == hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
            name=f"hydrogen_balance_{t}"
        )
    else:
        model.addConstr(
            hydrogen_storage_balance[t] == hydrogen_storage_balance[t-1] + hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
            name=f"hydrogen_balance_{t}"
        )
    # 氢气供需平衡：电解槽产氢需满足甲醇合成及储氢要求
    model.addConstr(
       	electrolyzer_power_AE[t] * conversion_factor_AE +
        electrolyzer_power_PEM[t] * conversion_factor_PEM + hydrogen_discharge[t] >=
        FTL_synthesis_power[t] * FTL_hydrogen_ratio + hydrogen_charge[t],
        name=f"hydrogen_PD_balance_{t}"
    )
    # DAC部分：热、电耗与捕集CO2量关系
    model.addConstr(dac_heat_input[t] == dac_CO2[t] * dac_heat_per_kg,
                    name=f"dac_heat_consumption_{t}")
    model.addConstr(dac_electricity[t] == dac_CO2[t] * dac_elec_per_kg,
                    name=f"dac_electricity_consumption_{t}")
    model.addConstr(dac_CO2[t] <= dac_capacity,
                    name=f"dac_capacity_limit_{t}")
    
    # CO2分流：直接供给与进入储存
    model.addConstr(direct_co2[t] + stored_co2[t] == dac_CO2[t],
                    name=f"co2_split_{t}")
    model.addConstr(stored_co2[t] <= co2_compressor_capacity,
                    name=f"co2_compressor_capacity_limit_{t}")
    model.addConstr(co2_comp_elec[t] == stored_co2[t] * co2_compression_energy,
                    name=f"co2_compression_energy_{t}")
    model.addConstr(co2_charge[t] == stored_co2[t],
                    name=f"co2_charge_relation_{t}")
    
    # 热泵与热储热平衡
    model.addConstr(
        heat_pump_output[t] - thermal_charge[t] + thermal_discharge[t] >= dac_heat_input[t],
        name=f"heat_supply_balance_{t}"
    )
    # CO2储存动态及容量约束
    if t == 0:
        model.addConstr(
            co2_storage_level[t] == co2_charge[t] - co2_discharge[t],
            name="co2_storage_balance_0"
        )
    else:
        model.addConstr(
            co2_storage_level[t] == co2_storage_level[t-1] + co2_charge[t] - co2_discharge[t],
            name=f"co2_storage_balance_{t}"
        )
    model.addConstr(co2_storage_level[t] <= co2_storage_capacity,
                    name=f"co2_storage_level_limit_{t}")
    
    # 储热系统动态及容量约束
    if t == 0:
        model.addConstr(
            thermal_storage_level[t] == thermal_charge[t] * thermal_storage_efficiency - thermal_discharge[t] / thermal_storage_efficiency,
            name="thermal_storage_balance_0"
        )
    else:
        model.addConstr(
            thermal_storage_level[t] == thermal_storage_level[t-1] + thermal_charge[t] * thermal_storage_efficiency - thermal_discharge[t] / thermal_storage_efficiency,
            name=f"thermal_storage_balance_{t}"
        )
    model.addConstr(thermal_storage_level[t] <= thermal_storage_capacity,
                    name=f"thermal_storage_level_limit_{t}")

# 5.5 装机容量与最小运行约束
model.addConstrs((heat_pump_capacity >= heat_pump_output[t] for t in range(time_steps)),
                 name="heat_pump_capacity_constraint")
model.addConstrs((heat_pump_output[t] >= 0.25 * heat_pump_capacity for t in range(168, time_steps)),
                 name="heat_pump_min_output")
model.addConstrs((electrolyzer_capacity_AE >= electrolyzer_power_AE[t] for t in range(time_steps)),
                 name="electrolyzer_AE_capacity_constraint")
model.addConstrs((electrolyzer_capacity_PEM >= electrolyzer_power_PEM[t] for t in range(time_steps)),
                 name="electrolyzer_PEM_capacity_constraint")
model.addConstrs((electrolyzer_power_AE[t] >= 0.1 * electrolyzer_capacity_AE for t in range(168, time_steps)),
                 name="electrolyzer_AE_min_power")
model.addConstrs((dac_CO2[t] >= 0.25 * dac_capacity for t in range(168, time_steps)),
                 name="heat_pump_min_output")
#model.addConstrs((FTL_synthesis_power[t] >= 0.25 * FTL_synthesis_capacity for t in range(168, time_steps)),
#                    name="FTL_synthesis_min_output")
model.addConstrs((FTL_synthesis_power[t] <= FTL_synthesis_capacity for t in range(time_steps)),
                 name="FTL_synthesis_max_output")

# 5.6 每小时CO2供需平衡约束：直接供给加储存放出需满足当期甲醇CO2需求

model.addConstrs((direct_co2[t] + co2_discharge[t] >= FTL_synthesis_power[t] * FTL_co2_ratio for t in range(time_steps)),
        name="co2_hourly_balance")

# 5.7 年度CO2供给约束
model.addConstr(gp.quicksum(direct_co2[t] + co2_discharge[t] for t in range(time_steps)) >= target_FTL_production * FTL_co2_ratio,
                name="annual_co2_supply_constraint")

# 5.8 新增：AE型电解槽必须配备至少10%（此处0.5倍）储能电池
model.addConstr(battery_energy_capacity >= 0.5 * electrolyzer_capacity_AE,
                name="min_battery_for_AE")

# =============================================================================
# 6. 求解并输出结果
# =============================================================================
model.optimize()

if model.status == GRB.OPTIMAL:
    print(f"最优目标值：${model.objVal:.2f}")
    print("\n装机容量决策：")
    print(f"风电装机容量：{wind_capacity.x:.2f} kW")
    print(f"光伏装机容量：{pv_capacity.x:.2f} kW")
    print(f"电池能量功率：{battery_energy_capacity.x:.2f} kW")
    print(f"电池储存容量：{battery_storage_capacity.x:.2f} kWh")
    print(f"电解槽装机(AE)：{electrolyzer_capacity_AE.x:.2f} kW")
    print(f"电解槽装机(PEM)：{electrolyzer_capacity_PEM.x:.2f} kW")
    print(f"FTL合成装机：{FTL_synthesis_capacity.x:.2f} kg/h")
    print(f"氢气储存容量：{hydrogen_storage_capacity.x:.2f} kg")
    print(f"热泵装机：{heat_pump_capacity.x:.2f} kW_th")
    print(f"热储容量：{thermal_storage_capacity.x:.2f} kWh_th")
    print(f"DAC装机：{dac_capacity.x:.2f} kgCO2/h")
    print(f"CO2压缩机：{co2_compressor_capacity.x:.2f} kgCO2/h")
    print(f"CO2储存：{co2_storage_capacity.x:.2f} kg")
    print(f"FTL储存：{FTL_storage_capacity.x:.2f} kg")
    
    # 年产量汇总
    total_FTL_production = sum(FTL_synthesis_power[t].x for t in range(time_steps))
    print(f"\n年FTL产量：{total_FTL_production:.2f} kg")
    print(f"年产氢量：{sum(electrolyzer_power_AE[t].x * conversion_factor_AE + electrolyzer_power_PEM[t].x * conversion_factor_PEM for t in range(time_steps)):.2f} kg")
    print(f"年DAC CO2：{sum(dac_CO2[t].x for t in range(time_steps)):.2f} kg")
    
    # 设备利用率（年均运行时间除以8760）
    avg_wind_power = sum(wind_output[t] * wind_capacity.x for t in range(time_steps)) / 8760
    avg_pv_power = sum(pv_output[t] * pv_capacity.x for t in range(time_steps)) / 8760
    avg_AE_power = sum(electrolyzer_power_AE[t].x for t in range(time_steps)) / 8760
    avg_PEM_power = sum(electrolyzer_power_PEM[t].x for t in range(time_steps)) / 8760
    avg_FTL_power = sum(FTL_synthesis_power[t].x for t in range(time_steps)) / 8760
    avg_dac_usage = sum(dac_CO2[t].x for t in range(time_steps)) / 8760
    
    print(f"\n设备利用率：")
    print(f"风电平均出力：{avg_wind_power:.2f} kW (容量系数: {avg_wind_power/wind_capacity.x*100:.1f}%)")
    print(f"光伏平均出力：{avg_pv_power:.2f} kW (容量系数: {avg_pv_power/pv_capacity.x*100:.1f}%)")
    if electrolyzer_capacity_AE.x > 0:
        print(f"AE电解槽平均功率：{avg_AE_power:.2f} kW (利用率: {avg_AE_power/electrolyzer_capacity_AE.x*100:.1f}%)")
    if electrolyzer_capacity_PEM.x > 0:
        print(f"PEM电解槽平均功率：{avg_PEM_power:.2f} kW (利用率: {avg_PEM_power/electrolyzer_capacity_PEM.x*100:.1f}%)")
    if FTL_synthesis_capacity.x > 0:
        print(f"FTL合成平均功率：{avg_FTL_power:.2f} kg/h (利用率: {avg_FTL_power/FTL_synthesis_capacity.x*100:.1f}%)")
    if dac_capacity.x > 0:
        print(f"DAC平均使用：{avg_dac_usage:.2f} kgCO2/h (利用率: {avg_dac_usage/dac_capacity.x*100:.1f}%)")
    
    # 计算年发电量和多余电量
    annual_wind_generation = sum(wind_output[t] * wind_capacity.x for t in range(time_steps))
    annual_pv_generation = sum(pv_output[t] * pv_capacity.x for t in range(time_steps))
    annual_surplus = sum(surplus[t].x for t in range(time_steps))
    annual_battery_charge = sum(battery_charge[t].x for t in range(time_steps))
    annual_battery_discharge = sum(battery_discharge[t].x for t in range(time_steps))
    
    print(f"\n能量统计：")
    print(f"年风电发电量：{annual_wind_generation:.2f} kWh")
    print(f"年光伏发电量：{annual_pv_generation:.2f} kWh")
    print(f"年总发电量：{annual_wind_generation + annual_pv_generation:.2f} kWh")
    print(f"年电池充电量：{annual_battery_charge:.2f} kWh")
    print(f"年电池放电量：{annual_battery_discharge:.2f} kWh")
    print(f"年多余电量：{annual_surplus:.2f} kWh ({annual_surplus/(annual_wind_generation + annual_pv_generation)*100:.2f}%)")
    
    # 成本分解分析
    wind_cost_contribution = wind_annual_cost * wind_capacity.x
    pv_cost_contribution = pv_annual_cost * pv_capacity.x
    battery_energy_cost_contribution = battery_energy_annual_cost * battery_energy_capacity.x
    battery_storage_cost_contribution = battery_storage_annual_cost * battery_storage_capacity.x
    electrolyzer_AE_cost_contribution = electrolyzer_annual_cost_AE * electrolyzer_capacity_AE.x
    electrolyzer_PEM_cost_contribution = electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM.x
    FTL_synthesis_cost_contribution = FTL_synthesis_annual_cost * FTL_synthesis_capacity.x
    heat_pump_cost_contribution = heat_pump_annual_cost * heat_pump_capacity.x
    thermal_storage_cost_contribution = thermal_storage_annual_cost * thermal_storage_capacity.x
    hydrogen_storage_cost_contribution = hydrogen_storage_annual_cost * hydrogen_storage_capacity.x
    dac_cost_contribution = dac_annual_cost * dac_capacity.x
    co2_compressor_cost_contribution = co2_compressor_annual_cost * co2_compressor_capacity.x
    co2_storage_cost_contribution = co2_storage_annual_cost * co2_storage_capacity.x
    FTL_storage_cost_contribution = FTL_storage_annual_cost * FTL_storage_capacity.x
    variable_cost_contribution = (surplus_penalty * annual_surplus + 
                                battery_charge_penalty * annual_battery_charge +
                                sorbent_cost * sum(dac_CO2[t].x for t in range(time_steps)) +
                                hydrogen_charge_penalty * sum(hydrogen_charge[t].x for t in range(time_steps)) +
                                FTL_charge_penalty * sum(FTL_storage_charge[t].x for t in range(time_steps)))
    
    print(f"\n成本分解：")
    print(f"风电成本：${wind_cost_contribution:.2f} ({wind_cost_contribution/model.objVal*100:.1f}%)")
    print(f"光伏成本：${pv_cost_contribution:.2f} ({pv_cost_contribution/model.objVal*100:.1f}%)")
    print(f"电池能量成本：${battery_energy_cost_contribution:.2f} ({battery_energy_cost_contribution/model.objVal*100:.1f}%)")
    print(f"电池储存成本：${battery_storage_cost_contribution:.2f} ({battery_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"AE电解槽成本：${electrolyzer_AE_cost_contribution:.2f} ({electrolyzer_AE_cost_contribution/model.objVal*100:.1f}%)")
    print(f"PEM电解槽成本：${electrolyzer_PEM_cost_contribution:.2f} ({electrolyzer_PEM_cost_contribution/model.objVal*100:.1f}%)")
    print(f"FTL合成成本：${FTL_synthesis_cost_contribution:.2f} ({FTL_synthesis_cost_contribution/model.objVal*100:.1f}%)")
    print(f"热泵成本：${heat_pump_cost_contribution:.2f} ({heat_pump_cost_contribution/model.objVal*100:.1f}%)")
    print(f"热储成本：${thermal_storage_cost_contribution:.2f} ({thermal_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"氢储成本：${hydrogen_storage_cost_contribution:.2f} ({hydrogen_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"DAC成本：${dac_cost_contribution:.2f} ({dac_cost_contribution/model.objVal*100:.1f}%)")
    print(f"CO2压缩成本：${co2_compressor_cost_contribution:.2f} ({co2_compressor_cost_contribution/model.objVal*100:.1f}%)")
    print(f"CO2储存成本：${co2_storage_cost_contribution:.2f} ({co2_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"FTL储存成本：${FTL_storage_cost_contribution:.2f} ({FTL_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"运行变动成本：${variable_cost_contribution:.2f} ({variable_cost_contribution/model.objVal*100:.1f}%)")
    
    print(f"\nFTL生产平准化成本：${model.objVal/total_FTL_production*1000:.2f} $/t")
else:
    print("模型未找到最优解")
    print(f"求解状态：{model.status}")

In [ ]:
# 衍生平准化成本：LCOE、LCOH、供热LCOHeat、DAC LCO₂
try:
    assert model.status == GRB.OPTIMAL
except Exception:
    print("未检测到最优解，跳过平准化成本派生计算。")
else:
    # 1) 基础年量统计（沿用上一单元格变量）
    time_steps = 8760
    gross_gen_kWh = annual_wind_generation + annual_pv_generation
    net_gen_kWh = max(gross_gen_kWh - annual_surplus, 1e-9)

    # 各用电端合计（kWh）
    elec_for_h2_kWh = sum(electrolyzer_power_AE[t].x + electrolyzer_power_PEM[t].x for t in range(time_steps))
    elec_for_ftl_kWh = sum(FTL_synthesis_power[t].x * FTL_electricity_ratio for t in range(time_steps))
    elec_for_h2_storage_kWh = sum(hydrogen_charge[t].x * hydrogen_storage_elec for t in range(time_steps))
    elec_for_hp_kWh = sum(heat_pump_output[t].x / heat_pump_cop for t in range(time_steps))
    elec_for_dac_kWh = sum(dac_electricity[t].x for t in range(time_steps))
    elec_for_co2_comp_kWh = sum(co2_comp_elec[t].x for t in range(time_steps))

    loads_kWh = (elec_for_h2_kWh + elec_for_ftl_kWh + elec_for_h2_storage_kWh +
                 elec_for_hp_kWh + elec_for_dac_kWh + elec_for_co2_comp_kWh)
    loads_kWh = max(loads_kWh, 1e-9)

    # 2) LCOE（仅计入发电/储能侧固定成本，避免与下游双计）
    lcoe_numerator = (
        wind_annual_cost * wind_capacity.x +
        pv_annual_cost * pv_capacity.x +
        battery_energy_annual_cost * battery_energy_capacity.x +
        battery_storage_annual_cost * battery_storage_capacity.x
    )

    lcoe_gross = lcoe_numerator / max(gross_gen_kWh, 1e-9)
    lcoe_net = lcoe_numerator / net_gen_kWh
    lcoe_load = lcoe_numerator / loads_kWh

    print("\n—— 平准化成本（派生指标） ——")
    print(f"LCOE-发电侧（总发电量基数）: ${lcoe_gross:.4f} $/kWh")
    print(f"LCOE-用电侧（负载基数）:    ${lcoe_load:.4f} $/kWh")
    print(f"LCOE-净供电（剔除弃电）:    ${lcoe_net:.4f} $/kWh")

    # 3) LCOH（$/kg-H2）：电解槽固定成本 + 用电成本（按LCOE-用电侧计价）/ 产氢量
    total_H2_kg = sum(
        electrolyzer_power_AE[t].x * conversion_factor_AE +
        electrolyzer_power_PEM[t].x * conversion_factor_PEM
        for t in range(time_steps)
    )

    if total_H2_kg > 0:
        electrolyzer_fixed_annual = (
            electrolyzer_annual_cost_AE * electrolyzer_capacity_AE.x +
            electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM.x +
            hydrogen_storage_annual_cost * hydrogen_storage_capacity.x
        )
        lcoh_usd_per_kg = (electrolyzer_fixed_annual + elec_for_h2_kWh * lcoe_load) / total_H2_kg
        print(f"平准化制氢成本（LCOH）: ${lcoh_usd_per_kg:.2f} $/kg-H2")
    else:
        print("平准化制氢成本（LCOH）：无产氢，未计算")

    # 4) 供热LCOHeat（$/kWh_th）：热泵+储热固定成本 + 热泵用电成本（按LCOE-用电侧计价）/ 供热量
    total_heat_kWhth = sum(heat_pump_output[t].x for t in range(time_steps))
    if total_heat_kWhth > 0:
        heat_fixed_annual = (
            heat_pump_annual_cost * heat_pump_capacity.x +
            thermal_storage_annual_cost * thermal_storage_capacity.x
        )
        lcoheat_usd_per_kWhth = (heat_fixed_annual + elec_for_hp_kWh * lcoe_load) / total_heat_kWhth
        print(f"平准化供热成本（LCOHeat）: ${lcoheat_usd_per_kWhth:.4f} $/kWh_th  ({lcoheat_usd_per_kWhth*1000:.2f} $/MWh_th)")
    else:
        print("平准化供热成本（LCOHeat）：无供热，未计算")

    # 5) DAC LCO₂（$/t-CO2）：DAC+压缩+储存固定成本 + 用电与供热成本（供热按LCOHeat计价）/ 捕集CO2
    total_CO2_kg = sum(dac_CO2[t].x for t in range(time_steps))
    if total_CO2_kg > 0:
        total_dac_heat_kWhth = sum(dac_heat_input[t].x for t in range(time_steps))
        total_dac_elec_kWh = elec_for_dac_kWh + elec_for_co2_comp_kWh
        dac_fixed_annual = (
            dac_annual_cost * dac_capacity.x +
            co2_compressor_annual_cost * co2_compressor_capacity.x +
            co2_storage_annual_cost * co2_storage_capacity.x
        )
        # sorbent_cost 已在目标函数中按捕集量计入，这里作为变量成本显式加入
        dac_variable_sorbent = sorbent_cost * total_CO2_kg
        # 若未计算出LCOHeat，则改用电价按等效COP折算
        if total_heat_kWhth > 0:
            heat_price = lcoheat_usd_per_kWhth
        else:
            heat_price = lcoe_load * (1.0 / max(heat_pump_cop, 1e-9))
        lcodac_usd_per_tCO2 = (
            dac_fixed_annual + dac_variable_sorbent +
            total_dac_elec_kWh * lcoe_load + total_dac_heat_kWhth * heat_price
        ) / (total_CO2_kg / 1000.0)
        print(f"平准化直接空气捕集成本（DAC，全系统）: ${lcodac_usd_per_tCO2:.2f} $/t-CO2")
    else:
        print("平准化直接空气捕集成本：无CO2捕集量，未计算")

In [ ]:
# 衍生平准化成本：LCOE、LCOH、供热LCOHeat、DAC LCO₂
try:
    assert model.status == GRB.OPTIMAL
except Exception:
    print("未检测到最优解，跳过平准化成本派生计算。")
else:
    # 1) 基础年量统计（沿用上一单元格变量）
    time_steps = 8760
    gross_gen_kWh = annual_wind_generation + annual_pv_generation
    net_gen_kWh = max(gross_gen_kWh - annual_surplus, 1e-9)

    # 各用电端合计（kWh）
    elec_for_h2_kWh = sum(electrolyzer_power_AE[t].x + electrolyzer_power_PEM[t].x for t in range(time_steps))
    elec_for_ftl_kWh = sum(FTL_synthesis_power[t].x * FTL_electricity_ratio for t in range(time_steps))
    elec_for_h2_storage_kWh = sum(hydrogen_charge[t].x * hydrogen_storage_elec for t in range(time_steps))
    elec_for_hp_kWh = sum(heat_pump_output[t].x / heat_pump_cop for t in range(time_steps))
    elec_for_dac_kWh = sum(dac_electricity[t].x for t in range(time_steps))
    elec_for_co2_comp_kWh = sum(co2_comp_elec[t].x for t in range(time_steps))

    loads_kWh = (elec_for_h2_kWh + elec_for_ftl_kWh + elec_for_h2_storage_kWh +
                 elec_for_hp_kWh + elec_for_dac_kWh + elec_for_co2_comp_kWh)
    loads_kWh = max(loads_kWh, 1e-9)

    # 2) LCOE（仅计入发电/储能侧固定成本，避免与下游双计）
    lcoe_numerator = (
        wind_annual_cost * wind_capacity.x +
        pv_annual_cost * pv_capacity.x +
        battery_energy_annual_cost * battery_energy_capacity.x +
        battery_storage_annual_cost * battery_storage_capacity.x
    )

    lcoe_gross = lcoe_numerator / max(gross_gen_kWh, 1e-9)
    lcoe_net = lcoe_numerator / net_gen_kWh
    lcoe_load = lcoe_numerator / loads_kWh

    print("\n—— 平准化成本（派生指标） ——")
    print(f"LCOE-发电侧（总发电量基数）: ${lcoe_gross:.4f} $/kWh")
    print(f"LCOE-用电侧（负载基数）:    ${lcoe_load:.4f} $/kWh")
    print(f"LCOE-净供电（剔除弃电）:    ${lcoe_net:.4f} $/kWh")

    # 3) LCOH（$/kg-H2）：电解槽固定成本 + 用电成本（按LCOE-用电侧计价）/ 产氢量
    total_H2_kg = sum(
        electrolyzer_power_AE[t].x * conversion_factor_AE +
        electrolyzer_power_PEM[t].x * conversion_factor_PEM
        for t in range(time_steps)
    )

    if total_H2_kg > 0:
        electrolyzer_fixed_annual = (
            electrolyzer_annual_cost_AE * electrolyzer_capacity_AE.x +
            electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM.x +
            hydrogen_storage_annual_cost * hydrogen_storage_capacity.x
        )
        lcoh_usd_per_kg = (electrolyzer_fixed_annual + elec_for_h2_kWh * lcoe_load) / total_H2_kg
        print(f"平准化制氢成本（LCOH）: ${lcoh_usd_per_kg:.2f} $/kg-H2")
    else:
        print("平准化制氢成本（LCOH）：无产氢，未计算")

    # 4) 供热LCOHeat（$/kWh_th）：热泵+储热固定成本 + 热泵用电成本（按LCOE-用电侧计价）/ 供热量
    total_heat_kWhth = sum(heat_pump_output[t].x for t in range(time_steps))
    if total_heat_kWhth > 0:
        heat_fixed_annual = (
            heat_pump_annual_cost * heat_pump_capacity.x +
            thermal_storage_annual_cost * thermal_storage_capacity.x
        )
        lcoheat_usd_per_kWhth = (heat_fixed_annual + elec_for_hp_kWh * lcoe_load) / total_heat_kWhth
        print(f"平准化供热成本（LCOHeat）: ${lcoheat_usd_per_kWhth:.4f} $/kWh_th  ({lcoheat_usd_per_kWhth*1000:.2f} $/MWh_th)")
    else:
        print("平准化供热成本（LCOHeat）：无供热，未计算")

    # 5) DAC LCO₂（$/t-CO2）：DAC+压缩+储存固定成本 + 用电与供热成本（供热按LCOHeat计价）/ 捕集CO2
    total_CO2_kg = sum(dac_CO2[t].x for t in range(time_steps))
    if total_CO2_kg > 0:
        total_dac_heat_kWhth = sum(dac_heat_input[t].x for t in range(time_steps))
        total_dac_elec_kWh = elec_for_dac_kWh + elec_for_co2_comp_kWh
        dac_fixed_annual = (
            dac_annual_cost * dac_capacity.x +
            co2_compressor_annual_cost * co2_compressor_capacity.x +
            co2_storage_annual_cost * co2_storage_capacity.x
        )
        # sorbent_cost 已在目标函数中按捕集量计入，这里作为变量成本显式加入
        dac_variable_sorbent = sorbent_cost * total_CO2_kg
        # 若未计算出LCOHeat，则改用电价按等效COP折算
        if total_heat_kWhth > 0:
            heat_price = lcoheat_usd_per_kWhth
        else:
            heat_price = lcoe_load * (1.0 / max(heat_pump_cop, 1e-9))
        lcodac_usd_per_tCO2 = (
            dac_fixed_annual + dac_variable_sorbent +
            total_dac_elec_kWh * lcoe_load + total_dac_heat_kWhth * heat_price
        ) / (total_CO2_kg / 1000.0)
        print(f"平准化直接空气捕集成本（DAC，全系统）: ${lcodac_usd_per_tCO2:.2f} $/t-CO2")
    else:
        print("平准化直接空气捕集成本：无CO2捕集量，未计算")

# 2. 甲醇生产优化模型

## 2.1 模型概述
基于CO2加氢合成甲醇的可再生能源制甲醇系统优化模型。

### 主要设备
- **可再生能源发电**：风电、光伏  
- **储能系统**：锂电池（功率/容量分离建模）
- **制氢系统**：AE电解槽、PEM电解槽
- **甲醇合成装置**：3H2 + CO2 → CH3OH + H2O
- **DAC系统**：直接空气捕获CO2
- **热管理**：热泵+储热系统
- **储存系统**：氢气储存、CO2储存、甲醇储存

### 关键技术参数
- 甲醇合成氢耗：0.19 kg H2/kg CH3OH
- 甲醇合成CO2耗：1.4 kg CO2/kg CH3OH  
- 甲醇合成电耗：0.2 kWh/kg CH3OH

### 目标
最小化系统年化成本，满足年产10吨甲醇需求

In [ ]:
# =============================================================================
# 甲醇 + DAC 优化（拆分电池电力/储能，区分 AE/PEM 效率）
# =============================================================================

# =============================================================================
# 1. 导入库与数据加载
# =============================================================================
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB

# =============================================================================
# 1. 导入库与数据加载
# =============================================================================
wind_solar_data = pd.read_csv('CF_NM_2023.csv')
pv_output   = wind_solar_data.iloc[:, 0].values  # 假设第一列为 'pv'
wind_output = wind_solar_data.iloc[:, 1].values  # 假设第二列为 'wind'

# =============================================================================
# 2. 使用全局参数
# =============================================================================
# 所有参数已在全局配置单元格中定义，这里直接使用全局变量

# =============================================================================
# 3. 模型创建与变量定义
# =============================================================================

# 创建模型
model = Model("HydrogenMethanolOptimization")
model.setParam('TimeLimit', 1000)
model.setParam('MIPFocus', 1)

# 强制用 Barrier，跳过 Crossover，把收敛精度放宽到 1e-6 ，并行 8 线程
model.setParam('Method', 2)        # 2 = Barrier
model.setParam('Crossover', 0)     # 关闭回退
model.setParam('BarConvTol', 1e-6) # 默认 1e-8 → 1e-6
model.setParam('Threads', 1)

# 选用 Homogeneous KKT 因式分解策略，通常更稳定
model.setParam('BarHomogeneous', 1)

# 关闭过度 Presolve（有时 Aggressive 反而慢）
model.setParam('Presolve', 1)      # 1 = 默认

time_steps = 8760

# 3.1 装机容量决策变量
# —— 可再生发电、储能、工艺设备及CO2处理装置
wind_capacity = model.addVar(lb=0, name="wind_capacity")
pv_capacity   = model.addVar(lb=0, name="pv_capacity")
battery_energy_capacity = model.addVar(lb=0, name="battery_energy_capacity")
battery_storage_capacity = model.addVar(lb=0, name="battery_storage_capacity")
electrolyzer_capacity_AE = model.addVar(lb=0, name="electrolyzer_capacity_AE")
electrolyzer_capacity_PEM = model.addVar(lb=0, name="electrolyzer_capacity_PEM")
methanol_synthesis_capacity = model.addVar(lb=0, name="methanol_capacity")
heat_pump_capacity = model.addVar(lb=0, name="heat_pump_capacity")
thermal_storage_capacity = model.addVar(lb=0, name="thermal_storage_capacity")
methanol_storage_capacity = model.addVar(lb=0, name="methanol_storage_capacity")
hydrogen_storage_capacity = model.addVar(lb=0, name="hydrogen_storage_capacity")
dac_capacity = model.addVar(lb=0, name="dac_capacity")
co2_compressor_capacity = model.addVar(lb=0, name="co2_compressor_capacity")
co2_storage_capacity = model.addVar(lb=0, name="co2_storage_capacity")

# 3.2 时间步变量（每小时变量，共 time_steps 个）
# 电池相关变量
energy_balance = model.addVars(time_steps, lb=0, name="energy_balance")
battery_charge = model.addVars(time_steps, lb=0, name="battery_charge")
battery_discharge = model.addVars(time_steps, lb=0, name="battery_discharge")

# 电解槽产氢变量
electrolyzer_power_AE = model.addVars(time_steps, lb=0, name="electrolyzer_power_AE")
electrolyzer_power_PEM = model.addVars(time_steps, lb=0, name="electrolyzer_power_PEM")

# 甲醇合成与甲醇储存变量
methanol_synthesis_power = model.addVars(time_steps, lb=0, name="methanol_synthesis_power")
methanol_storage_balance = model.addVars(time_steps, lb=0, name="methanol_storage_balance")
methanol_storage_charge = model.addVars(time_steps, lb=0, name="methanol_storage_charge")
methanol_storage_discharge = model.addVars(time_steps, lb=0, name="methanol_storage_discharge")

# 氢气存储变量
hydrogen_storage_balance = model.addVars(time_steps, lb=0, name="hydrogen_storage_balance")
hydrogen_charge = model.addVars(time_steps, lb=0, name="hydrogen_charge")
hydrogen_discharge = model.addVars(time_steps, lb=0, name="hydrogen_discharge")

# DAC（CO2捕集）变量
dac_CO2 = model.addVars(time_steps, lb=0, name="dac_CO2")
dac_electricity = model.addVars(time_steps, lb=0, name="dac_electricity")
dac_heat_input = model.addVars(time_steps, lb=0, name="dac_heat_input")

# 热泵与热储变量
heat_pump_output = model.addVars(time_steps, lb=0, name="heat_pump_output")
thermal_charge = model.addVars(time_steps, lb=0, name="thermal_charge")
thermal_discharge = model.addVars(time_steps, lb=0, name="thermal_discharge")
thermal_storage_level = model.addVars(time_steps, lb=0, name="thermal_storage_level")

# CO2压缩及储存变量
co2_comp_elec = model.addVars(time_steps, lb=0, name="co2_comp_elec")
co2_storage_level = model.addVars(time_steps, lb=0, name="co2_storage_level")
co2_charge = model.addVars(time_steps, lb=0, name="co2_charge")
co2_discharge = model.addVars(time_steps, lb=0, name="co2_discharge")
direct_co2 = model.addVars(time_steps, lb=0, name="direct_co2")
stored_co2 = model.addVars(time_steps, lb=0, name="stored_co2")

# 3.3 其它变量（例如多余电量）
surplus = model.addVars(time_steps, lb=0, name="surplus")

# =============================================================================
# 4. 目标函数
# =============================================================================
model.setObjective(
    (wind_annual_cost * wind_capacity +
     pv_annual_cost * pv_capacity +
     battery_energy_annual_cost * battery_energy_capacity +
     battery_storage_annual_cost * battery_storage_capacity +
     electrolyzer_annual_cost_AE * electrolyzer_capacity_AE +
     electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM +
     methanol_synthesis_annual_cost * methanol_synthesis_capacity +
     heat_pump_annual_cost * heat_pump_capacity +
     thermal_storage_annual_cost * thermal_storage_capacity +
     hydrogen_storage_annual_cost * hydrogen_storage_capacity +
     methanol_storage_annual_cost * methanol_storage_capacity +
     dac_annual_cost * dac_capacity +
     co2_compressor_annual_cost * co2_compressor_capacity +
     co2_storage_annual_cost * co2_storage_capacity +
     surplus_penalty * gp.quicksum(surplus[t] for t in range(time_steps)) +
     battery_charge_penalty * gp.quicksum(battery_charge[t] for t in range(time_steps)) +
     battery_discharge_penalty * gp.quicksum(battery_discharge[t] for t in range(time_steps)) +
     sorbent_cost * gp.quicksum(dac_CO2[t] for t in range(time_steps))+
     hydrogen_charge_penalty * gp.quicksum(hydrogen_charge[t] for t in range(time_steps))+
     methanol_charge_penalty * gp.quicksum(methanol_storage_charge[t] for t in range(time_steps))
    ),
    GRB.MINIMIZE
)

# =============================================================================
# 5. 约束条件
# =============================================================================

# 5.1 逐时设备灵活性（爬坡）约束
for t in range(1, time_steps):
    # 电解槽 AE 爬坡限制
    model.addConstr(electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] <= Flex_down * electrolyzer_capacity_AE,
                    name=f"electrolyzer_AE_ramp_up_{t}")
    model.addConstr(electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] >= -Flex_down * electrolyzer_capacity_AE,
                    name=f"electrolyzer_AE_ramp_down_{t}")
    # 甲醇合成爬坡限制
    model.addConstr(methanol_synthesis_power[t] - methanol_synthesis_power[t-1] <= Flex_up * methanol_synthesis_capacity,
                    name=f"methanol_ramp_up_{t}")
    model.addConstr(methanol_synthesis_power[t] - methanol_synthesis_power[t-1] >= -Flex_down * methanol_synthesis_capacity,
                    name=f"methanol_ramp_down_{t}")
    # DAC爬坡限制
    model.addConstr(dac_CO2[t] - dac_CO2[t-1] <= Flex_up * dac_capacity,
                    name=f"dac_ramp_up_{t}")
    model.addConstr(dac_CO2[t] - dac_CO2[t-1] >= -Flex_down * dac_capacity,
                    name=f"dac_ramp_down_{t}")

# 5.2 甲醇供需平衡约束（含储存充放，支持多种聚合尺度）
start_index = 168  # 避开初始不稳定阶段
if aggregation_mode == "hourly":
    for t in range(start_index, time_steps):
        model.addConstr(
            methanol_synthesis_power[t] + methanol_storage_discharge[t] >=
            hourly_methanol_demand + methanol_storage_charge[t],
            name=f"methanol_balance_hourly_{t}"
        )
elif aggregation_mode == "3hourly":
    interval = 3
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(methanol_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(methanol_storage_discharge[t] for t in range(t_start, t_end))
            ==
            hourly_methanol_demand * interval +
            gp.quicksum(methanol_storage_charge[t] for t in range(t_start, t_end)),
            name=f"methanol_balance_3hourly_{i}"
        )
elif aggregation_mode == "daily":
    interval = 24
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(methanol_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(methanol_storage_discharge[t] for t in range(t_start, t_end))
            >=
            0.8 * hourly_methanol_demand * interval +
            gp.quicksum(methanol_storage_charge[t] for t in range(t_start, t_end)),
            name=f"methanol_balance_daily_{i}"
        )
elif aggregation_mode == "weekly":
    interval = 168
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(methanol_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(methanol_storage_discharge[t] for t in range(t_start, t_end))
            ==
            hourly_methanol_demand * interval +
            gp.quicksum(methanol_storage_charge[t] for t in range(t_start, t_end)),
            name=f"methanol_balance_weekly_{i}"
        )
elif aggregation_mode == "monthly":
    interval = 730
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(methanol_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(methanol_storage_discharge[t] for t in range(t_start, t_end))
            ==
            hourly_methanol_demand * interval +
            gp.quicksum(methanol_storage_charge[t] for t in range(t_start, t_end)),
            name=f"methanol_balance_monthly_{i}"
        )
elif aggregation_mode == "annual":
    model.addConstr(
        gp.quicksum(methanol_synthesis_power[t] for t in range(start_index, time_steps)) +
        gp.quicksum(methanol_storage_discharge[t] for t in range(start_index, time_steps))
        ==
        hourly_methanol_demand * (time_steps - start_index) +
        gp.quicksum(methanol_storage_charge[t] for t in range(start_index, time_steps)),
        name="methanol_balance_annual"
    )
else:
    raise ValueError("Unsupported aggregation_mode value!")

# 附加：年生产总量约束
model.addConstr(
    gp.quicksum(methanol_synthesis_power[t] for t in range(time_steps)) >= hourly_methanol_demand * 8760,
    name="annual_methanol_production"
)

# 5.3 甲醇储存动态与容量约束
model.addConstr(methanol_storage_balance[0] ==
                methanol_storage_charge[0] - methanol_storage_discharge[0],
                name="methanol_storage_balance_0")
for t in range(1, time_steps):
    model.addConstr(
        methanol_storage_balance[t] == methanol_storage_balance[t-1] + methanol_storage_charge[t] - methanol_storage_discharge[t],
        name=f"methanol_storage_balance_{t}"
    )
    model.addConstr(methanol_storage_balance[t] <= methanol_storage_capacity,
                    name=f"methanol_storage_capacity_limit_{t}")

# 5.4 能量平衡与储能动态约束
conversion_factor_AE  = electrolyzer_eff_AE  * 3.6 / 120
conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

for t in range(time_steps):
    # 计算当期风电与光伏发电
    wind_power = wind_output[t] * wind_capacity
    pv_power = pv_output[t] * pv_capacity
    
    # 甲醇储存：要求合成功率与储存放出至少满足充入量
    model.addConstr(methanol_synthesis_power[t] + methanol_storage_discharge[t] >=
                    methanol_storage_charge[t],
                    name=f"methanol_min_balance_{t}")
                    
    # 电池储能及充放电限额
    model.addConstr(energy_balance[t] <= battery_storage_capacity,
                    name=f"energy_balance_limit_{t}")
    model.addConstr(battery_charge[t] + battery_discharge[t] <= battery_energy_capacity,
                    name=f"battery_charge_discharge_limit_{t}")
    # 氢气存储上限
    model.addConstr(hydrogen_storage_balance[t] <= hydrogen_storage_capacity,
                    name=f"hydrogen_storage_capacity_limit_{t}")
    model.addConstr(hydrogen_charge[t] + hydrogen_discharge[t] <= hydrogen_storage_capacity,
                    name=f"hydrogen_charge_limit_{t}")
    
    # 电力平衡（含电解槽、甲醇合成、储能、热泵、DAC与CO2压缩电耗等）
    model.addConstr(
        wind_power + pv_power + battery_discharge[t] ==
        (electrolyzer_power_AE[t] + electrolyzer_power_PEM[t]) +
        methanol_synthesis_power[t] * methanol_electricity_ratio +
        battery_charge[t] + hydrogen_charge[t] * hydrogen_storage_elec +
        (heat_pump_output[t] / heat_pump_cop) + dac_electricity[t] + surplus[t] + co2_comp_elec[t],
        name=f"power_balance_{t}"
    )
    # 电池动态平衡
    if t == 0:
        model.addConstr(
            energy_balance[t] == battery_charge[t] * battery_charge_efficiency
                                - battery_discharge[t] / battery_discharge_efficiency,
            name=f"battery_balance_{t}" )
    else:
        model.addConstr(
            energy_balance[t] == energy_balance[t-1] +
                                battery_charge[t] * battery_charge_efficiency
                              - battery_discharge[t] / battery_discharge_efficiency,
            name=f"battery_balance_{t}" )
    # 氢气存储动态平衡
    if t == 0:
        model.addConstr(
            hydrogen_storage_balance[t] == hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
            name=f"hydrogen_balance_{t}"
        )
    else:
        model.addConstr(
            hydrogen_storage_balance[t] == hydrogen_storage_balance[t-1] + hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
            name=f"hydrogen_balance_{t}"
        )
    # 氢气供需平衡：电解槽产氢需满足甲醇合成及储氢要求
    model.addConstr(
       	electrolyzer_power_AE[t] * conversion_factor_AE +
        electrolyzer_power_PEM[t] * conversion_factor_PEM + hydrogen_discharge[t] >=
        methanol_synthesis_power[t] * methanol_hydrogen_ratio + hydrogen_charge[t],
        name=f"hydrogen_PD_balance_{t}"
    )
    # DAC部分：热、电耗与捕集CO2量关系
    model.addConstr(dac_heat_input[t] == dac_CO2[t] * dac_heat_per_kg,
                    name=f"dac_heat_consumption_{t}")
    model.addConstr(dac_electricity[t] == dac_CO2[t] * dac_elec_per_kg,
                    name=f"dac_electricity_consumption_{t}")
    model.addConstr(dac_CO2[t] <= dac_capacity,
                    name=f"dac_capacity_limit_{t}")
    
    # CO2分流：直接供给与进入储存
    model.addConstr(direct_co2[t] + stored_co2[t] == dac_CO2[t],
                    name=f"co2_split_{t}")
    model.addConstr(stored_co2[t] <= co2_compressor_capacity,
                    name=f"co2_compressor_capacity_limit_{t}")
    model.addConstr(co2_comp_elec[t] == stored_co2[t] * co2_compression_energy,
                    name=f"co2_compression_energy_{t}")
    model.addConstr(co2_charge[t] == stored_co2[t],
                    name=f"co2_charge_relation_{t}")
    
    # 热泵与热储热平衡
    model.addConstr(
        heat_pump_output[t] - thermal_charge[t] + thermal_discharge[t] >= dac_heat_input[t],
        name=f"heat_supply_balance_{t}"
    )
    # CO2储存动态及容量约束
    if t == 0:
        model.addConstr(
            co2_storage_level[t] == co2_charge[t] - co2_discharge[t],
            name="co2_storage_balance_0"
        )
    else:
        model.addConstr(
            co2_storage_level[t] == co2_storage_level[t-1] + co2_charge[t] - co2_discharge[t],
            name=f"co2_storage_balance_{t}"
        )
    model.addConstr(co2_storage_level[t] <= co2_storage_capacity,
                    name=f"co2_storage_level_limit_{t}")
    
    # 储热系统动态及容量约束
    if t == 0:
        model.addConstr(
            thermal_storage_level[t] == thermal_charge[t] * thermal_storage_efficiency - thermal_discharge[t] / thermal_storage_efficiency,
            name="thermal_storage_balance_0"
        )
    else:
        model.addConstr(
            thermal_storage_level[t] == thermal_storage_level[t-1] + thermal_charge[t] * thermal_storage_efficiency - thermal_discharge[t] / thermal_storage_efficiency,
            name=f"thermal_storage_balance_{t}"
        )
    model.addConstr(thermal_storage_level[t] <= thermal_storage_capacity,
                    name=f"thermal_storage_level_limit_{t}")

# 5.5 装机容量与最小运行约束
model.addConstrs((heat_pump_capacity >= heat_pump_output[t] for t in range(time_steps)),
                 name="heat_pump_capacity_constraint")
model.addConstrs((heat_pump_output[t] >= 0.25 * heat_pump_capacity for t in range(168, time_steps)),
                 name="heat_pump_min_output")
model.addConstrs((electrolyzer_capacity_AE >= electrolyzer_power_AE[t] for t in range(time_steps)),
                 name="electrolyzer_AE_capacity_constraint")
model.addConstrs((electrolyzer_capacity_PEM >= electrolyzer_power_PEM[t] for t in range(time_steps)),
                 name="electrolyzer_PEM_capacity_constraint")
model.addConstrs((electrolyzer_power_AE[t] >= 0.1 * electrolyzer_capacity_AE for t in range(168, time_steps)),
                 name="electrolyzer_AE_min_power")
model.addConstrs((dac_CO2[t] >= 0.25 * dac_capacity for t in range(168, time_steps)),
                 name="heat_pump_min_output")
#model.addConstrs((methanol_synthesis_power[t] >= 0.25 * methanol_synthesis_capacity for t in range(168, time_steps)),
#                    name="methanol_synthesis_min_output")
model.addConstrs((methanol_synthesis_power[t] <= methanol_synthesis_capacity for t in range(time_steps)),
                 name="methanol_synthesis_max_output")

# 5.6 每小时CO2供需平衡约束：直接供给加储存放出需满足当期甲醇CO2需求

model.addConstrs((direct_co2[t] + co2_discharge[t] >= methanol_synthesis_power[t] * methanol_co2_ratio for t in range(time_steps)),
        name="co2_hourly_balance")

# 5.7 年度CO2供给约束
model.addConstr(gp.quicksum(direct_co2[t] + co2_discharge[t] for t in range(time_steps)) >= target_methanol_production * methanol_co2_ratio,
                name="annual_co2_supply_constraint")

# 5.8 新增：AE型电解槽必须配备至少10%（此处0.5倍）储能电池
model.addConstr(battery_energy_capacity >= 0.5 * electrolyzer_capacity_AE,
                name="min_battery_for_AE")

# =============================================================================
# 6. 求解并输出结果
# =============================================================================
model.optimize()

if model.status == GRB.OPTIMAL:
    print(f"最优目标值：${model.objVal:.2f}")
    print("\n装机容量决策：")
    print(f"风电装机容量：{wind_capacity.x:.2f} kW")
    print(f"光伏装机容量：{pv_capacity.x:.2f} kW")
    print(f"电池能量功率：{battery_energy_capacity.x:.2f} kW")
    print(f"电池储存容量：{battery_storage_capacity.x:.2f} kWh")
    print(f"电解槽装机(AE)：{electrolyzer_capacity_AE.x:.2f} kW")
    print(f"电解槽装机(PEM)：{electrolyzer_capacity_PEM.x:.2f} kW")
    print(f"甲醇合成装机：{methanol_synthesis_capacity.x:.2f} kg/h")
    print(f"氢气储存容量：{hydrogen_storage_capacity.x:.2f} kg")
    print(f"热泵装机：{heat_pump_capacity.x:.2f} kW_th")
    print(f"热储容量：{thermal_storage_capacity.x:.2f} kWh_th")
    print(f"DAC装机：{dac_capacity.x:.2f} kgCO2/h")
    print(f"CO2压缩机：{co2_compressor_capacity.x:.2f} kgCO2/h")
    print(f"CO2储存：{co2_storage_capacity.x:.2f} kg")
    print(f"甲醇储存：{methanol_storage_capacity.x:.2f} kg")
    
    # 年产量汇总
    total_methanol_production = sum(methanol_synthesis_power[t].x for t in range(time_steps))
    print(f"\n年甲醇产量：{total_methanol_production:.2f} kg")
    print(f"年产氢量：{sum(electrolyzer_power_AE[t].x * conversion_factor_AE + electrolyzer_power_PEM[t].x * conversion_factor_PEM for t in range(time_steps)):.2f} kg")
    print(f"年DAC CO2：{sum(dac_CO2[t].x for t in range(time_steps)):.2f} kg")
    
    # 设备利用率（年均运行时间除以8760）
    avg_wind_power = sum(wind_output[t] * wind_capacity.x for t in range(time_steps)) / 8760
    avg_pv_power = sum(pv_output[t] * pv_capacity.x for t in range(time_steps)) / 8760
    avg_AE_power = sum(electrolyzer_power_AE[t].x for t in range(time_steps)) / 8760
    avg_PEM_power = sum(electrolyzer_power_PEM[t].x for t in range(time_steps)) / 8760
    avg_methanol_power = sum(methanol_synthesis_power[t].x for t in range(time_steps)) / 8760
    avg_dac_usage = sum(dac_CO2[t].x for t in range(time_steps)) / 8760
    
    print(f"\n设备利用率：")
    print(f"风电平均出力：{avg_wind_power:.2f} kW (容量系数: {avg_wind_power/wind_capacity.x*100:.1f}%)")
    print(f"光伏平均出力：{avg_pv_power:.2f} kW (容量系数: {avg_pv_power/pv_capacity.x*100:.1f}%)")
    if electrolyzer_capacity_AE.x > 0:
        print(f"AE电解槽平均功率：{avg_AE_power:.2f} kW (利用率: {avg_AE_power/electrolyzer_capacity_AE.x*100:.1f}%)")
    if electrolyzer_capacity_PEM.x > 0:
        print(f"PEM电解槽平均功率：{avg_PEM_power:.2f} kW (利用率: {avg_PEM_power/electrolyzer_capacity_PEM.x*100:.1f}%)")
    if methanol_synthesis_capacity.x > 0:
        print(f"甲醇合成平均功率：{avg_methanol_power:.2f} kg/h (利用率: {avg_methanol_power/methanol_synthesis_capacity.x*100:.1f}%)")
    if dac_capacity.x > 0:
        print(f"DAC平均使用：{avg_dac_usage:.2f} kgCO2/h (利用率: {avg_dac_usage/dac_capacity.x*100:.1f}%)")
    
    # 计算年发电量和多余电量
    annual_wind_generation = sum(wind_output[t] * wind_capacity.x for t in range(time_steps))
    annual_pv_generation = sum(pv_output[t] * pv_capacity.x for t in range(time_steps))
    annual_surplus = sum(surplus[t].x for t in range(time_steps))
    annual_battery_charge = sum(battery_charge[t].x for t in range(time_steps))
    annual_battery_discharge = sum(battery_discharge[t].x for t in range(time_steps))
    
    print(f"\n能量统计：")
    print(f"年风电发电量：{annual_wind_generation:.2f} kWh")
    print(f"年光伏发电量：{annual_pv_generation:.2f} kWh")
    print(f"年总发电量：{annual_wind_generation + annual_pv_generation:.2f} kWh")
    print(f"年电池充电量：{annual_battery_charge:.2f} kWh")
    print(f"年电池放电量：{annual_battery_discharge:.2f} kWh")
    print(f"年多余电量：{annual_surplus:.2f} kWh ({annual_surplus/(annual_wind_generation + annual_pv_generation)*100:.2f}%)")
    
    # 成本分解分析
    wind_cost_contribution = wind_annual_cost * wind_capacity.x
    pv_cost_contribution = pv_annual_cost * pv_capacity.x
    battery_energy_cost_contribution = battery_energy_annual_cost * battery_energy_capacity.x
    battery_storage_cost_contribution = battery_storage_annual_cost * battery_storage_capacity.x
    electrolyzer_AE_cost_contribution = electrolyzer_annual_cost_AE * electrolyzer_capacity_AE.x
    electrolyzer_PEM_cost_contribution = electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM.x
    methanol_synthesis_cost_contribution = methanol_synthesis_annual_cost * methanol_synthesis_capacity.x
    heat_pump_cost_contribution = heat_pump_annual_cost * heat_pump_capacity.x
    thermal_storage_cost_contribution = thermal_storage_annual_cost * thermal_storage_capacity.x
    hydrogen_storage_cost_contribution = hydrogen_storage_annual_cost * hydrogen_storage_capacity.x
    dac_cost_contribution = dac_annual_cost * dac_capacity.x
    co2_compressor_cost_contribution = co2_compressor_annual_cost * co2_compressor_capacity.x
    co2_storage_cost_contribution = co2_storage_annual_cost * co2_storage_capacity.x
    methanol_storage_cost_contribution = methanol_storage_annual_cost * methanol_storage_capacity.x
    variable_cost_contribution = (surplus_penalty * annual_surplus + 
                                battery_charge_penalty * annual_battery_charge +
                                sorbent_cost * sum(dac_CO2[t].x for t in range(time_steps)) +
                                hydrogen_charge_penalty * sum(hydrogen_charge[t].x for t in range(time_steps)) +
                                methanol_charge_penalty * sum(methanol_storage_charge[t].x for t in range(time_steps)))
    
    print(f"\n成本分解：")
    print(f"风电成本：${wind_cost_contribution:.2f} ({wind_cost_contribution/model.objVal*100:.1f}%)")
    print(f"光伏成本：${pv_cost_contribution:.2f} ({pv_cost_contribution/model.objVal*100:.1f}%)")
    print(f"电池能量成本：${battery_energy_cost_contribution:.2f} ({battery_energy_cost_contribution/model.objVal*100:.1f}%)")
    print(f"电池储存成本：${battery_storage_cost_contribution:.2f} ({battery_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"AE电解槽成本：${electrolyzer_AE_cost_contribution:.2f} ({electrolyzer_AE_cost_contribution/model.objVal*100:.1f}%)")
    print(f"PEM电解槽成本：${electrolyzer_PEM_cost_contribution:.2f} ({electrolyzer_PEM_cost_contribution/model.objVal*100:.1f}%)")
    print(f"甲醇合成成本：${methanol_synthesis_cost_contribution:.2f} ({methanol_synthesis_cost_contribution/model.objVal*100:.1f}%)")
    print(f"热泵成本：${heat_pump_cost_contribution:.2f} ({heat_pump_cost_contribution/model.objVal*100:.1f}%)")
    print(f"热储成本：${thermal_storage_cost_contribution:.2f} ({thermal_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"氢储成本：${hydrogen_storage_cost_contribution:.2f} ({hydrogen_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"DAC成本：${dac_cost_contribution:.2f} ({dac_cost_contribution/model.objVal*100:.1f}%)")
    print(f"CO2压缩成本：${co2_compressor_cost_contribution:.2f} ({co2_compressor_cost_contribution/model.objVal*100:.1f}%)")
    print(f"CO2储存成本：${co2_storage_cost_contribution:.2f} ({co2_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"甲醇储存成本：${methanol_storage_cost_contribution:.2f} ({methanol_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"运行变动成本：${variable_cost_contribution:.2f} ({variable_cost_contribution/model.objVal*100:.1f}%)")
    
    print(f"\n甲醇生产平准化成本：${model.objVal/total_methanol_production*1000:.2f} $/t")
else:
    print("模型未找到最优解")
    print(f"求解状态：{model.status}")

In [ ]:
# 衍生平准化成本：LCOE、LCOH、供热LCOHeat、DAC LCO₂
try:
    assert model.status == GRB.OPTIMAL
except Exception:
    print("未检测到最优解，跳过平准化成本派生计算。")
else:
    # 1) 基础年量统计（沿用上一单元格变量）
    time_steps = 8760
    gross_gen_kWh = annual_wind_generation + annual_pv_generation
    net_gen_kWh = max(gross_gen_kWh - annual_surplus, 1e-9)

    # 各用电端合计（kWh）
    elec_for_h2_kWh = sum(electrolyzer_power_AE[t].x + electrolyzer_power_PEM[t].x for t in range(time_steps))
    elec_for_ftl_kWh = sum(FTL_synthesis_power[t].x * FTL_electricity_ratio for t in range(time_steps))
    elec_for_h2_storage_kWh = sum(hydrogen_charge[t].x * hydrogen_storage_elec for t in range(time_steps))
    elec_for_hp_kWh = sum(heat_pump_output[t].x / heat_pump_cop for t in range(time_steps))
    elec_for_dac_kWh = sum(dac_electricity[t].x for t in range(time_steps))
    elec_for_co2_comp_kWh = sum(co2_comp_elec[t].x for t in range(time_steps))

    loads_kWh = (elec_for_h2_kWh + elec_for_ftl_kWh + elec_for_h2_storage_kWh +
                 elec_for_hp_kWh + elec_for_dac_kWh + elec_for_co2_comp_kWh)
    loads_kWh = max(loads_kWh, 1e-9)

    # 2) LCOE（仅计入发电/储能侧固定成本，避免与下游双计）
    lcoe_numerator = (
        wind_annual_cost * wind_capacity.x +
        pv_annual_cost * pv_capacity.x +
        battery_energy_annual_cost * battery_energy_capacity.x +
        battery_storage_annual_cost * battery_storage_capacity.x
    )

    lcoe_gross = lcoe_numerator / max(gross_gen_kWh, 1e-9)
    lcoe_net = lcoe_numerator / net_gen_kWh
    lcoe_load = lcoe_numerator / loads_kWh

    print("\n—— 平准化成本（派生指标） ——")
    print(f"LCOE-发电侧（总发电量基数）: ${lcoe_gross:.4f} $/kWh")
    print(f"LCOE-用电侧（负载基数）:    ${lcoe_load:.4f} $/kWh")
    print(f"LCOE-净供电（剔除弃电）:    ${lcoe_net:.4f} $/kWh")

    # 3) LCOH（$/kg-H2）：电解槽固定成本 + 用电成本（按LCOE-用电侧计价）/ 产氢量
    total_H2_kg = sum(
        electrolyzer_power_AE[t].x * conversion_factor_AE +
        electrolyzer_power_PEM[t].x * conversion_factor_PEM
        for t in range(time_steps)
    )

    if total_H2_kg > 0:
        electrolyzer_fixed_annual = (
            electrolyzer_annual_cost_AE * electrolyzer_capacity_AE.x +
            electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM.x +
            hydrogen_storage_annual_cost * hydrogen_storage_capacity.x
        )
        lcoh_usd_per_kg = (electrolyzer_fixed_annual + elec_for_h2_kWh * lcoe_load) / total_H2_kg
        print(f"平准化制氢成本（LCOH）: ${lcoh_usd_per_kg:.2f} $/kg-H2")
    else:
        print("平准化制氢成本（LCOH）：无产氢，未计算")

    # 4) 供热LCOHeat（$/kWh_th）：热泵+储热固定成本 + 热泵用电成本（按LCOE-用电侧计价）/ 供热量
    total_heat_kWhth = sum(heat_pump_output[t].x for t in range(time_steps))
    if total_heat_kWhth > 0:
        heat_fixed_annual = (
            heat_pump_annual_cost * heat_pump_capacity.x +
            thermal_storage_annual_cost * thermal_storage_capacity.x
        )
        lcoheat_usd_per_kWhth = (heat_fixed_annual + elec_for_hp_kWh * lcoe_load) / total_heat_kWhth
        print(f"平准化供热成本（LCOHeat）: ${lcoheat_usd_per_kWhth:.4f} $/kWh_th  ({lcoheat_usd_per_kWhth*1000:.2f} $/MWh_th)")
    else:
        print("平准化供热成本（LCOHeat）：无供热，未计算")

    # 5) DAC LCO₂（$/t-CO2）：DAC+压缩+储存固定成本 + 用电与供热成本（供热按LCOHeat计价）/ 捕集CO2
    total_CO2_kg = sum(dac_CO2[t].x for t in range(time_steps))
    if total_CO2_kg > 0:
        total_dac_heat_kWhth = sum(dac_heat_input[t].x for t in range(time_steps))
        total_dac_elec_kWh = elec_for_dac_kWh + elec_for_co2_comp_kWh
        dac_fixed_annual = (
            dac_annual_cost * dac_capacity.x +
            co2_compressor_annual_cost * co2_compressor_capacity.x +
            co2_storage_annual_cost * co2_storage_capacity.x
        )
        # sorbent_cost 已在目标函数中按捕集量计入，这里作为变量成本显式加入
        dac_variable_sorbent = sorbent_cost * total_CO2_kg
        # 若未计算出LCOHeat，则改用电价按等效COP折算
        if total_heat_kWhth > 0:
            heat_price = lcoheat_usd_per_kWhth
        else:
            heat_price = lcoe_load * (1.0 / max(heat_pump_cop, 1e-9))
        lcodac_usd_per_tCO2 = (
            dac_fixed_annual + dac_variable_sorbent +
            total_dac_elec_kWh * lcoe_load + total_dac_heat_kWhth * heat_price
        ) / (total_CO2_kg / 1000.0)
        print(f"平准化直接空气捕集成本（DAC，全系统）: ${lcodac_usd_per_tCO2:.2f} $/t-CO2")
    else:
        print("平准化直接空气捕集成本：无CO2捕集量，未计算")

In [ ]:
if model.status == GRB.OPTIMAL:
    # 计算各时段输出与状态曲线
    wind_power_profile = wind_output * wind_capacity.x
    pv_power_profile = pv_output * pv_capacity.x
    # 电解槽实际出力（考虑效率），单位为 kW
    electrolyzer_demand_AE = [(electrolyzer_power_AE[t].x) * electrolyzer_eff_AE for t in range(time_steps)]
    electrolyzer_demand_PEM = [(electrolyzer_power_PEM[t].x) * electrolyzer_eff_PEM for t in range(time_steps)]
    # 甲醇生产曲线，单位为 kg/h（满足小时供需平衡约束）
    methanol_synthesis_profile = [methanol_synthesis_power[t].x for t in range(time_steps)]
    # 甲醇储存状态曲线，单位为 kg（这里不做缩放，直接展示）
    methanol_storage_profile = [methanol_storage_balance[t].x/100 for t in range(time_steps)]
    
    # 电池充放电及电池储能状态
    battery_charge_profile = [battery_charge[t].x for t in range(time_steps)]
    battery_discharge_profile = [battery_discharge[t].x for t in range(time_steps)]
    energy_balance_profile = [energy_balance[t].x for t in range(time_steps)]
    
    # 提取各时刻电力平衡约束的对偶价格（电价）
    electricity_prices = [model.getConstrByName(f"power_balance_{t}").pi for t in range(time_steps)]
    # 对于甲醇供需平衡约束（在小时模式下，从 t=168 开始定义），提取其对偶价格作为甲醇价格
    methanol_prices = [0] * time_steps
    for t in range(168, time_steps):
        constr = model.getConstrByName(f"methanol_balance_{t}")
        if constr is not None:
            methanol_prices[t] = constr.pi
        else:
            methanol_prices[t] = 0

    # 绘制样本周（168小时）时序图，示例取第11周（从第 168*10 小时开始）
    week_start = 168 * 10
    week_end = week_start + 168
    
    battery_charge_week = battery_charge_profile[week_start:week_end]
    battery_discharge_week = battery_discharge_profile[week_start:week_end]
    energy_balance_week = energy_balance_profile[week_start:week_end]
    wind_power_week = wind_power_profile[week_start:week_end]
    pv_power_week = pv_power_profile[week_start:week_end]
    electrolyzer_demand_week_AE = electrolyzer_demand_AE[week_start:week_end]
    electrolyzer_demand_week_PEM = electrolyzer_demand_PEM[week_start:week_end]
    methanol_synthesis_week = methanol_synthesis_profile[week_start:week_end]
    methanol_storage_week = methanol_storage_profile[week_start:week_end]
    electricity_prices_week = electricity_prices[week_start:week_end]
    methanol_prices_week = methanol_prices[week_start:week_end]
    
    plt.figure(figsize=(15, 8))
    plt.plot(wind_power_week, label='Wind Output (kW)', linestyle='-', color='blue')
    plt.plot(pv_power_week, label='PV Output (kW)', linestyle='-', color='orange')
    plt.plot(battery_charge_week, label='Battery Charge (kW)', linestyle='--', color='green')
    plt.plot(battery_discharge_week, label='Battery Discharge (kW)', linestyle='--', color='red')
    plt.plot(electrolyzer_demand_week_AE, label='Electrolyzer Demand AE(kW)', linestyle='-', color='purple')
    plt.plot(electrolyzer_demand_week_PEM, label='Electrolyzer Demand PEM(kW)', linestyle='-', color='yellow')
    plt.plot(methanol_synthesis_week, label='Methanol Synthesis Production (kg/h)', linestyle='-', color='magenta')
    plt.plot(methanol_storage_week, label='Methanol Storage Balance (100kg)', linestyle='-', color='cyan')
    plt.plot(electricity_prices_week, label='Electricity Price ($/kWh)', linestyle='-', color='brown')
    plt.plot(methanol_prices_week, label='Methanol Price ($/kg)', linestyle='-', color='black')
    
    plt.xlabel('Hour')
    plt.ylabel('Value')
    plt.title('Power, Production and Storage Profiles (Sample Week)')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 按月绘图（以每月小时数为分段，近似划分）
    hours_in_month = [0, 744, 1416, 2160, 2880, 3624, 4344, 5088, 5832, 6552, 7296, 8016, 8760]
    month_names = ['January', 'February', 'March', 'April', 'May', 'June', 
                   'July', 'August', 'September', 'October', 'November', 'December']

    for i in range(12):
        start = hours_in_month[i]
        end = hours_in_month[i + 1]
        
        battery_charge_month = battery_charge_profile[start:end]
        battery_discharge_month = battery_discharge_profile[start:end]
        energy_balance_month = energy_balance_profile[start:end]
        wind_power_month = wind_power_profile[start:end]
        pv_power_month = pv_power_profile[start:end]
        electrolyzer_demand_month_AE = electrolyzer_demand_AE[start:end]
        electrolyzer_demand_month_PEM = electrolyzer_demand_PEM[start:end]
        methanol_synthesis_month = methanol_synthesis_profile[start:end]
        methanol_storage_month = methanol_storage_profile[start:end]
        electricity_prices_month = electricity_prices[start:end]
        methanol_prices_month = methanol_prices[start:end]
    
        plt.figure(figsize=(15, 8))
        plt.plot(wind_power_month, label='Wind Output (kW)', linestyle='-', color='blue')
        plt.plot(pv_power_month, label='PV Output (kW)', linestyle='-', color='orange')
        plt.plot(battery_charge_month, label='Battery Charge (kW)', linestyle='--', color='green')
        plt.plot(battery_discharge_month, label='Battery Discharge (kW)', linestyle='--', color='red')
        plt.plot(electrolyzer_demand_month_AE, label='Electrolyzer Demand AE (kW)', linestyle='-', color='purple')
        plt.plot(electrolyzer_demand_month_PEM, label='Electrolyzer Demand PEM (kW)', linestyle='-', color='yellow')
        plt.plot(methanol_synthesis_month, label='Methanol Synthesis Production (kg/h)', linestyle='-', color='magenta')
        plt.plot(methanol_storage_month, label='Methanol Storage Balance (kg)', linestyle='-', color='cyan')
        plt.plot(electricity_prices_month, label='Electricity Price ($/kWh)', linestyle='-', color='brown')
        plt.plot(methanol_prices_month, label='Methanol Price ($/kg)', linestyle='-', color='black')
    
        plt.xlabel('Hour')
        plt.ylabel('Value')
        plt.title(f'Power, Production and Storage Profiles ({month_names[i]})')
        plt.legend()
        plt.grid(True)
        plt.show()
else:
    print("No feasible solution found.")


## 2.2 甲醇系统时序分析

对甲醇生产系统的运行状态进行时序分析，包括：

### 分析内容
- **功率曲线**：风光发电、电解槽功率、甲醇合成功率
- **储能状态**：电池、氢气、甲醇储存动态
- **价格信号**：电力边际价格、甲醇边际价格

### 时间尺度
- 样本周分析（第11周，168小时）
- 逐月分析（12个月完整时序）

### 图表说明
- 实线：主要设备出力
- 虚线：储能充放电
- 价格曲线反映系统供需紧张程度

In [ ]:
# ---------------------------
# Nature 风格配色定义
# ---------------------------
wind_color       = '#4F81BD'  # 柔和蓝
pv_color         = '#C0504D'  # 粉砖红
battery_color    = '#9BBB59'  # 暖绿色

ae_color         = '#8064A2'  # 紫罗兰
pem_color        = '#F79646'  # 橙褐
methanol_color   = '#4BACC6'  # 浅青
bat_charge_color = '#F05542'  # 珊瑚红
hydrogen_color   = '#8F5CC1'  # 淡紫
heatpump_color   = '#4C9F70'  # 松绿色
dac_color        = '#D99694'  # 玫瑰灰
surplus_color    = '#B0B0B0'  # 浅灰
co2_color        = '#696969'  # 深灰

if model.status == GRB.OPTIMAL:
    # 取第11周样本
    week_start = 168 * 10
    week_end   = week_start + 168
    x = np.arange(168)
    
    # 计算各部分
    supply_wind    = np.array([wind_output[t] * wind_capacity.x for t in range(week_start, week_end)])
    supply_pv      = np.array([pv_output[t] * pv_capacity.x     for t in range(week_start, week_end)])
    supply_battery = np.array([battery_discharge[t].x           for t in range(week_start, week_end)])
    
    demand_ae       = np.array([electrolyzer_power_AE[t].x    for t in range(week_start, week_end)])
    demand_pem      = np.array([electrolyzer_power_PEM[t].x   for t in range(week_start, week_end)])
    demand_methanol = np.array([methanol_synthesis_power[t].x * methanol_electricity_ratio
                                for t in range(week_start, week_end)])
    demand_bat_ch   = np.array([battery_charge[t].x            for t in range(week_start, week_end)])
    demand_h2       = np.array([hydrogen_charge[t].x * hydrogen_storage_elec
                                for t in range(week_start, week_end)])
    demand_hp       = np.array([heat_pump_output[t].x / heat_pump_cop
                                for t in range(week_start, week_end)])
    demand_dac      = np.array([dac_electricity[t].x         for t in range(week_start, week_end)])
    demand_surplus  = np.array([surplus[t].x                for t in range(week_start, week_end)])
    demand_co2      = np.array([co2_comp_elec[t].x          for t in range(week_start, week_end)])
    
    plt.figure(figsize=(15, 8))
    # ———— 供给侧 ————
    plt.bar(x, supply_wind,    label='Wind Supply',        color=wind_color)
    plt.bar(x, supply_pv,      bottom=supply_wind,        label='PV Supply',          color=pv_color)
    plt.bar(x, supply_battery, bottom=supply_wind+ supply_pv, label='Battery Discharge', color=battery_color)
    
    # ———— 需求侧 ————
    bottom1 = -(demand_ae)
    plt.bar(x, -demand_ae,       label='Electrolyzer AE',      color=ae_color)
    plt.bar(x, -demand_pem,      bottom=bottom1,              label='Electrolyzer PEM',    color=pem_color)
    
    bottom2 = -(demand_ae + demand_pem)
    plt.bar(x, -demand_methanol, bottom=bottom2,              label='Methanol Synthesis',  color=methanol_color)
    
    bottom3 = -(demand_ae + demand_pem + demand_methanol)
    plt.bar(x, -demand_bat_ch,   bottom=bottom3,              label='Battery Charge',      color=bat_charge_color)
    
    bottom4 = -(demand_ae + demand_pem + demand_methanol + demand_bat_ch)
    plt.bar(x, -demand_h2,       bottom=bottom4,              label='Hydrogen Charge',     color=hydrogen_color)
    
    bottom5 = -(demand_ae + demand_pem + demand_methanol + demand_bat_ch + demand_h2)
    plt.bar(x, -demand_hp,       bottom=bottom5,              label='Heat Pump',           color=heatpump_color)
    
    bottom6 = -(demand_ae + demand_pem + demand_methanol + demand_bat_ch + demand_h2 + demand_hp)
    plt.bar(x, -demand_dac,      bottom=bottom6,              label='DAC Electricity',     color=dac_color)
    
    bottom7 = -(demand_ae + demand_pem + demand_methanol + demand_bat_ch + demand_h2 + demand_hp + demand_dac)
    plt.bar(x, -demand_surplus,  bottom=bottom7,              label='Surplus',             color=surplus_color)
    
    bottom8 = -(demand_ae + demand_pem + demand_methanol + demand_bat_ch + demand_h2 + demand_hp + demand_dac + demand_surplus)
    plt.bar(x, -demand_co2,      bottom=bottom8,              label='CO₂ Compression',     color=co2_color)
    
    plt.axhline(0, color='black', linewidth=0.8)
    plt.xlabel('Hour')
    plt.ylabel('Power (kW)')
    plt.title('Sample Week Power Supply & Demand')
    plt.legend(loc='upper right', ncol=2, frameon=False)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # 月度图同理，只需将上面的 color=xxx 对应替换即可
else:
    print("No feasible solution found.")


## 2.3 电力供需平衡分析（堆积图）

采用Nature期刊风格配色的堆积柱状图，直观展示系统电力供需平衡。

### 可视化设计
- **供给侧**（正值）：风电（蓝）、光伏（红）、电池放电（绿）
- **需求侧**（负值）：
  - 电解槽AE（紫）、PEM（橙）
  - 甲醇合成（青）、电池充电（珊瑚红）
  - 氢气储存、热泵、DAC、弃电等

### 颜色编码
基于Nature期刊标准配色方案，确保：
- 色彩对比度适中，便于识别
- 色盲友好设计
- 印刷适配性良好

### 分析价值
- 直观识别供需匹配情况
- 发现系统瓶颈和优化空间
- 验证储能配置合理性

In [ ]:
# ---------------------------
# 月度堆积柱状图（Nature 风格配色），并保存图片
# ---------------------------
hours_in_month = [0, 744, 1416, 2160, 2880, 3624, 4344, 5088, 5832, 6552, 7296, 8016, 8760]
month_names = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']

for i in range(12):
    start = hours_in_month[i]
    end   = hours_in_month[i + 1]
    x = np.arange(end - start)
    
    # 供给侧
    supply_wind_month    = np.array([wind_output[t] * wind_capacity.x for t in range(start, end)])
    supply_pv_month      = np.array([pv_output[t]   * pv_capacity.x   for t in range(start, end)])
    supply_battery_month = np.array([battery_discharge[t].x         for t in range(start, end)])
    
    # 需求侧
    ae_m   = np.array([electrolyzer_power_AE[t].x    for t in range(start, end)])
    pem_m  = np.array([electrolyzer_power_PEM[t].x   for t in range(start, end)])
    met_m  = np.array([methanol_synthesis_power[t].x * methanol_electricity_ratio 
                       for t in range(start, end)])
    bat_m  = np.array([battery_charge[t].x           for t in range(start, end)])
    h2_m   = np.array([hydrogen_charge[t].x * hydrogen_storage_elec 
                       for t in range(start, end)])
    hp_m   = np.array([heat_pump_output[t].x / heat_pump_cop 
                       for t in range(start, end)])
    dac_m  = np.array([dac_electricity[t].x         for t in range(start, end)])
    sur_m  = np.array([surplus[t].x                for t in range(start, end)])
    co2_m  = np.array([co2_comp_elec[t].x          for t in range(start, end)])
    
    # 画图
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # —— 供给堆积 —— 
    ax.bar(x, supply_wind_month,    label='Wind Supply',        color=wind_color)
    ax.bar(x, supply_pv_month,      bottom=supply_wind_month,  label='PV Supply',          color=pv_color)
    bottom_sup = supply_wind_month + supply_pv_month
    ax.bar(x, supply_battery_month, bottom=bottom_sup,          label='Battery Discharge',  color=battery_color)
    
    # —— 需求堆积 —— 
    bottom_dem = np.zeros_like(x, dtype=float)
    ax.bar(x, -ae_m,  bottom=bottom_dem,             label='Electrolyzer AE',       color=ae_color)
    bottom_dem -= ae_m
    
    ax.bar(x, -pem_m, bottom=bottom_dem,             label='Electrolyzer PEM',      color=pem_color)
    bottom_dem -= pem_m
    
    ax.bar(x, -met_m, bottom=bottom_dem,             label='Methanol Synthesis',     color=methanol_color)
    bottom_dem -= met_m
    
    ax.bar(x, -bat_m, bottom=bottom_dem,             label='Battery Charge',         color=bat_charge_color)
    bottom_dem -= bat_m
    
    ax.bar(x, -h2_m,  bottom=bottom_dem,             label='Hydrogen Charge',        color=hydrogen_color)
    bottom_dem -= h2_m
    
    ax.bar(x, -hp_m,  bottom=bottom_dem,             label='Heat Pump',              color=heatpump_color)
    bottom_dem -= hp_m
    
    ax.bar(x, -dac_m, bottom=bottom_dem,             label='DAC Electricity',        color=dac_color)
    bottom_dem -= dac_m
    
    ax.bar(x, -sur_m, bottom=bottom_dem,             label='Surplus',                color=surplus_color)
    bottom_dem -= sur_m
    
    ax.bar(x, -co2_m, bottom=bottom_dem,             label='CO₂ Compression',        color=co2_color)
    
    # 美化
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Hour')
    ax.set_ylabel('Power (kW)')
    ax.set_title(f'Stacked Power Supply & Demand ({month_names[i]})', fontsize=14)
    ax.legend(loc='upper right', ncol=2, frameon=False)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    
    # —— 保存当前月度图到文件 —— 
    filename = f'stacked_power_{month_names[i]}.png'
    fig.savefig(filename, dpi=300, bbox_inches='tight')
    print(f'Saved figure for {month_names[i]} as {filename}')
    
    plt.show()


## 2.4 月度电力平衡分析与图表保存

生成12个月的详细电力供需平衡图表，并自动保存为高分辨率图片文件。

### 功能特点
- **逐月分析**：按月生成供需平衡图
- **自动保存**：300 DPI高分辨率PNG格式
- **文件命名**：stacked_power_[月份].png

### 输出文件
```
stacked_power_January.png
stacked_power_February.png
...
stacked_power_December.png
```

### 应用场景
- 学术论文插图
- 项目报告图表
- 系统性能展示
- 季节性特征分析

In [ ]:
if model.status == GRB.OPTIMAL:
    # ---------------------------
    # 样本周（168小时）氢能供需堆积柱状图
    # ---------------------------
    week_start = 168 * 10   # 例如取第11周
    week_end = week_start + 168
    x = np.arange(week_end - week_start)
    
    # 供给侧：
    # 1. 氢气生产：由AE和PEM电解槽产生的氢气（单位：kg/h）
    supply_h2_production = np.array([
        (electrolyzer_power_AE[t].x * electrolyzer_eff_AE + electrolyzer_power_PEM[t].x * electrolyzer_eff_PEM) * 3.6 / 120
        for t in range(week_start, week_end)
    ])
    # 2. 氢气放电：氢气储存释放的氢气（单位：kg/h）
    supply_h2_discharge = np.array([hydrogen_discharge[t].x for t in range(week_start, week_end)])

    demand_methanol_production = np.array([
        methanol_synthesis_power[t].x * methanol_hydrogen_ratio
        for t in range(week_start, week_end)
    ])
    demand_h2_charge = np.array([hydrogen_charge[t].x for t in range(week_start, week_end)])

    plt.figure(figsize=(15, 8))
    # 绘制供给侧堆积柱状图（正值）
    bar1 = plt.bar(x, supply_h2_production, label='Hydrogen Production', color='blue')
    bar2 = plt.bar(x, supply_h2_discharge, bottom=supply_h2_production, label='Hydrogen Discharge', color='cyan')

    bar3 = plt.bar(x, -demand_methanol_production, label='Hydrogen Demand', color='purple')
    bar4 = plt.bar(x, -demand_h2_charge, bottom = -demand_methanol_production, label='Hydrogen Charge', color='red')
    
    plt.axhline(0, color='black', linewidth=0.8)
    plt.xlabel('Hour')
    plt.ylabel('Hydrogen Flow (kg/h)')
    plt.title('Stacked Hydrogen Supply (Positive) & Demand (Negative) - Sample Week')
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.show()
    
    # ---------------------------
    # 按月氢能供需堆积柱状图
    # ---------------------------
    # 近似每月的小时数划分（可根据实际情况调整）
    hours_in_month = [0, 744, 1416, 2160, 2880, 3624, 4344, 5088, 5832, 6552, 7296, 8016, 8760]
    month_names = ['January', 'February', 'March', 'April', 'May', 'June', 
                   'July', 'August', 'September', 'October', 'November', 'December']
    
    for i in range(12):
        start = hours_in_month[i]
        end = hours_in_month[i+1]
        x_month = np.arange(end - start)
        
        # 供给侧数据
        supply_h2_production_month = np.array([
            (electrolyzer_power_AE[t].x * electrolyzer_eff_AE + electrolyzer_power_PEM[t].x * electrolyzer_eff_PEM) * 3.6 / 120
            for t in range(start, end)
        ])
        supply_h2_discharge_month = np.array([hydrogen_discharge[t].x for t in range(start, end)])

        demand_methanol_production_month = np.array([
        methanol_synthesis_power[t].x * methanol_hydrogen_ratio
        for t in range(start, end)
        ])
        demand_h2_charge_month = np.array([hydrogen_charge[t].x for t in range(start, end)])
        
        plt.figure(figsize=(15, 8))
        plt.bar(x_month, supply_h2_production_month, label='Hydrogen Production', color='blue')
        plt.bar(x_month, supply_h2_discharge_month, bottom=supply_h2_production_month, label='Hydrogen Discharge', color='cyan')

        plt.bar(x_month, -demand_methanol_production_month, label='Hydrogen Demand', color='purple')
        plt.bar(x_month, -demand_h2_charge_month, -demand_methanol_production_month, label='Hydrogen Charge', color='red')
        
        plt.axhline(0, color='black', linewidth=0.8)
        plt.xlabel('Hour')
        plt.ylabel('Hydrogen Flow (kg/h)')
        plt.title(f'Stacked Hydrogen Supply & Demand ({month_names[i]})')
        plt.legend(loc='upper right')
        plt.grid(True)
        plt.show()
else:
    print("No feasible solution found.")


## 2.5 氢气供需平衡分析

专门针对氢气子系统的供需平衡分析，展示氢气流动的动态特征。

### 分析维度
**供给侧**：
- 氢气生产：AE+PEM电解槽制氢（kg/h）
- 氢气放电：储氢罐释放氢气

**需求侧**：
- 甲醇生产：合成反应氢气消耗
- 氢气充电：储氢罐充入氢气

### 转换系数
- 电解效率转换：kW → kg H2/h
- AE电解槽：eff_AE × 3.6/120
- PEM电解槽：eff_PEM × 3.6/120

### 可视化特点
- 正值：氢气供给（蓝色调）
- 负值：氢气需求（暖色调）
- 零轴分界线清晰标识

### 应用价值
- 氢气储存容量优化
- 电解槽配比优化
- 氢气系统瓶颈识别

## 3.1 合成氨

In [ ]:
# =============================================================================
# 合成氨优化（拆分电池与区分AE/PEM电解效率）
# =============================================================================
import os
os.environ["GRB_WLSACCESSID"] = "e53ba8c8-d918-4dbc-8b70-719fdccb385c"
os.environ["GRB_WLSSECRET"] = "0ee54d8d-f1c2-4642-b8de-f37edffd7059"
os.environ["GRB_LICENSEID"] = "2570583"

import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB

# =============================================================================
# 1. 数据加载
# =============================================================================
wind_solar_data = pd.read_csv('CF_NM_2023.csv')
pv_output   = wind_solar_data.iloc[:, 0].values
wind_output = wind_solar_data.iloc[:, 1].values

# =============================================================================
# 2. 使用全局参数
# =============================================================================
# 所有参数已在全局配置单元格中定义，这里直接使用全局变量

# =============================================================================
# 3. 模型创建与变量定义
# =============================================================================
model = Model("AmmoniaOptimization_SplitBattery_AEPem")
model.setParam('TimeLimit', 1000)
model.setParam('MIPFocus', 1)

# 强制用 Barrier，跳过 Crossover，把收敛精度放宽到 1e-6 ，并行 8 线程
model.setParam('Method', 2)        # 2 = Barrier
model.setParam('Crossover', 0)     # 关闭回退
model.setParam('BarConvTol', 1e-6) # 默认 1e-8 → 1e-6
model.setParam('Threads', 1)

# 选用 Homogeneous KKT 因式分解策略，通常更稳定
model.setParam('BarHomogeneous', 1)

# 关闭过度 Presolve（有时 Aggressive 反而慢）
model.setParam('Presolve', 1)      # 1 = 默认

time_steps = 8760

# 3.1 装机容量决策变量
wind_capacity = model.addVar(lb=0, name="wind_capacity")
pv_capacity   = model.addVar(lb=0, name="pv_capacity")
battery_energy_capacity = model.addVar(lb=0, name="battery_energy_capacity")
battery_storage_capacity = model.addVar(lb=0, name="battery_storage_capacity")
electrolyzer_capacity_AE = model.addVar(lb=0, name="electrolyzer_capacity_AE")
electrolyzer_capacity_PEM = model.addVar(lb=0, name="electrolyzer_capacity_PEM")
ammonia_synthesis_capacity = model.addVar(lb=0, name="ammonia_synthesis_capacity")
hydrogen_storage_capacity = model.addVar(lb=0, name="hydrogen_storage_capacity")
ammonia_storage_capacity = model.addVar(lb=0, name="ammonia_storage_capacity")

# 3.2 时变变量（每小时）
energy_balance = model.addVars(time_steps, lb=0, name="energy_balance")
battery_charge = model.addVars(time_steps, lb=0, name="battery_charge")
battery_discharge = model.addVars(time_steps, lb=0, name="battery_discharge")

electrolyzer_power_AE = model.addVars(time_steps, lb=0, name="electrolyzer_power_AE")
electrolyzer_power_PEM = model.addVars(time_steps, lb=0, name="electrolyzer_power_PEM")

ammonia_synthesis_power = model.addVars(time_steps, lb=0, name="ammonia_synthesis_power")
ammonia_storage_balance = model.addVars(time_steps, lb=0, name="ammonia_storage_balance")
ammonia_storage_charge = model.addVars(time_steps, lb=0, name="ammonia_storage_charge")
ammonia_storage_discharge = model.addVars(time_steps, lb=0, name="ammonia_storage_discharge")

hydrogen_storage_balance = model.addVars(time_steps, lb=0, name="hydrogen_storage_balance")
hydrogen_charge = model.addVars(time_steps, lb=0, name="hydrogen_charge")
hydrogen_discharge = model.addVars(time_steps, lb=0, name="hydrogen_discharge")

surplus = model.addVars(time_steps, lb=0, name="surplus")

# =============================================================================
# 4. 目标函数
# =============================================================================
model.setObjective(
    wind_annual_cost * wind_capacity +
    pv_annual_cost * pv_capacity +
    battery_energy_annual_cost * battery_energy_capacity +
    battery_storage_annual_cost * battery_storage_capacity +
    electrolyzer_annual_cost_AE * electrolyzer_capacity_AE +
    electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM +
    ammonia_synthesis_annual_cost * ammonia_synthesis_capacity +
    hydrogen_storage_annual_cost * hydrogen_storage_capacity +
    ammonia_storage_annual_cost * ammonia_storage_capacity +
    surplus_penalty * gp.quicksum(surplus[t] for t in range(time_steps)) +
    battery_charge_penalty * gp.quicksum(battery_charge[t] for t in range(time_steps)) +
    battery_discharge_penalty * gp.quicksum(battery_discharge[t] for t in range(time_steps)) +
    hydrogen_charge_penalty * gp.quicksum(hydrogen_charge[t] for t in range(time_steps)),
    GRB.MINIMIZE
)

# =============================================================================
# 5. 约束条件
# =============================================================================
# 5.1 逐时灵活性
for t in range(1, time_steps):
    model.addConstr(electrolyzer_power_AE[t]   - electrolyzer_power_AE[t-1]   <= Flex_down * electrolyzer_capacity_AE,
                    name=f"electrolyzer_AE_ramp_up_{t}")
    model.addConstr(electrolyzer_power_AE[t]   - electrolyzer_power_AE[t-1]   >= -Flex_down * electrolyzer_capacity_AE,
                    name=f"electrolyzer_AE_ramp_down_{t}")
    model.addConstr(ammonia_synthesis_power[t] - ammonia_synthesis_power[t-1] <= Flex_mid  * ammonia_synthesis_capacity,
                    name=f"ammonia_ramp_up_{t}")
    model.addConstr(ammonia_synthesis_power[t] - ammonia_synthesis_power[t-1] >= -Flex_down * ammonia_synthesis_capacity,
                    name=f"ammonia_ramp_down_{t}")

# 5.2 合成氨供需平衡约束（含氨储存充放，支持不同时间聚合尺度）
start_index = 168  # 避免初始不稳定阶段
if aggregation_mode == "hourly":
    for t in range(start_index, time_steps):
        model.addConstr(
            ammonia_synthesis_power[t] + ammonia_storage_discharge[t] ==
            hourly_ammonia_demand + ammonia_storage_charge[t],
            name=f"ammonia_balance_{t}"
        )
elif aggregation_mode == "3hourly":
    interval = 3
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
            ==
            hourly_ammonia_demand * interval +
            gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
            name=f"ammonia_balance_3hourly_{i}"
        )
elif aggregation_mode == "daily":
    interval = 24
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
            >=
            0.8 * hourly_ammonia_demand * interval +
            gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
            name=f"ammonia_balance_daily_{i}"
        )
elif aggregation_mode == "weekly":
    interval = 168  # 一周168小时
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
            ==
            hourly_ammonia_demand * interval +
            gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
            name=f"ammonia_balance_weekly_{i}"
        )
elif aggregation_mode == "monthly":
    # 近似每月730小时（可根据实际情况调整）
    interval = 730
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
            ==
            hourly_ammonia_demand * interval +
            gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
            name=f"ammonia_balance_monthly_{i}"
        )
elif aggregation_mode == "annual":
    model.addConstr(
        gp.quicksum(ammonia_synthesis_power[t] for t in range(start_index, time_steps)) +
        gp.quicksum(ammonia_storage_discharge[t] for t in range(start_index, time_steps))
        ==
        hourly_ammonia_demand * (time_steps - start_index) +
        gp.quicksum(ammonia_storage_charge[t] for t in range(start_index, time_steps)),
        name="ammonia_balance_annual"
    )
else:
    raise ValueError("Unsupported aggregation_mode value!")

# 附加：年度合成氨生产总量约束
model.addConstr(
    gp.quicksum(ammonia_synthesis_power[t] for t in range(time_steps)) >= hourly_ammonia_demand * 8760,
    name="annual_ammonia_production_constraint"
)

# 5.3 电力与存储平衡
conversion_factor_AE  = electrolyzer_eff_AE  * 3.6 / 120
conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

for t in range(time_steps):
    wind_power = wind_output[t] * wind_capacity
    pv_power   = pv_output[t]   * pv_capacity
    
    # 电池储能约束
    model.addConstr(energy_balance[t] <= battery_storage_capacity,
                    name=f"energy_balance_limit_{t}")
    model.addConstr(battery_charge[t] + battery_discharge[t] <= battery_energy_capacity,
                    name=f"battery_charge_discharge_limit_{t}")

    # 动态电池平衡
    if t == 0:
        model.addConstr(
            energy_balance[t] == battery_charge[t] * battery_charge_efficiency
                                - battery_discharge[t] / battery_discharge_efficiency,
            name=f"battery_balance_{t}" )
    else:
        model.addConstr(
            energy_balance[t] == energy_balance[t-1] +
                                battery_charge[t] * battery_charge_efficiency
                              - battery_discharge[t] / battery_discharge_efficiency,
            name=f"battery_balance_{t}" )

    # 氢气存储约束
    model.addConstr(hydrogen_storage_balance[t] <= hydrogen_storage_capacity,
                    name=f"hydrogen_storage_capacity_limit_{t}")
    model.addConstr(hydrogen_charge[t] + hydrogen_discharge[t] <= hydrogen_storage_capacity,
                    name=f"hydrogen_charge_limit_{t}")

    # 氢气存储动态平衡
    if t == 0:
        model.addConstr(
            hydrogen_storage_balance[t] == hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
            name=f"hydrogen_balance_{t}"
        )
    else:
        model.addConstr(
            hydrogen_storage_balance[t] == hydrogen_storage_balance[t-1] + hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
            name=f"hydrogen_balance_{t}"
        )
    
    # 氢气供需平衡：电解槽产氢（转换系数 3.6/120）加上氢气释放，
    # 需满足合成氨生产所需的氢气（按氢消耗比）加上充氢量

    # 氨储存平衡
    model.addConstr(ammonia_storage_balance[t] == (ammonia_storage_balance[t-1] if t>0 else ammonia_storage_charge[0])
                              + ammonia_storage_charge[t] - ammonia_storage_discharge[t],
                    name=f"ammonia_storage_balance_{t}")
    model.addConstr(ammonia_storage_balance[t] <= ammonia_storage_capacity,
                    name=f"ammonia_storage_capacity_limit_{t}")
    
    # 氨储存：确保合成氨生产加上储存放出不少于储入量
    model.addConstr(
        ammonia_synthesis_power[t] + ammonia_storage_discharge[t] >= ammonia_storage_charge[t],
        name=f"ammonia_min_balance_{t}"
    )
    
    # 电力平衡
    model.addConstr(
        wind_power + pv_power + battery_discharge[t] ==
        electrolyzer_power_AE[t] + electrolyzer_power_PEM[t] +
        battery_charge[t] + hydrogen_charge[t] * hydrogen_storage_elec +
        ammonia_synthesis_power[t] * ammonia_electricity_ratio + surplus[t],
        name=f"power_balance_{t}" )

    # 氢气供需平衡
    model.addConstr(
        electrolyzer_power_AE[t] * conversion_factor_AE +
        electrolyzer_power_PEM[t] * conversion_factor_PEM +
        hydrogen_discharge[t] >=
        ammonia_synthesis_power[t] * ammonia_hydrogen_ratio + hydrogen_charge[t],
        name=f"hydrogen_PD_balance_{t}" )

# 5.4 容量与最小运行
model.addConstrs((electrolyzer_capacity_AE >= electrolyzer_power_AE[t] for t in range(time_steps)), name="electrolyzer_AE_capacity")
model.addConstrs((electrolyzer_capacity_PEM >= electrolyzer_power_PEM[t] for t in range(time_steps)), name="electrolyzer_PEM_capacity")
model.addConstrs((electrolyzer_power_AE[t] >= 0.1 * electrolyzer_capacity_AE for t in range(start_index, time_steps)), name="electrolyzer_AE_min")
model.addConstrs((ammonia_synthesis_capacity >= ammonia_synthesis_power[t] for t in range(time_steps)), name="ammonia_capacity")
model.addConstrs((ammonia_synthesis_power[t] >= 0.5 * ammonia_synthesis_capacity for t in range(start_index, time_steps)), name="ammonia_min") 
# Source:DEA 2024 data_sheets_for_renewable_fuels 5.4 Hydrogen to Ammonia for 2040 estimated 

# AE电解需配备电池能量：
model.addConstr(battery_energy_capacity >= 0.5 * electrolyzer_capacity_AE, name="min_battery_for_AE")

# =============================================================================
# 6. 求解并输出结果
# =============================================================================
model.optimize()

if model.status == GRB.OPTIMAL:
    print(f"最优目标值：${model.objVal:.2f}")
    print("\n装机容量决策：")
    print(f"风电装机容量：{wind_capacity.x:.2f} kW")
    print(f"光伏装机容量：{pv_capacity.x:.2f} kW")
    print(f"电池能量功率：{battery_energy_capacity.x:.2f} kW")
    print(f"电池储存容量：{battery_storage_capacity.x:.2f} kWh")
    print(f"电解槽装机(AE)：{electrolyzer_capacity_AE.x:.2f} kW")
    print(f"电解槽装机(PEM)：{electrolyzer_capacity_PEM.x:.2f} kW")
    print(f"合成氨装机：{ammonia_synthesis_capacity.x:.2f} kg/h")
    print(f"氢气储存容量：{hydrogen_storage_capacity.x:.2f} kg")
    print(f"氨气储存容量：{ammonia_storage_capacity.x:.2f} kg")
    
    conv_AE  = conversion_factor_AE
    conv_PEM = conversion_factor_PEM
    total_hydrogen_production = sum(
        electrolyzer_power_AE[t].x * conv_AE +
        electrolyzer_power_PEM[t].x * conv_PEM
        for t in range(time_steps)
    )

    # 年产量汇总
    total_ammonia_production = sum(ammonia_synthesis_power[t].x for t in range(time_steps))
    print(f"\n年合成氨产量：{total_ammonia_production:.2f} kg")
    print(f"年产氢量：{sum(electrolyzer_power_AE[t].x * conv_AE + electrolyzer_power_PEM[t].x * conv_PEM for t in range(time_steps)):.2f} kg")
    
    # 设备利用率（年均运行时间除以8760）
    avg_wind_power = sum(wind_output[t] * wind_capacity.x for t in range(time_steps)) / 8760
    avg_pv_power = sum(pv_output[t] * pv_capacity.x for t in range(time_steps)) / 8760
    avg_AE_power = sum(electrolyzer_power_AE[t].x for t in range(time_steps)) / 8760
    avg_PEM_power = sum(electrolyzer_power_PEM[t].x for t in range(time_steps)) / 8760
    avg_ammonia_power = sum(ammonia_synthesis_power[t].x for t in range(time_steps)) / 8760
    
    print(f"\n设备利用率：")
    print(f"风电平均出力：{avg_wind_power:.2f} kW (容量系数: {avg_wind_power/wind_capacity.x*100:.1f}%)")
    print(f"光伏平均出力：{avg_pv_power:.2f} kW (容量系数: {avg_pv_power/pv_capacity.x*100:.1f}%)")
    if electrolyzer_capacity_AE.x > 0:
        print(f"AE电解槽平均功率：{avg_AE_power:.2f} kW (利用率: {avg_AE_power/electrolyzer_capacity_AE.x*100:.1f}%)")
    if electrolyzer_capacity_PEM.x > 0:
        print(f"PEM电解槽平均功率：{avg_PEM_power:.2f} kW (利用率: {avg_PEM_power/electrolyzer_capacity_PEM.x*100:.1f}%)")
    if ammonia_synthesis_capacity.x > 0:
        print(f"合成氨平均功率：{avg_ammonia_power:.2f} kg/h (利用率: {avg_ammonia_power/ammonia_synthesis_capacity.x*100:.1f}%)")
    
    # 计算年发电量和多余电量
    annual_wind_generation = sum(wind_output[t] * wind_capacity.x for t in range(time_steps))
    annual_pv_generation = sum(pv_output[t] * pv_capacity.x for t in range(time_steps))
    annual_surplus = sum(surplus[t].x for t in range(time_steps))
    annual_battery_charge = sum(battery_charge[t].x for t in range(time_steps))
    annual_battery_discharge = sum(battery_discharge[t].x for t in range(time_steps))
    
    print(f"\n能量统计：")
    print(f"年风电发电量：{annual_wind_generation:.2f} kWh")
    print(f"年光伏发电量：{annual_pv_generation:.2f} kWh")
    print(f"年总发电量：{annual_wind_generation + annual_pv_generation:.2f} kWh")
    print(f"年电池充电量：{annual_battery_charge:.2f} kWh")
    print(f"年电池放电量：{annual_battery_discharge:.2f} kWh")
    print(f"年多余电量：{annual_surplus:.2f} kWh ({annual_surplus/(annual_wind_generation + annual_pv_generation)*100:.2f}%)")
    
    # 成本分解分析
    wind_cost_contribution = wind_annual_cost * wind_capacity.x
    pv_cost_contribution = pv_annual_cost * pv_capacity.x
    battery_energy_cost_contribution = battery_energy_annual_cost * battery_energy_capacity.x
    battery_storage_cost_contribution = battery_storage_annual_cost * battery_storage_capacity.x
    electrolyzer_AE_cost_contribution = electrolyzer_annual_cost_AE * electrolyzer_capacity_AE.x
    electrolyzer_PEM_cost_contribution = electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM.x
    ammonia_synthesis_cost_contribution = ammonia_synthesis_annual_cost * ammonia_synthesis_capacity.x
    hydrogen_storage_cost_contribution = hydrogen_storage_annual_cost * hydrogen_storage_capacity.x
    ammonia_storage_cost_contribution = ammonia_storage_annual_cost * ammonia_storage_capacity.x
    variable_cost_contribution = (surplus_penalty * annual_surplus + 
                                battery_charge_penalty * annual_battery_charge +
                                battery_discharge_penalty * annual_battery_discharge +
                                hydrogen_charge_penalty * sum(hydrogen_charge[t].x for t in range(time_steps)))
    # 平准化电价（$/kWh）
    levelized_Elec_cost = (
        wind_annual_cost * wind_capacity.x +
        pv_annual_cost   * pv_capacity.x +
        battery_energy_annual_cost  * battery_energy_capacity.x +
        battery_storage_annual_cost * battery_storage_capacity.x +
        battery_charge_penalty   * sum(battery_charge[t].x   for t in range(time_steps)) +
        battery_discharge_penalty* sum(battery_discharge[t].x for t in range(time_steps))
    ) / total_Elec_generation

    print(f"\n成本分解：")
    print(f"风电成本：${wind_cost_contribution:.2f} ({wind_cost_contribution/model.objVal*100:.1f}%)")
    print(f"光伏成本：${pv_cost_contribution:.2f} ({pv_cost_contribution/model.objVal*100:.1f}%)")
    print(f"电池能量成本：${battery_energy_cost_contribution:.2f} ({battery_energy_cost_contribution/model.objVal*100:.1f}%)")
    print(f"电池储存成本：${battery_storage_cost_contribution:.2f} ({battery_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"AE电解槽成本：${electrolyzer_AE_cost_contribution:.2f} ({electrolyzer_AE_cost_contribution/model.objVal*100:.1f}%)")
    print(f"PEM电解槽成本：${electrolyzer_PEM_cost_contribution:.2f} ({electrolyzer_PEM_cost_contribution/model.objVal*100:.1f}%)")
    print(f"合成氨成本：${ammonia_synthesis_cost_contribution:.2f} ({ammonia_synthesis_cost_contribution/model.objVal*100:.1f}%)")
    print(f"氢储成本：${hydrogen_storage_cost_contribution:.2f} ({hydrogen_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"氨储成本：${ammonia_storage_cost_contribution:.2f} ({ammonia_storage_cost_contribution/model.objVal*100:.1f}%)")
    print(f"运行变动成本：${variable_cost_contribution:.2f} ({variable_cost_contribution/model.objVal*100:.1f}%)")
    print(f"Levelized Elec cost:           ${levelized_Elec_cost:.2f} per kWh")
    print(f"\n合成氨平准化成本：${model.objVal/total_ammonia_production*1000:.2f} $/t")
else:
    print("模型未找到最优解")
    print(f"求解状态：{model.status}")

# ✅ 参数统一重构完成

## 重构说明
所有优化模型已完成参数重构，实现了统一的参数管理：

### 🎯 重构目标
- **统一参数管理**：所有技术经济参数统一定义在顶部全局配置单元格
- **避免重复定义**：消除各模型中的冗余参数定义
- **便于参数调整**：只需在一处修改参数，即可影响所有相关模型
- **提高一致性**：确保所有模型使用相同的参数基准

### 📋 已完成模型
1. **FT液体燃料模型** - 移除本地参数，引用全局配置
2. **甲醇生产模型** - 移除本地参数，引用全局配置  
3. **合成氨模型** - 移除本地参数，引用全局配置

### 🔧 使用方法
1. **修改参数**：在顶部全局配置单元格中调整所需参数
2. **运行模型**：依次执行各个优化模型即可使用新参数
3. **对比分析**：可快速进行不同参数情景下的对比分析

### 📊 全局参数包含
- 可再生能源成本参数（风电、光伏）
- 储能系统参数（电池功率/容量成本）
- 电解制氢参数（AE/PEM电解槽）
- 化学合成参数（FTL、甲醇、氨气合成）
- 存储系统参数（氢气、CO2、产品储存）
- DAC及热管理参数
- 经济参数（折现率、设备寿命）

---

# 3. 合成氨生产优化模型

## 3.1 模型概述
基于Haber-Bosch工艺的可再生能源制氨系统优化模型。

### 主要设备
- **可再生能源**：风电、光伏发电系统
- **电解制氢**：AE电解槽、PEM电解槽（区分效率）
- **储能系统**：锂电池（功率/容量解耦建模）
- **合成氨装置**：N2 + 3H2 → 2NH3
- **储存系统**：氢气储存、氨储存

### 关键参数
- **氨合成氢耗**：0.18 kg H2/kg NH3
- **氨合成电耗**：0.315 kWh/kg NH3  
- **氨合成投资**：6500 $/(kg NH3/h)
- **最小负荷**：50%（行业实际约束）

### 技术特点
- 不需要CO2，工艺链相对简单
- 氨储存密度高，便于长期储存
- 既是化工原料又是储氢载体

### 目标
最小化制氨系统年化总成本，满足年产10吨氨需求

## 3.1 合成氨（向外输电）

In [ ]:
# =============================================================================
# 合成氨优化（拆分电池与区分AE/PEM电解效率）
# =============================================================================
import os
os.environ["GRB_WLSACCESSID"] = "e53ba8c8-d918-4dbc-8b70-719fdccb385c"
os.environ["GRB_WLSSECRET"] = "0ee54d8d-f1c2-4642-b8de-f37edffd7059"
os.environ["GRB_LICENSEID"] = "2570583"


import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB

# =============================================================================
# 1. 数据加载
# =============================================================================
wind_solar_data = pd.read_csv('CF_NM_2023.csv')
pv_output   = wind_solar_data.iloc[:, 0].values
wind_output = wind_solar_data.iloc[:, 1].values

# =============================================================================
# 2. 参数定义
# =============================================================================
electricity_cost = 0.15

# 时间聚合可选："hourly","3hourly","daily","weekly","monthly","annual"
aggregation_mode = "daily"

# =============================================================================
# 3. 模型创建与变量定义
# =============================================================================
model = Model("AmmoniaOptimization_SplitBattery_AEPem")
model.setParam('TimeLimit', 1000)
model.setParam('MIPFocus', 1)

# 强制用 Barrier，跳过 Crossover，把收敛精度放宽到 1e-6 ，并行 8 线程
model.setParam('Method', 2)        # 2 = Barrier
model.setParam('Crossover', 0)     # 关闭回退
model.setParam('BarConvTol', 1e-6) # 默认 1e-8 → 1e-6
model.setParam('Threads', 1)

# 选用 Homogeneous KKT 因式分解策略，通常更稳定
model.setParam('BarHomogeneous', 1)

# 关闭过度 Presolve（有时 Aggressive 反而慢）
model.setParam('Presolve', 1)      # 1 = 默认

# 3.1 装机容量决策变量
wind_capacity     = model.addVar(lb=0, name="wind_capacity")
pv_capacity       = model.addVar(lb=0, name="pv_capacity")
battery_energy_capacity  = model.addVar(lb=0, name="battery_energy_capacity")
battery_storage_capacity = model.addVar(lb=0, name="battery_storage_capacity")
electrolyzer_capacity_AE  = model.addVar(lb=0, name="electrolyzer_capacity_AE")
electrolyzer_capacity_PEM = model.addVar(lb=0, name="electrolyzer_capacity_PEM")
ammonia_synthesis_capacity = model.addVar(lb=0, name="ammonia_synthesis_capacity")
hydrogen_storage_capacity  = model.addVar(lb=0, name="hydrogen_storage_capacity")
ammonia_storage_capacity   = model.addVar(lb=0, name="ammonia_storage_capacity")

time_steps = 8760

# 3.2 时间步变量
energy_balance           = model.addVars(time_steps, lb=0, name="energy_balance")
battery_charge          = model.addVars(time_steps, lb=0, name="battery_charge")
battery_discharge       = model.addVars(time_steps, lb=0, name="battery_discharge")
electrolyzer_power_AE   = model.addVars(time_steps, lb=0, name="electrolyzer_power_AE")
electrolyzer_power_PEM  = model.addVars(time_steps, lb=0, name="electrolyzer_power_PEM")
ammonia_synthesis_power = model.addVars(time_steps, lb=0, name="ammonia_synthesis_power")
surplus                 = model.addVars(time_steps, lb=0, name="surplus")
hydrogen_storage_balance = model.addVars(time_steps, lb=0, name="hydrogen_storage_balance")
hydrogen_charge          = model.addVars(time_steps, lb=0, name="hydrogen_charge")
hydrogen_discharge       = model.addVars(time_steps, lb=0, name="hydrogen_discharge")
ammonia_storage_balance  = model.addVars(time_steps, lb=0, name="ammonia_storage_balance")
ammonia_storage_charge   = model.addVars(time_steps, lb=0, name="ammonia_storage_charge")
ammonia_storage_discharge= model.addVars(time_steps, lb=0, name="ammonia_storage_discharge")

elec_demand = model.addVars(time_steps, lb=0, name="elec_demand")

# =============================================================================
# 4. 目标函数
# =============================================================================
model.setObjective(
    wind_annual_cost * wind_capacity +
    pv_annual_cost * pv_capacity +
    battery_energy_annual_cost * battery_energy_capacity +
    battery_storage_annual_cost * battery_storage_capacity +
    electrolyzer_annual_cost_AE * electrolyzer_capacity_AE +
    electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM +
    ammonia_synthesis_annual_cost * ammonia_synthesis_capacity +
    hydrogen_storage_annual_cost * hydrogen_storage_capacity +
    ammonia_storage_annual_cost * ammonia_storage_capacity +
    surplus_penalty * gp.quicksum(surplus[t] for t in range(time_steps)) +
    elec_demand_penalty * gp.quicksum(elec_demand[t] for t in range(time_steps)) +
    battery_charge_penalty * gp.quicksum(battery_charge[t] for t in range(time_steps)) +
    battery_discharge_penalty * gp.quicksum(battery_discharge[t] for t in range(time_steps)) +
    hydrogen_charge_penalty * gp.quicksum(hydrogen_charge[t] for t in range(time_steps)),
    GRB.MINIMIZE
)

# =============================================================================
# 5. 约束条件
# =============================================================================
# 5.1 逐时灵活性
for t in range(1, time_steps):
    model.addConstr(electrolyzer_power_AE[t]   - electrolyzer_power_AE[t-1]   <= Flex_down * electrolyzer_capacity_AE,
                    name=f"electrolyzer_AE_ramp_up_{t}")
    model.addConstr(electrolyzer_power_AE[t]   - electrolyzer_power_AE[t-1]   >= -Flex_down * electrolyzer_capacity_AE,
                    name=f"electrolyzer_AE_ramp_down_{t}")
    model.addConstr(ammonia_synthesis_power[t] - ammonia_synthesis_power[t-1] <= Flex_mid  * ammonia_synthesis_capacity,
                    name=f"ammonia_ramp_up_{t}")
    model.addConstr(ammonia_synthesis_power[t] - ammonia_synthesis_power[t-1] >= -Flex_down * ammonia_synthesis_capacity,
                    name=f"ammonia_ramp_down_{t}")
    
    #尝试增加弃电
    model.addConstr(elec_demand[t] >= elec_demand[t-1],
                    name=f"elec_demand_up_{t}")
    #model.addConstr(elec_demand[t] <= (1+Flex_up)*elec_demand[t-1],
    #                name=f"elec_demand_down_{t}")
    
# 5.2 合成氨供需平衡约束（含氨储存充放，支持不同时间聚合尺度）
start_index = 168  # 避免初始不稳定阶段
if aggregation_mode == "hourly":
    for t in range(start_index, time_steps):
        model.addConstr(
            ammonia_synthesis_power[t] + ammonia_storage_discharge[t] ==
            hourly_ammonia_demand + ammonia_storage_charge[t],
            name=f"ammonia_balance_{t}"
        )
elif aggregation_mode == "3hourly":
    interval = 3
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
            ==
            hourly_ammonia_demand * interval +
            gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
            name=f"ammonia_balance_3hourly_{i}"
        )
elif aggregation_mode == "daily":
    interval = 24
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
            >=
            0.8 * hourly_ammonia_demand * interval +
            gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
            name=f"ammonia_balance_daily_{i}"
        )

        # model.addConstr(
        #     gp.quicksum(elec_demand[t] for t in range(t_start, t_end)) 
        #     >= 300,
        #     name=f"elec_demand_balance_daily_{i}"
        # )
elif aggregation_mode == "weekly":
    interval = 168  # 一周168小时
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
            ==
            hourly_ammonia_demand * interval +
            gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
            name=f"ammonia_balance_weekly_{i}"
        )
elif aggregation_mode == "monthly":
    # 近似每月730小时（可根据实际情况调整）
    interval = 730
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
            gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
            ==
            hourly_ammonia_demand * interval +
            gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
            name=f"ammonia_balance_monthly_{i}"
        )
elif aggregation_mode == "annual":
    model.addConstr(
        gp.quicksum(ammonia_synthesis_power[t] for t in range(start_index, time_steps)) +
        gp.quicksum(ammonia_storage_discharge[t] for t in range(start_index, time_steps))
        ==
        hourly_ammonia_demand * (time_steps - start_index) +
        gp.quicksum(ammonia_storage_charge[t] for t in range(start_index, time_steps)),
        name="ammonia_balance_annual"
    )
else:
    raise ValueError("Unsupported aggregation_mode value!")

# 附加：年度合成氨生产总量约束
model.addConstr(
    gp.quicksum(ammonia_synthesis_power[t] for t in range(time_steps)) >= hourly_ammonia_demand * 8760,
    name="annual_ammonia_production_constraint"
)

# 5.3 电力与存储平衡
conversion_factor_AE  = electrolyzer_eff_AE  * 3.6 / 120
conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

for t in range(time_steps):
    wind_power = wind_output[t] * wind_capacity
    pv_power   = pv_output[t]   * pv_capacity
    
    # 电池储能约束
    model.addConstr(energy_balance[t] <= battery_storage_capacity,
                    name=f"energy_balance_limit_{t}")
    model.addConstr(battery_charge[t] + battery_discharge[t] <= battery_energy_capacity,
                    name=f"battery_charge_discharge_limit_{t}")

    # 动态电池平衡
    if t == 0:
        model.addConstr(
            energy_balance[t] == battery_charge[t] * battery_charge_efficiency
                                - battery_discharge[t] / battery_discharge_efficiency,
            name=f"battery_balance_{t}" )
    else:
        model.addConstr(
            energy_balance[t] == energy_balance[t-1] +
                                battery_charge[t] * battery_charge_efficiency
                              - battery_discharge[t] / battery_discharge_efficiency,
            name=f"battery_balance_{t}" )

    # 氢气存储约束
    model.addConstr(hydrogen_storage_balance[t] <= hydrogen_storage_capacity,
                    name=f"hydrogen_storage_capacity_limit_{t}")
    model.addConstr(hydrogen_charge[t] + hydrogen_discharge[t] <= hydrogen_storage_capacity,
                    name=f"hydrogen_charge_limit_{t}")

    # 氢气存储动态平衡
    if t == 0:
        model.addConstr(
            hydrogen_storage_balance[t] == hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
            name=f"hydrogen_balance_{t}"
        )
    else:
        model.addConstr(
            hydrogen_storage_balance[t] == hydrogen_storage_balance[t-1] + hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
            name=f"hydrogen_balance_{t}"
        )
    
    # 氢气供需平衡：电解槽产氢（转换系数 3.6/120）加上氢气释放，
    # 需满足合成氨生产所需的氢气（按氢消耗比）加上充氢量

    # 氨储存平衡
    model.addConstr(ammonia_storage_balance[t] == (ammonia_storage_balance[t-1] if t>0 else ammonia_storage_charge[0])
                              + ammonia_storage_charge[t] - ammonia_storage_discharge[t],
                    name=f"ammonia_storage_balance_{t}")
    model.addConstr(ammonia_storage_balance[t] <= ammonia_storage_capacity,
                    name=f"ammonia_storage_capacity_limit_{t}")
    
    # 氨储存：确保合成氨生产加上储存放出不少于储入量
    model.addConstr(
        ammonia_synthesis_power[t] + ammonia_storage_discharge[t] >= ammonia_storage_charge[t],
        name=f"ammonia_min_balance_{t}"
    )
    
    # 电力平衡
    model.addConstr(
        wind_power + pv_power + battery_discharge[t] ==
        electrolyzer_power_AE[t] + electrolyzer_power_PEM[t] +
        battery_charge[t] + hydrogen_charge[t] * hydrogen_storage_elec +
        ammonia_synthesis_power[t] * ammonia_electricity_ratio + surplus[t] + elec_demand[t],
        name=f"power_balance_{t}" )

    # 氢气供需平衡
    model.addConstr(
        electrolyzer_power_AE[t] * conversion_factor_AE +
        electrolyzer_power_PEM[t] * conversion_factor_PEM +
        hydrogen_discharge[t] >=
        ammonia_synthesis_power[t] * ammonia_hydrogen_ratio + hydrogen_charge[t],
        name=f"hydrogen_PD_balance_{t}" )

# 5.4 容量与最小运行
model.addConstrs((electrolyzer_capacity_AE >= electrolyzer_power_AE[t] for t in range(time_steps)), name="electrolyzer_AE_capacity")
model.addConstrs((electrolyzer_capacity_PEM >= electrolyzer_power_PEM[t] for t in range(time_steps)), name="electrolyzer_PEM_capacity")
model.addConstrs((electrolyzer_power_AE[t] >= 0.1 * electrolyzer_capacity_AE for t in range(start_index, time_steps)), name="electrolyzer_AE_min")
model.addConstrs((ammonia_synthesis_capacity >= ammonia_synthesis_power[t] for t in range(time_steps)), name="ammonia_capacity")
model.addConstrs((ammonia_synthesis_power[t] >= 0.2 * ammonia_synthesis_capacity for t in range(start_index, time_steps)), name="ammonia_min") 
# Source:DEA 2024 data_sheets_for_renewable_fuels 5.4 Hydrogen to Ammonia for 2040 estimated 

# AE电解需配备电池能量：
model.addConstr(battery_energy_capacity >= 0.5 * electrolyzer_capacity_AE, name="min_battery_for_AE")

# 6. 求解
model.optimize()

# =============================================================================
# 7. 输出结果
# =============================================================================
if model.status == GRB.OPTIMAL:
    # 总氨产量（kg/年）
    total_ammonia_production = sum(ammonia_synthesis_power[t].x for t in range(time_steps))
    
    # 各电解槽年累计产氢（kg H2）
    conv_AE  = conversion_factor_AE
    conv_PEM = conversion_factor_PEM
    total_hydrogen_production = sum(
        electrolyzer_power_AE[t].x * conv_AE +
        electrolyzer_power_PEM[t].x * conv_PEM
        for t in range(time_steps)
    )
    
    # 年累计电解耗电（kWh）
    total_elec_consumed = sum(
        electrolyzer_power_AE[t].x +
        electrolyzer_power_PEM[t].x
        for t in range(time_steps)
    )
    
    # 电解槽综合利用率（%）
    installed_elec_cap = electrolyzer_capacity_AE.x + electrolyzer_capacity_PEM.x
    electrolyzer_utilization = total_elec_consumed / (installed_elec_cap * 8760) * 100
    
    # 氢气储存所用电量（kWh）
    hydrogen_storage_kwh = sum(hydrogen_charge[t].x * hydrogen_storage_elec for t in range(time_steps))
    
    # 合成氨装置利用率（%）
    ammonia_utilization = total_ammonia_production / (ammonia_synthesis_capacity.x * 8760) * 100
    
    # 年度总发电（kWh）
    total_Elec_generation = (
        sum(wind_output[t] for t in range(time_steps)) * wind_capacity.x +
        sum(pv_output[t]   for t in range(time_steps)) * pv_capacity.x -
        sum(surplus[t].x    for t in range(time_steps))
    )
    
    # 平准化电价（$/kWh）
    levelized_Elec_cost = (
        wind_annual_cost * wind_capacity.x +
        pv_annual_cost   * pv_capacity.x +
        battery_energy_annual_cost  * battery_energy_capacity.x +
        battery_storage_annual_cost * battery_storage_capacity.x +
        battery_charge_penalty   * sum(battery_charge[t].x   for t in range(time_steps)) +
        battery_discharge_penalty* sum(battery_discharge[t].x for t in range(time_steps))
    ) / total_Elec_generation
    
    # 平准化制氢成本 ($/kgH2)
    levelized_H2_cost = (
        electrolyzer_annual_cost_AE  * electrolyzer_capacity_AE.x +
        electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM.x +
        levelized_Elec_cost * total_elec_consumed +
        levelized_Elec_cost * hydrogen_storage_kwh +
        hydrogen_storage_annual_cost * hydrogen_storage_capacity.x
    ) / total_hydrogen_production
    
    print("Optimal system configuration:")
    print(f"Wind capacity:                 {wind_capacity.x:.2f} kW")
    print(f"PV   capacity:                 {pv_capacity.x:.2f} kW")
    print(f"Battery power capacity:        {battery_energy_capacity.x:.2f} kW")
    print(f"Battery storage capacity:      {battery_storage_capacity.x:.2f} kWh")
    print(f"AE Electrolyzer capacity:      {electrolyzer_capacity_AE.x:.2f} kW")
    print(f"PEM Electrolyzer capacity:     {electrolyzer_capacity_PEM.x:.2f} kW")
    print(f"Ammonia synthesis capacity:    {ammonia_synthesis_capacity.x*8760/1000:.2f} t/y")
    print(f"Hydrogen storage capacity:     {hydrogen_storage_capacity.x:.2f} kgH2")
    print(f"Ammonia storage capacity:      {ammonia_storage_capacity.x:.2f} kgNH3")
    print(f"Total ammonia production:      {total_ammonia_production:.2f} kg/yr")
    print(f"Total hydrogen production:     {total_hydrogen_production:.2f} kgH2/yr")
    print(f"Electrolyzer utilization:      {electrolyzer_utilization:.2f} %")
    print(f"Ammonia synthesis utilization: {ammonia_utilization:.2f} %")
    print(f"Levelized Elec cost:           ${levelized_Elec_cost:.2f} per kWh")
    print(f"Levelized H2 cost:             ${levelized_H2_cost:.2f} per kgH2")
    print(f"Levelized ammonia production cost: ${model.objVal / total_ammonia_production * 1000:.2f} per ton")
else:
    print("No feasible solution found.")


## 3.2 合成氨系统（考虑向外输电）

在基础合成氨模型基础上，增加向电网输电功能的扩展模型。

### 新增功能
- **电网互联**：系统可向电网输送电力
- **电价激励**：通过负的电力需求惩罚系数实现收益
- **爬坡约束**：输电功率单调递增，避免频繁波动

### 关键参数修改
- **电价收益**：0.05 $/kWh（电网收购价）
- **惩罚系数**：-0.05 $/kWh（负值表示收益）
- **技术成本**：采用中国实际项目成本参数

### 约束条件
```python
# 输电功率爬坡约束
elec_demand[t] >= elec_demand[t-1]  # 单调递增
```

### 应用场景
- 新能源基地多元化收益模式
- 化工+电力双重产出系统
- 电网调峰辅助服务
- 提高新能源消纳能力

### 经济价值
通过电力销售收入补贴化工产品成本，提升项目经济性

## 3.2 折线图

In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB

import matplotlib.pyplot as plt    # ← 确保这一行存在

if model.status == GRB.OPTIMAL:
    # 计算各时段输出与状态曲线
    wind_power_profile = wind_output * wind_capacity.x
    pv_power_profile   = pv_output   * pv_capacity.x

    # 电解槽实际出力（考虑各自效率），单位 kW
    electrolyzer_demand_AE  = [electrolyzer_power_AE[t].x  * electrolyzer_eff_AE  for t in range(time_steps)]
    electrolyzer_demand_PEM = [electrolyzer_power_PEM[t].x * electrolyzer_eff_PEM for t in range(time_steps)]

    # 合成氨生产曲线（kg/h）和氨储存状态（以 100kg 为单位）
    ammonia_synthesis_profile = [ammonia_synthesis_power[t].x for t in range(time_steps)]
    ammonia_storage_profile   = [ammonia_storage_balance[t].x / 100 for t in range(time_steps)]

    # 电池充放电与储能状态
    battery_charge_profile    = [battery_charge[t].x    for t in range(time_steps)]
    battery_discharge_profile = [battery_discharge[t].x for t in range(time_steps)]
    energy_balance_profile    = [energy_balance[t].x    for t in range(time_steps)]

    # 提取电价（power_balance 对偶）与氨价（ammonia_balance 对偶）
    electricity_prices = [model.getConstrByName(f"power_balance_{t}").pi for t in range(time_steps)]
    ammonia_prices     = [0.0]*time_steps
    for t in range(168, time_steps):
        c = model.getConstrByName(f"ammonia_balance_{t}")
        ammonia_prices[t] = c.pi if c is not None else 0.0

    # -- 样本周绘图（第11周） --
    week_start = 168 * 10
    week_end   = week_start + 168

    plt.figure(figsize=(15, 8))
    plt.plot(wind_power_profile[week_start:week_end],    label='Wind Output (kW)')
    plt.plot(pv_power_profile[week_start:week_end],      label='PV Output (kW)')
    plt.plot(battery_charge_profile[week_start:week_end],    label='Battery Charge (kW)')
    plt.plot(battery_discharge_profile[week_start:week_end], label='Battery Discharge (kW)')
    plt.plot(electrolyzer_demand_AE[week_start:week_end],    label='Electrolyzer AE Demand (kW)')
    plt.plot(electrolyzer_demand_PEM[week_start:week_end],   label='Electrolyzer PEM Demand (kW)')
    plt.plot(ammonia_synthesis_profile[week_start:week_end], label='Ammonia Prod (kg/h)')
    plt.plot(ammonia_storage_profile[week_start:week_end],   label='Ammonia Storage (100kg)')
    plt.plot(electricity_prices[week_start:week_end],        label='Electricity Price ($/kWh)')
    plt.plot(ammonia_prices[week_start:week_end],            label='Ammonia Price ($/kg)')
    plt.xlabel('Hour')
    plt.ylabel('Value')
    plt.title('Sample Week Profiles (Hour 1680–1848)')
    plt.legend()
    plt.grid(True)
    plt.show()

    # -- 按月绘图 --
    hours_in_month = [0, 744, 1416, 2160, 2880, 3624, 4344, 5088, 5832, 6552, 7296, 8016, 8760]
    month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

    for i in range(12):
        start = hours_in_month[i]
        end   = hours_in_month[i+1]

        plt.figure(figsize=(15, 8))
        plt.plot(wind_power_profile[start:end],    label='Wind Output (kW)')
        plt.plot(pv_power_profile[start:end],      label='PV Output (kW)')
        plt.plot(battery_charge_profile[start:end],    label='Battery Charge (kW)')
        plt.plot(battery_discharge_profile[start:end], label='Battery Discharge (kW)')
        plt.plot(electrolyzer_demand_AE[start:end],    label='Electrolyzer AE Demand (kW)')
        plt.plot(electrolyzer_demand_PEM[start:end],   label='Electrolyzer PEM Demand (kW)')
        plt.plot(ammonia_synthesis_profile[start:end], label='Ammonia Prod (kg/h)')
        plt.plot(ammonia_storage_profile[start:end],   label='Ammonia Storage (100kg)')
        plt.plot(electricity_prices[start:end],        label='Electricity Price ($/kWh)')
        plt.plot(ammonia_prices[start:end],            label='Ammonia Price ($/kg)')
        plt.xlabel('Hour')
        plt.ylabel('Value')
        plt.title(f'Profiles for {month_names[i]}')
        plt.legend()
        plt.grid(True)
        plt.show()
else:
    print("No feasible solution found.")


## 3.3 合成氨系统时序分析

对合成氨生产系统进行详细的时序运行分析。

### 分析内容
**功率曲线**：
- 风光发电出力曲线
- AE/PEM电解槽功率曲线  
- 合成氨装置运行曲线
- 电池充放电曲线

**储存状态**：
- 电池储能水平（kWh）
- 氨储存量（以100kg为单位显示）

**价格信号**：
- 电力边际价格（$/kWh）
- 氨产品边际价格（$/kg）

### 时间窗口
- **样本周**：第11周（1680-1848小时）
- **逐月分析**：12个月完整覆盖

### 关键指标
- 电解槽利用率
- 合成氨装置利用率  
- 储能循环效率
- 供需匹配程度

### 可视化特点
- 多条线图同时展示
- 图例清晰标注
- 网格线辅助读数
- 单位统一标准化

## 3.3 电力平衡

In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB

import matplotlib.pyplot as plt    # ← 确保这一行存在

if model.status == GRB.OPTIMAL:
    # ---------------------------
    # 样本周（168小时）堆积柱状图
    # ---------------------------
    week_start = 168 * 10   # 例如取第11周
    week_end   = week_start + 168
    x = np.arange(week_end - week_start)

    # 供给侧：风电、光伏、以及电池放电（均为正值）
    supply_wind    = np.array([wind_output[t] * wind_capacity.x     for t in range(week_start, week_end)])
    supply_pv      = np.array([pv_output[t]   * pv_capacity.x       for t in range(week_start, week_end)])
    supply_batt    = np.array([battery_discharge[t].x               for t in range(week_start, week_end)])

    # 需求侧：电解槽、电池充电、氢气充电、合成氨用电、以及 surplus
    dem_el_AE = np.array([electrolyzer_power_AE[t].x for t in range(week_start, week_end)])
    dem_el_PEM= np.array([electrolyzer_power_PEM[t].x for t in range(week_start, week_end)])
    dem_batt  = np.array([battery_charge[t].x         for t in range(week_start, week_end)])
    dem_h2    = np.array([hydrogen_charge[t].x * hydrogen_storage_elec
                          for t in range(week_start, week_end)])
    dem_nh3   = np.array([ammonia_synthesis_power[t].x * ammonia_electricity_ratio
                          for t in range(week_start, week_end)])
    dem_surp  = np.array([surplus[t].x                for t in range(week_start, week_end)])

    plt.figure(figsize=(15, 8))
    # 供给堆积
    plt.bar(x, supply_wind,    label='Wind Supply')
    plt.bar(x, supply_pv,      bottom=supply_wind,                 label='PV Supply')
    plt.bar(x, supply_batt,    bottom=supply_wind + supply_pv,     label='Battery Discharge')

    # 需求堆积（负值）
    bottom1 = -dem_el_AE
    plt.bar(x, -dem_el_AE,      label='Electrolyzer AE Demand',   bottom=0)
    plt.bar(x, -dem_el_PEM,     label='Electrolyzer PEM Demand',  bottom=bottom1)
    bottom2 = bottom1 - dem_el_PEM
    plt.bar(x, -dem_batt,       label='Battery Charge',           bottom=bottom2)
    bottom3 = bottom2 - dem_batt
    plt.bar(x, -dem_h2,         label='Hydrogen Charge',          bottom=bottom3)
    bottom4 = bottom3 - dem_h2
    plt.bar(x, -dem_nh3,        label='Ammonia Synthesis',        bottom=bottom4)
    bottom5 = bottom4 - dem_nh3
    plt.bar(x, -dem_surp,       label='Surplus',                  bottom=bottom5)

    plt.axhline(0, linewidth=0.8)
    plt.xlabel('Hour')
    plt.ylabel('Power (kW)')
    plt.title('Stacked Power Supply & Demand – Sample Week')
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.show()

    # ---------------------------
    # 按月堆积柱状图
    # ---------------------------
    hours_in_month = [0, 744, 1416, 2160, 2880, 3624, 4344, 5088, 5832, 6552, 7296, 8016, 8760]
    month_names   = ['January','February','March','April','May','June',
                     'July','August','September','October','November','December']

    for i in range(1):
        start = hours_in_month[i]
        end   = hours_in_month[i+1]
        x_m   = np.arange(end - start)

        sw = np.array([wind_output[t] * wind_capacity.x     for t in range(start, end)])
        sp = np.array([pv_output[t]   * pv_capacity.x       for t in range(start, end)])
        sb = np.array([battery_discharge[t].x               for t in range(start, end)])
        de_AE = np.array([electrolyzer_power_AE[t].x       for t in range(start, end)])
        de_PEM= np.array([electrolyzer_power_PEM[t].x      for t in range(start, end)])
        db   = np.array([battery_charge[t].x               for t in range(start, end)])
        dh2  = np.array([hydrogen_charge[t].x * hydrogen_storage_elec
                         for t in range(start, end)])
        dn   = np.array([ammonia_synthesis_power[t].x * ammonia_electricity_ratio
                         for t in range(start, end)])
        ds   = np.array([surplus[t].x                      for t in range(start, end)])

        plt.figure(figsize=(15, 8))
        plt.bar(x_m, sw,    label='Wind Supply')
        plt.bar(x_m, sp,    bottom=sw,                    label='PV Supply')
        plt.bar(x_m, sb,    bottom=sw+sp,                 label='Battery Discharge')

        b1 = -de_AE
        plt.bar(x_m, -de_AE,   label='Electrolyzer AE Demand',  bottom=0)
        plt.bar(x_m, -de_PEM,  label='Electrolyzer PEM Demand', bottom=b1)
        b2 = b1 - de_PEM
        plt.bar(x_m, -db,      label='Battery Charge',          bottom=b2)
        b3 = b2 - db
        plt.bar(x_m, -dh2,     label='Hydrogen Charge',         bottom=b3)
        b4 = b3 - dh2
        plt.bar(x_m, -dn,      label='Ammonia Synthesis',       bottom=b4)
        b5 = b4 - dn
        plt.bar(x_m, -ds,      label='Surplus',                 bottom=b5)

        plt.axhline(0, linewidth=0.8)
        plt.xlabel('Hour')
        plt.ylabel('Power (kW)')
        plt.title(f'Stacked Power Supply & Demand – {month_names[i]}')
        plt.legend(loc='upper right')
        plt.grid(True)
        plt.show()
else:
    print("No feasible solution found.")


## 3.4 合成氨系统电力平衡分析

采用堆积柱状图展示合成氨系统的电力供需平衡状况。

### 供给侧组成
- **风电供给**：风电场发电出力
- **光伏供给**：光伏阵列发电出力  
- **电池放电**：储能系统向负荷供电

### 需求侧组成
- **AE电解槽**：碱性电解槽耗电
- **PEM电解槽**：质子交换膜电解槽耗电
- **电池充电**：储能系统充电耗电
- **氢气储存**：压缩储氢耗电（如有）
- **合成氨**：氨合成装置辅助耗电
- **弃电**：无法消纳的多余电力

### 可视化设计
- **正值区域**：供给侧（蓝、橙、绿色系）
- **负值区域**：需求侧（紫、黄、红色系）
- **零轴线**：供需平衡基准线

### 分析价值
- 识别供需失衡时段
- 优化储能容量配置
- 评估弃电损失程度
- 指导运行策略调整

## 3.3 电力平衡（向外输电）

In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB

import matplotlib.pyplot as plt    # ← 确保这一行存在

if model.status == GRB.OPTIMAL:
    # ---------------------------
    # 样本周（168小时）堆积柱状图
    # ---------------------------
    week_start = 168 * 10   # 例如取第11周
    week_end   = week_start + 168
    x = np.arange(week_end - week_start)

    # 供给侧：风电、光伏、以及电池放电（均为正值）
    supply_wind    = np.array([wind_output[t] * wind_capacity.x     for t in range(week_start, week_end)])
    supply_pv      = np.array([pv_output[t]   * pv_capacity.x       for t in range(week_start, week_end)])
    supply_batt    = np.array([battery_discharge[t].x               for t in range(week_start, week_end)])

    # 需求侧：电解槽、电池充电、氢气充电、合成氨用电、以及 surplus
    dem_el_AE = np.array([electrolyzer_power_AE[t].x for t in range(week_start, week_end)])
    dem_el_PEM= np.array([electrolyzer_power_PEM[t].x for t in range(week_start, week_end)])
    dem_batt  = np.array([battery_charge[t].x         for t in range(week_start, week_end)])
    dem_h2    = np.array([hydrogen_charge[t].x * hydrogen_storage_elec
                          for t in range(week_start, week_end)])
    dem_nh3   = np.array([ammonia_synthesis_power[t].x * ammonia_electricity_ratio
                          for t in range(week_start, week_end)])
    dem_surp  = np.array([surplus[t].x                for t in range(week_start, week_end)])
    
    dem_elec  = np.array([elec_demand[t].x                for t in range(week_start, week_end)])
    plt.figure(figsize=(15, 8))
    # 供给堆积
    plt.bar(x, supply_wind,    label='Wind Supply')
    plt.bar(x, supply_pv,      bottom=supply_wind,                 label='PV Supply')
    plt.bar(x, supply_batt,    bottom=supply_wind + supply_pv,     label='Battery Discharge')

    # 需求堆积（负值）
    bottom0 = -dem_elec
    plt.bar(x, -dem_elec,      label='Electricity Demand',   bottom=0)
    bottom1 = bottom0 - dem_el_AE
    plt.bar(x, -dem_el_AE,      label='Electrolyzer AE Demand',   bottom=bottom0)
    plt.bar(x, -dem_el_PEM,     label='Electrolyzer PEM Demand',  bottom=bottom1)
    bottom2 = bottom1 - dem_el_PEM
    plt.bar(x, -dem_batt,       label='Battery Charge',           bottom=bottom2)
    bottom3 = bottom2 - dem_batt
    plt.bar(x, -dem_h2,         label='Hydrogen Charge',          bottom=bottom3)
    bottom4 = bottom3 - dem_h2
    plt.bar(x, -dem_nh3,        label='Ammonia Synthesis',        bottom=bottom4)
    bottom5 = bottom4 - dem_nh3
    plt.bar(x, -dem_surp,       label='Surplus',                  bottom=bottom5)

    plt.axhline(0, linewidth=0.8)
    plt.xlabel('Hour')
    plt.ylabel('Power (kW)')
    plt.title('Stacked Power Supply & Demand – Sample Week')
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.show()

    # ---------------------------
    # 按月堆积柱状图
    # ---------------------------
    hours_in_month = [0, 744, 1416, 2160, 2880, 3624, 4344, 5088, 5832, 6552, 7296, 8016, 8760]
    month_names   = ['January','February','March','April','May','June',
                     'July','August','September','October','November','December']

    for i in range(1):
        start = hours_in_month[i]
        end   = hours_in_month[i+1]
        x_m   = np.arange(end - start)

        sw = np.array([wind_output[t] * wind_capacity.x     for t in range(start, end)])
        sp = np.array([pv_output[t]   * pv_capacity.x       for t in range(start, end)])
        sb = np.array([battery_discharge[t].x               for t in range(start, end)])
        de_AE = np.array([electrolyzer_power_AE[t].x       for t in range(start, end)])
        de_PEM= np.array([electrolyzer_power_PEM[t].x      for t in range(start, end)])
        db   = np.array([battery_charge[t].x               for t in range(start, end)])
        dh2  = np.array([hydrogen_charge[t].x * hydrogen_storage_elec
                         for t in range(start, end)])
        dn   = np.array([ammonia_synthesis_power[t].x * ammonia_electricity_ratio
                         for t in range(start, end)])
        ds   = np.array([surplus[t].x                      for t in range(start, end)])

        de   = np.array([elec_demand[t].x                      for t in range(start, end)])
        
        plt.figure(figsize=(15, 8))
        plt.bar(x_m, sw,    label='Wind Supply')
        plt.bar(x_m, sp,    bottom=sw,                    label='PV Supply')
        plt.bar(x_m, sb,    bottom=sw+sp,                 label='Battery Discharge')
        
        b0 = -de
        plt.bar(x_m, -de,   label='Electricity Demand',  bottom=0)
        b1 = b0 - de_AE
        plt.bar(x_m, -de_AE,   label='Electrolyzer AE Demand',  bottom=b0)
        plt.bar(x_m, -de_PEM,  label='Electrolyzer PEM Demand', bottom=b1)
        b2 = b1 - de_PEM
        plt.bar(x_m, -db,      label='Battery Charge',          bottom=b2)
        b3 = b2 - db
        plt.bar(x_m, -dh2,     label='Hydrogen Charge',         bottom=b3)
        b4 = b3 - dh2
        plt.bar(x_m, -dn,      label='Ammonia Synthesis',       bottom=b4)
        b5 = b4 - dn
        plt.bar(x_m, -ds,      label='Surplus',                 bottom=b5)

        plt.axhline(0, linewidth=0.8)
        plt.xlabel('Hour')
        plt.ylabel('Power (kW)')
        plt.title(f'Stacked Power Supply & Demand – {month_names[i]}')
        plt.legend(loc='upper right')
        plt.grid(True)
        plt.show()
else:
    print("No feasible solution found.")


## 3.5 向外输电系统电力平衡分析

展示包含向电网输电功能的合成氨系统电力平衡状况。

### 需求侧新增组件
- **电网输电**：向外部电网输送的电力
- **收益机制**：通过电力销售获得收入

### 特殊约束
```python
# 输电功率单调递增约束
elec_demand[t] >= elec_demand[t-1]
```

### 经济机制
- **输电收益**：0.05 $/kWh × 输电量
- **成本补偿**：电力销售收入抵扣制氨成本
- **协同优化**：化工生产与电网调峰双重收益

### 可视化区别
相比基础模型，新增：
- **电网需求**（底部负值区域）：显示向电网输电量
- **收益显示**：通过负的惩罚系数体现经济收益

### 应用场景
- **多能互补**：新能源基地综合开发
- **电网服务**：参与电网调峰调频
- **降本增效**：通过电力销售提升项目收益率

### 分析要点
- 观察输电与制氨的功率分配
- 评估双重收益的经济效益
- 识别系统运行的最优策略

## 3.4 氢能平衡

In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB

import matplotlib.pyplot as plt    # ← 确保这一行存在

if model.status == GRB.OPTIMAL:
    # ---------------------------
    # 样本周（168小时）氢能供需堆积柱状图
    # ---------------------------
    week_start = 168 * 10   # 例如取第11周
    week_end = week_start + 168*2
    x = np.arange(week_end - week_start)
    
    # 供给侧：
    # 1. 氢气生产：由AE和PEM电解槽产生的氢气（单位：kg/h）
    supply_h2_production = np.array([
        (electrolyzer_power_AE[t].x * electrolyzer_eff_AE + electrolyzer_power_PEM[t].x * electrolyzer_eff_PEM) * 3.6 / 120
        for t in range(week_start, week_end)
    ])
    # 2. 氢气放电：氢气储存释放的氢气（单位：kg/h）
    supply_h2_discharge = np.array([hydrogen_discharge[t].x for t in range(week_start, week_end)])

    demand_ammonia_production = np.array([
        ammonia_synthesis_power[t].x * ammonia_hydrogen_ratio
        for t in range(week_start, week_end)
    ])
    demand_h2_charge = np.array([hydrogen_charge[t].x for t in range(week_start, week_end)])

    plt.figure(figsize=(15, 8))
    # 绘制供给侧堆积柱状图（正值）
    bar1 = plt.bar(x, supply_h2_production, label='Hydrogen Production', color='blue')
    bar2 = plt.bar(x, supply_h2_discharge, bottom=supply_h2_production, label='Hydrogen Discharge', color='cyan')

    bar3 = plt.bar(x, -demand_ammonia_production, label='Hydrogen Demand', color='purple')
    bar4 = plt.bar(x, -demand_h2_charge, bottom = -demand_ammonia_production, label='Hydrogen Charge', color='red')
    
    plt.axhline(0, color='black', linewidth=0.8)
    plt.xlabel('Hour')
    plt.ylabel('Hydrogen Flow (kg/h)')
    plt.title('Stacked Hydrogen Supply (Positive) & Demand (Negative) - Sample Week')
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.show()
    
    # ---------------------------
    # 按月氢能供需堆积柱状图
    # ---------------------------
    # 近似每月的小时数划分（可根据实际情况调整）
    hours_in_month = [0, 744, 1416, 2160, 2880, 3624, 4344, 5088, 5832, 6552, 7296, 8016, 8760]
    month_names = ['January', 'February', 'March', 'April', 'May', 'June', 
                   'July', 'August', 'September', 'October', 'November', 'December']
    
    for i in range():
        start = hours_in_month[i]
        end = hours_in_month[i+1]
        x_month = np.arange(end - start)
        
        # 供给侧数据
        supply_h2_production_month = np.array([
            (electrolyzer_power_AE[t].x + electrolyzer_power_PEM[t].x) * electrolyzer_eff * 3.6 / 120
            for t in range(start, end)
        ])
        supply_h2_discharge_month = np.array([hydrogen_discharge[t].x for t in range(start, end)])

        demand_ammonia_production_month = np.array([
        ammonia_synthesis_power[t].x * ammonia_hydrogen_ratio
        for t in range(start, end)
        ])
        demand_h2_charge_month = np.array([hydrogen_charge[t].x for t in range(start, end)])
        
        plt.figure(figsize=(15, 8))
        plt.bar(x_month, supply_h2_production_month, label='Hydrogen Production', color='blue')
        plt.bar(x_month, supply_h2_discharge_month, bottom=supply_h2_production_month, label='Hydrogen Discharge', color='cyan')

        plt.bar(x_month, -demand_ammonia_production_month, label='Hydrogen Demand', color='purple')
        plt.bar(x_month, -demand_h2_charge_month, -demand_ammonia_production_month, label='Hydrogen Charge', color='red')
        
        plt.axhline(0, color='black', linewidth=0.8)
        plt.xlabel('Hour')
        plt.ylabel('Hydrogen Flow (kg/h)')
        plt.title(f'Stacked Hydrogen Supply & Demand ({month_names[i]})')
        plt.legend(loc='upper right')
        plt.grid(True)
        plt.show()
else:
    print("No feasible solution found.")


## 3.6 合成氨系统氢能平衡分析

专门分析合成氨系统中氢气子系统的供需平衡动态。

### 氢气供给
- **电解制氢**：AE+PEM电解槽产氢
- **储氢释放**：氢气储罐向系统释放氢气
- **转换效率**：
  - AE: eff_AE × 3.6/120 kg/kW
  - PEM: eff_PEM × 3.6/120 kg/kW

### 氢气需求  
- **合成氨反应**：N2 + 3H2 → 2NH3
- **氢气储存**：向储氢罐充入氢气
- **消耗比例**：0.18 kg H2/kg NH3

### 可视化特点
- **正值（供给）**：
  - 蓝色：电解制氢
  - 青色：储氢释放
- **负值（需求）**：
  - 紫色：合成氨消耗
  - 红色：储氢充入

### 分析维度
- **样本周**：详细时序分析（168小时）
- **逐月分析**：季节性特征识别（12个月）

### 关键指标
- 氢气供需匹配度
- 储氢系统循环效率
- 电解槽与合成氨的协调性
- 氢气储存容量优化需求

# 4. 制氢BU

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB

# =============================================================================
# 1. 数据加载
# =============================================================================
# 请确保文件 "CF_NM_2023.csv" 存在且第一列为PV数据、第二列为风电数据
wind_solar_data = pd.read_csv('CF_NM_2023.csv')
pv_output   = wind_solar_data.iloc[:, 0].values   # 光伏输出曲线（无量纲或功率比例）
wind_output = wind_solar_data.iloc[:, 1].values   # 风电输出曲线

# =============================================================================
# 2. 参数定义
# =============================================================================
# 2.1 可再生能源与电池参数
wind_cost = 800            # 风电安装成本 ($/kW) Source: IRENA Renewable Power Generation Costs in 2023 Figure 2.1 for 5th Global
pv_cost   = 350            # 光伏安装成本 ($/kW) Source: DEA 2024 technology_data_for_el_and_dh 22 Utility-scale PV for 2035 estimated
battery_energy_cost = 100         # 储能安装成本 ($/kW) Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery
battery_storage_cost = 100         # 储能安装成本 ($/kW) Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery
battery_charge_efficiency = 0.985    # Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery
battery_discharge_efficiency = 0.975    # Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery
battery_charge_penalty = 0.002  # ($/kWh)  # Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery
battery_discharge_penalty = 0.000 # ($/kWh)
battery_energy_om = battery_energy_cost * 0.03 
battery_storage_om = battery_storage_cost * 0.03 
battery_lifetime = 20 # Source: DEA 2024 technology_datasheet_for_energy_storage 180 Lithium Ion Battery for 2020
wind_lifetime = 30 # Source: DEA 2024 technology_data_for_el_and_dh 20 Onshore turbines for 2035 estimated
pv_lifetime = 40 # Source: DEA 2024 technology_data_for_el_and_dh 22 Utility-scale PV for 2035 estimated

# 2.2 电解槽参数（制氢设备，分AE与PEM两种）
electrolyzer_cost_AE = 350      # ($/kW) # Source: DEA 2024 data_sheets_for_renewable_fuels 1.1 AEC 1 GW  for 2040 lower
electrolyzer_cost_PEM = 600      # ($/kW) # Source: DEA 2024 data_sheets_for_renewable_fuels 1.1 PEMEC 1 GW  for 2040 Central
electrolyzer_om_AE = electrolyzer_cost_AE * 0.03
electrolyzer_om_PEM = electrolyzer_cost_PEM * 0.03
electrolyzer_lifetime_AE = 25 # ($/kW) # Source: DEA 2024 data_sheets_for_renewable_fuels 1.1 AEC 1 GW  for 2040 estimated
electrolyzer_lifetime_PEM = 25 # ($/kW) # Source: DEA 2024 data_sheets_for_renewable_fuels 1.1 PEMEC 1 GW  for 2040 estimated
electrolyzer_eff_AE = 0.65       # Source: DEA 2024 data_sheets_for_renewable_fuels 1.1 AEC 1 GW  for 2040 estimated
electrolyzer_eff_PEM = 0.70       # Source: IEA 2024 Global Hydrogen Review: Assumptions annex 

# 2.3 氢气存储参数 Source: DEA 2024 technology_datasheet_for_energy_storage 151a Hydrogen Storage - Tanks
hydrogen_storage_cost = 1000  # ($/kgH2) 2040
# hydrogen_storage_cost = 1750  # ($/kgH2) 2030
# hydrogen_storage_cost = 800  # ($/kgH2) 2050

hydrogen_storage_om   = hydrogen_storage_cost * 0.03
hydrogen_charge_penalty = 0.00001  # ($/kgH2)
hydrogen_storage_efficiency = 0.9
hydrogen_storage_elec = 0.33       # kWh/kg H2（充氢所需电耗）
hydrogen_storage_lifetime = 30

# 2.4 目标制氢参数
target_hydrogen_production = 10 * 1000  # kg/年，即10吨/年
hourly_hydrogen_demand = target_hydrogen_production / 8760

# 2.5 年化成本计算（折现率）
discount_rate = 0.07
wind_annual_cost = wind_cost * discount_rate / (1 - (1 + discount_rate) ** -wind_lifetime) + wind_cost * 0.03
pv_annual_cost = pv_cost * discount_rate / (1 - (1 + discount_rate) ** -pv_lifetime) + pv_cost * 0.03
battery_energy_annual_cost = battery_energy_cost * discount_rate / (1 - (1 + discount_rate) ** -battery_lifetime) + battery_energy_om
battery_storage_annual_cost = battery_storage_cost * discount_rate / (1 - (1 + discount_rate) ** -battery_lifetime) + battery_storage_om
electrolyzer_annual_cost_AE = electrolyzer_cost_AE * discount_rate / (1 - (1 + discount_rate) ** -electrolyzer_lifetime_AE) + electrolyzer_om_AE
electrolyzer_annual_cost_PEM = electrolyzer_cost_PEM * discount_rate / (1 - (1 + discount_rate) ** -electrolyzer_lifetime_PEM) + electrolyzer_om_PEM
hydrogen_storage_annual_cost = hydrogen_storage_cost * discount_rate / (1 - (1 + discount_rate) ** -hydrogen_storage_lifetime) + hydrogen_storage_om

# 2.6 逐时灵活性参数（限制电解槽功率变化率） 
Flex_mid = 0.1  
surplus_penalty = 0.00001

# 2.7 供需平衡聚合尺度选择
# 可选值："hourly"、"3hourly"、"daily"、"weekly"、"monthly"、"annual"
aggregation_mode = "hourly"

# =============================================================================
# 3. 模型创建与变量定义
# =============================================================================
model = Model("HydrogenProductionOptimization")
# 设置模型求解参数
model.setParam('TimeLimit', 1000)
model.setParam('MIPFocus', 1)
model.setParam('Threads', 1)
model.setParam('MIPGap', 0.1) 
model.setParam('Method', 2)
model.setParam('Crossover', 0)

time_steps = 8760

# 3.1 装机设备决策变量
wind_capacity = model.addVar(lb=0, name="wind_capacity")
pv_capacity   = model.addVar(lb=0, name="pv_capacity")
battery_energy_capacity = model.addVar(lb=0, name="battery_energy_capacity")
battery_storage_capacity = model.addVar(lb=0, name="battery_storage_capacity")
hydrogen_storage_capacity = model.addVar(lb=0, name="hydrogen_storage_capacity")

# 分别定义 AE 与 PEM 型电解槽装机决策变量
electrolyzer_capacity_AE = model.addVar(lb=0, name="electrolyzer_capacity_AE")
electrolyzer_capacity_PEM = model.addVar(lb=0, name="electrolyzer_capacity_PEM")

# 3.2 时间步相关变量（共8760小时）
# 电解槽出力变量
electrolyzer_power_AE = model.addVars(time_steps, lb=0, name="electrolyzer_power_AE")
electrolyzer_power_PEM = model.addVars(time_steps, lb=0, name="electrolyzer_power_PEM")

# 电池储能相关变量
energy_balance = model.addVars(time_steps, lb=0, name="energy_balance")
battery_charge = model.addVars(time_steps, lb=0, name="battery_charge")
battery_discharge = model.addVars(time_steps, lb=0, name="battery_discharge")

# 多余电量变量（未利用电量）
surplus = model.addVars(time_steps, lb=0, name="surplus")

# 氢气存储相关变量
hydrogen_storage_balance = model.addVars(time_steps, lb=0, name="hydrogen_storage_balance")
hydrogen_charge = model.addVars(time_steps, lb=0, name="hydrogen_charge")
hydrogen_discharge = model.addVars(time_steps, lb=0, name="hydrogen_discharge")

# =============================================================================
# 4. 目标函数
# =============================================================================
model.setObjective(
    wind_annual_cost * wind_capacity +
    pv_annual_cost * pv_capacity +
    battery_energy_annual_cost * battery_energy_capacity +
    battery_storage_annual_cost * battery_storage_capacity +
    electrolyzer_annual_cost_AE * electrolyzer_capacity_AE + 
    electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM +
    hydrogen_storage_annual_cost * hydrogen_storage_capacity +
    surplus_penalty * gp.quicksum(surplus[t] for t in range(time_steps)) +
    battery_charge_penalty * gp.quicksum(battery_charge[t] for t in range(time_steps)) +
    battery_discharge_penalty * gp.quicksum(battery_discharge[t] for t in range(time_steps)) +
    hydrogen_charge_penalty * gp.quicksum(hydrogen_charge[t] for t in range(time_steps)),
    GRB.MINIMIZE
)

# =============================================================================
# 5. 约束条件
# =============================================================================

# 5.1 逐时设备灵活性约束（仅针对 AE 型电解槽，限制功率变化）
for t in range(1, time_steps):
    model.addConstr(electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] <= Flex_mid * electrolyzer_capacity_AE,
                    name=f"electrolyzer_AE_ramp_up_{t}")
    model.addConstr(electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] >= -Flex_mid * electrolyzer_capacity_AE,
                    name=f"electrolyzer_AE_ramp_down_{t}")

# 5.2 电力平衡约束（逐时）
for t in range(time_steps):
    wind_power = wind_output[t] * wind_capacity
    pv_power = pv_output[t] * pv_capacity
    model.addConstr(
        wind_power + pv_power + battery_discharge[t] ==
        (electrolyzer_power_AE[t] + electrolyzer_power_PEM[t]) +
        battery_charge[t] + hydrogen_charge[t] * hydrogen_storage_elec + surplus[t],
        name=f"power_balance_{t}"
    )
    # 电池储能与充放电限制
    model.addConstr(energy_balance[t] <= battery_storage_capacity,
                    name=f"energy_balance_limit_{t}")
    model.addConstr(battery_charge[t] + battery_discharge[t] <= battery_energy_capacity,
                    name=f"battery_charge_discharge_limit_{t}")
    # 电池动态平衡
    if t == 0:
        model.addConstr(
            energy_balance[t] == battery_charge[t] * battery_charge_efficiency - battery_discharge[t] / battery_discharge_efficiency,
            name=f"battery_balance_{t}"
        )
    else:
        model.addConstr(
            energy_balance[t] == energy_balance[t-1] + battery_charge[t] * battery_charge_efficiency - battery_discharge[t] / battery_discharge_efficiency,
            name=f"battery_balance_{t}"
        )

# 5.3 氢气供需平衡约束（支持不同时间聚合尺度）
# 转换系数：电解槽产氢量 = 出力 * electrolyzer_eff * 3.6/120
conversion_factor_AE = electrolyzer_eff_AE * 3.6 / 120
conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

if aggregation_mode == "hourly":
    for t in range(168, time_steps):
        model.addConstr( 
            (electrolyzer_power_AE[t] * conversion_factor_AE + electrolyzer_power_PEM[t] * conversion_factor_PEM) + hydrogen_discharge[t] >=
            0.8 * hourly_hydrogen_demand + hydrogen_charge[t],
            name=f"hydrogen_balance_{t}"
        )
elif aggregation_mode == "3hourly":
    interval = 3
    start_index = 168
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum((electrolyzer_power_AE[t] * conversion_factor_AE + electrolyzer_power_PEM[t] * conversion_factor_PEM) for t in range(t_start, t_end)) +
            gp.quicksum(hydrogen_discharge[t] for t in range(t_start, t_end))
            >= hourly_hydrogen_demand * interval +
            gp.quicksum(hydrogen_charge[t] for t in range(t_start, t_end)),
            name=f"hydrogen_balance_3hourly_{i}"
        )
elif aggregation_mode == "daily":
    interval = 24
    start_index = 168
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum((electrolyzer_power_AE[t] * conversion_factor_AE + electrolyzer_power_PEM[t] * conversion_factor_PEM) for t in range(t_start, t_end)) +
            gp.quicksum(hydrogen_discharge[t] for t in range(t_start, t_end))
            >= 0.8 * hourly_hydrogen_demand * interval +
            gp.quicksum(hydrogen_charge[t] for t in range(t_start, t_end)),
            name=f"hydrogen_balance_daily_{i}"
        )
elif aggregation_mode == "weekly":
    interval = 168  # 1周 = 168小时
    start_index = 168
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum((electrolyzer_power_AE[t] * conversion_factor_AE + electrolyzer_power_PEM[t] * conversion_factor_PEM) for t in range(t_start, t_end)) +
            gp.quicksum(hydrogen_discharge[t] for t in range(t_start, t_end))
            >= hourly_hydrogen_demand * interval +
            gp.quicksum(hydrogen_charge[t] for t in range(t_start, t_end)),
            name=f"hydrogen_balance_weekly_{i}"
        )
elif aggregation_mode == "monthly":
    interval = 730
    start_index = 168
    n_intervals = (time_steps - start_index) // interval
    for i in range(n_intervals):
        t_start = start_index + i * interval
        t_end = t_start + interval
        model.addConstr(
            gp.quicksum((electrolyzer_power_AE[t] * conversion_factor_AE + electrolyzer_power_PEM[t] * conversion_factor_PEM) for t in range(t_start, t_end)) +
            gp.quicksum(hydrogen_discharge[t] for t in range(t_start, t_end))
            >= hourly_hydrogen_demand * interval +
            gp.quicksum(hydrogen_charge[t] for t in range(t_start, t_end)),
            name=f"hydrogen_balance_monthly_{i}"
        )
elif aggregation_mode == "annual":
    model.addConstr(
        gp.quicksum((electrolyzer_power_AE[t] * conversion_factor_AE + electrolyzer_power_PEM[t] * conversion_factor_PEM) for t in range(168, time_steps)) +
        gp.quicksum(hydrogen_discharge[t] for t in range(168, time_steps))
        >= hourly_hydrogen_demand * (time_steps - 168) +
        gp.quicksum(hydrogen_charge[t] for t in range(168, time_steps)),
        name="hydrogen_balance_annual"
    )
else:
    raise ValueError("Unsupported aggregation_mode value!")

# 附加：年生产总量约束
model.addConstr(
    gp.quicksum((electrolyzer_power_AE[t] * conversion_factor_AE + electrolyzer_power_PEM[t] * conversion_factor_PEM) for t in range(time_steps)) >= hourly_hydrogen_demand * 8760,
    name="annual_hydrogen_production_constraint"
)

# 5.4 氢气储存动态平衡及容量限制
model.addConstr(hydrogen_storage_balance[0] == hydrogen_charge[0] - hydrogen_discharge[0],
                name="hydrogen_storage_balance_0")
for t in range(1, time_steps):
    model.addConstr(
        hydrogen_storage_balance[t] == hydrogen_storage_balance[t-1] + hydrogen_charge[t] - hydrogen_discharge[t],
        name=f"hydrogen_storage_balance_{t}"
    )
    model.addConstr(hydrogen_storage_balance[t] <= hydrogen_storage_capacity,
                    name=f"hydrogen_storage_level_limit_{t}")
    model.addConstr(hydrogen_charge[t] + hydrogen_discharge[t] <= hydrogen_storage_capacity,
                    name=f"hydrogen_charge_limit_{t}")

# 5.5 装机容量约束
model.addConstrs((electrolyzer_capacity_AE >= electrolyzer_power_AE[t] for t in range(time_steps)),
                 name="electrolyzer_AE_capacity_constraint")
model.addConstrs((electrolyzer_capacity_PEM >= electrolyzer_power_PEM[t] for t in range(time_steps)),
                 name="electrolyzer_PEM_capacity_constraint")
model.addConstrs((electrolyzer_power_AE[t] >= 0.1 * electrolyzer_capacity_AE for t in range(168, time_steps)),
                 name="electrolyzer_AE_power_constraint")

# 5.6 附加：要求AE型电解槽必须配备至少0.5倍（约10%）的储能电池
model.addConstr(battery_energy_capacity >= 0.5 * electrolyzer_capacity_AE,
                name="min_battery_for_AE")

#model.addConstr(electrolyzer_capacity_PEM == 0.2 * electrolyzer_capacity_AE,
#                name="min_PEM_for_AE")
# =============================================================================
# 6. 求解模型
# =============================================================================
model.optimize()

# =============================================================================
# 7. 输出结果
# =============================================================================
if model.status == GRB.OPTIMAL:
    total_hydrogen_production = sum((electrolyzer_power_AE[t].x * conversion_factor_AE+ electrolyzer_power_PEM[t].x * conversion_factor_PEM)  for t in range(time_steps))
    hydrogen_production_kwh = sum((electrolyzer_power_AE[t].x * electrolyzer_eff_AE + electrolyzer_power_PEM[t].x * electrolyzer_eff_PEM) for t in range(time_steps))
    hydrogen_electricity_demand =  sum(electrolyzer_power_AE[t].x  + electrolyzer_power_PEM[t].x  for t in range(time_steps))
    electrolyzer_utilization_hours = hydrogen_production_kwh / ((electrolyzer_capacity_AE.x * electrolyzer_eff_AE + electrolyzer_capacity_PEM.x * electrolyzer_eff_PEM) * 8760) * 100
    hydrogen_storage_kwh = sum(hydrogen_charge[t].x * hydrogen_storage_elec for t in range(time_steps))
    total_Elec_generation = (sum(wind_output[t] for t in range(time_steps)) * wind_capacity.x +
                             sum(pv_output[t] for t in range(time_steps)) * pv_capacity.x -
                             sum(surplus[t].x for t in range(time_steps)))
    
    levelized_Elec_cost = (
        wind_annual_cost * wind_capacity.x +
        pv_annual_cost * pv_capacity.x +
        battery_energy_annual_cost * battery_energy_capacity.x +
        battery_storage_annual_cost * battery_storage_capacity.x +
        battery_charge_penalty * sum(battery_charge[t].x for t in range(time_steps)) +
        battery_discharge_penalty * sum(battery_discharge[t].x for t in range(time_steps))
    ) / total_Elec_generation

    levelized_H2_cost = (
        electrolyzer_annual_cost_AE * electrolyzer_capacity_AE.x +
        electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM.x +
        levelized_Elec_cost * hydrogen_electricity_demand +
        levelized_Elec_cost * hydrogen_storage_kwh +
        hydrogen_storage_annual_cost * hydrogen_storage_capacity.x
    ) / (hydrogen_production_kwh / 33.3)

    print("Optimal system configuration:")
    print(f"Wind capacity: {wind_capacity.x:.2f} kW")
    print(f"PV capacity: {pv_capacity.x:.2f} kW")
    print(f"Battery Energy capacity: {battery_energy_capacity.x:.2f} kW")
    print(f"Battery Storage capacity: {battery_storage_capacity.x:.2f} kW")
    print(f"Hydrogen Storage capacity: {hydrogen_storage_capacity.x:.2f} kW")
    print(f"AE Electrolyzer capacity: {electrolyzer_capacity_AE.x:.2f} kW")
    print(f"PEM Electrolyzer capacity: {electrolyzer_capacity_PEM.x:.2f} kW")
    print(f"Total hydrogen production: {total_hydrogen_production:.2f} kg")
    print(f"Electrolyzer utilization hours: {electrolyzer_utilization_hours:.2f} %")
    print(f"Levelized Elec cost: ${levelized_Elec_cost:.2f} per kWh")
    print(f"Levelized H2 cost: ${levelized_H2_cost:.2f} per kgH2")
    print(f"Levelized hydrogen production cost: ${model.objVal / total_hydrogen_production:.2f} per kg")
else:
    print("No feasible solution found.")


In [ ]:
if model.status == GRB.OPTIMAL:
    print("Optimal system configuration:")
    print(f"Wind capacity: {wind_capacity.x:.2f} kW")
    print(f"PV capacity: {pv_capacity.x:.2f} kW")
    print(f"Battery capacity: {battery_capacity.x:.2f} kW")
    print(f"Electrolyzer capacity_AE: {electrolyzer_capacity_AE.x:.2f} kW")
    print(f"Electrolyzer capacity_PEM: {electrolyzer_capacity_PEM.x:.2f} kW")
    print(f"Hydrogen storage capacity: {hydrogen_storage_capacity.x:.2f} kgH2/h")
    print(f"Total hydrogen production: {total_hydrogen_production:.2f} kg")
    print(f"Electrolyzer utilization hours: {electrolyzer_utilization_hours:.2f} %")
    print(f"Levelized Elec cost: ${levelized_Elec_cost:.2f} per kWh")
    print(f"Levelized H2 cost: ${levelized_H2_cost:.2f} per kgH2")
    print(f"Levelized hydrogen production cost: ${model.objVal / total_hydrogen_production:.2f} per kg")

    # 计算各时刻功率和储能曲线
    wind_power_profile = wind_output * wind_capacity.x
    pv_power_profile = pv_output * pv_capacity.x
    electrolyzer_demand_AE = [electrolyzer_power_AE[t].x * electrolyzer_eff for t in range(time_steps)]
    electrolyzer_demand_PEM = [electrolyzer_power_PEM[t].x * electrolyzer_eff for t in range(time_steps)]
    hydrogen_storage_profile = [hydrogen_storage_balance[t].x for t in range(time_steps)]
    
    # 获取各时刻电力平衡约束和氢气小时供需平衡约束的对偶价格
    electricity_prices = [model.getConstrByName(f"power_balance_{t}").pi for t in range(time_steps)]
    hydrogen_prices = []
    for t in range(time_steps):
        constr = model.getConstrByName(f"hydrogen_balance_{t}")
        if constr is not None:
            hydrogen_prices.append(constr.pi)
        else:
            hydrogen_prices.append(0)

    # 绘制一个样本周（168小时）的时序图
    week_start = 168 * 10   # 例如，从第11周开始
    week_end = week_start + 168

    battery_charge_profile_week = [battery_charge[t].x for t in range(week_start, week_end)]
    battery_discharge_profile_week = [battery_discharge[t].x for t in range(week_start, week_end)]
    energy_balance_profile_week = [energy_balance[t].x for t in range(week_start, week_end)]
    wind_power_profile_week = wind_power_profile[week_start:week_end]
    pv_power_profile_week = pv_power_profile[week_start:week_end]
    electrolyzer_demand_week_AE = [electrolyzer_demand_AE[t] for t in range(week_start, week_end)]
    electrolyzer_demand_week_PEM = [electrolyzer_demand_PEM[t] for t in range(week_start, week_end)]
    hydrogen_storage_week = [hydrogen_storage_profile[t] for t in range(week_start, week_end)]
    
    electricity_prices_week = electricity_prices[week_start:week_end]
    hydrogen_prices_week = hydrogen_prices[week_start:week_end]

    plt.figure(figsize=(15, 8))
    plt.plot(wind_power_profile_week, label='Wind Output (kW)', linestyle='-', color='blue')
    plt.plot(pv_power_profile_week, label='PV Output (kW)', linestyle='-', color='orange')
    plt.plot(battery_charge_profile_week, label='Battery Charge (kW)', linestyle='--', color='green')
    plt.plot(battery_discharge_profile_week, label='Battery Discharge (kW)', linestyle='--', color='red')
    plt.plot(electrolyzer_demand_week_AE, label='Electrolyzer Demand AE(kW)', linestyle='-', color='purple')
    plt.plot(electrolyzer_demand_week_PEM, label='Electrolyzer Demand PEM(kW)', linestyle='-', color='red')
    plt.plot(hydrogen_storage_week, label='Hydrogen Storage Balance (kg)', linestyle='-', color='cyan')
    plt.plot(electricity_prices_week, label='Electricity Price ($/kWh)', linestyle='-', color='blue')
    plt.plot(hydrogen_prices_week, label='Hydrogen Price ($/kg)', linestyle='-', color='green')

    plt.xlabel('Hour')
    plt.ylabel('Value')
    plt.title('Power Output and Demand Profiles (Sample Week)')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 按月绘图
    hours_in_month = [0, 744, 1416, 2160, 2880, 3624, 4344, 5088, 5832, 6552, 7296, 8016, 8760]
    month_names = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

    for i in range(12):
        start = hours_in_month[i]
        end = hours_in_month[i + 1]

        battery_charge_profile = [battery_charge[t].x for t in range(start, end)]
        battery_discharge_profile = [battery_discharge[t].x for t in range(start, end)]
        energy_balance_profile = [energy_balance[t].x for t in range(start, end)]

        plt.figure(figsize=(15, 8))
        plt.plot(wind_power_profile[start:end], label='Wind Output (kW)', linestyle='-', color='blue')
        plt.plot(pv_power_profile[start:end], label='PV Output (kW)', linestyle='-', color='orange')
        plt.plot(battery_charge_profile, label='Battery Charge (kW)', linestyle='--', color='green')
        plt.plot(battery_discharge_profile, label='Battery Discharge (kW)', linestyle='--', color='red')
        plt.plot(electrolyzer_demand_AE[start:end], label='Electrolyzer Demand AE (kW)', linestyle='-', color='purple')
        plt.plot(electrolyzer_demand_PEM[start:end], label='Electrolyzer Demand PEM (kW)', linestyle='-', color='purple')
        plt.plot(hydrogen_storage_profile[start:end], label='Hydrogen Storage Balance (kg)', linestyle='-', color='cyan')
        plt.plot(electricity_prices[start:end], label='Electricity Price ($/kWh)', linestyle='-', color='blue')
        plt.plot(hydrogen_prices[start:end], label='Hydrogen Price ($/kg)', linestyle='-', color='green')

        plt.xlabel('Hour')
        plt.ylabel('Value')
        plt.title(f'Power Output and Demand Profiles ({month_names[i]})')
        plt.legend()
        plt.grid(True)
        plt.show()
else:
    print("No feasible solution found.")


# 4. 模型扩展与工具函数

## 4.1 概述
本节包含模型的扩展功能、工具函数和辅助代码，为主要优化模型提供支持。

### 主要内容
- **参数敏感性分析**：关键参数对系统成本的影响
- **多情景对比**：不同技术路线的经济性比较  
- **数据处理工具**：结果提取和格式化函数
- **可视化工具**：标准化图表生成函数
- **验证工具**：模型结果合理性检查

### 功能特点
- 模块化设计，便于复用
- 标准化接口，确保一致性
- 完整的文档注释
- 错误处理机制

### 使用说明
这些工具函数可以单独调用，也可以与主要优化模型组合使用，提供更丰富的分析功能。

In [ ]:
if model.status == GRB.OPTIMAL:
    # ---------------------------
    # 样本周（168小时）堆积柱状图
    # ---------------------------
    week_start = 168 * 10   # 例如取第11周
    week_end = week_start + 168
    x = np.arange(week_end - week_start)

    # 供给侧：风电、光伏、以及电池放电（均为正值）
    supply_wind = np.array([wind_output[t] * wind_capacity.x for t in range(week_start, week_end)])
    supply_pv = np.array([pv_output[t] * pv_capacity.x for t in range(week_start, week_end)])
    supply_battery = np.array([battery_discharge[t].x for t in range(week_start, week_end)])
    
    # 需求侧：电解槽、电池充电，以及氢气充电（乘以充氢所需电耗），显示为负值
    demand_electrolyzer_AE = np.array([electrolyzer_power_AE[t].x for t in range(week_start, week_end)])
    demand_electrolyzer_PEM = np.array([electrolyzer_power_PEM[t].x for t in range(week_start, week_end)])
    demand_battery = np.array([battery_charge[t].x for t in range(week_start, week_end)])
    demand_hydrogen = np.array([hydrogen_charge[t].x * hydrogen_storage_elec for t in range(week_start, week_end)])
    demand_surplus = np.array([surplus[t].x  for t in range(week_start, week_end)])
    
    plt.figure(figsize=(15, 8))
    # 绘制供给堆积柱状图（正值）
    bar1 = plt.bar(x, supply_wind, label='Wind Supply', color='blue')
    bar2 = plt.bar(x, supply_pv, bottom=supply_wind, label='PV Supply', color='orange')
    bar3 = plt.bar(x, supply_battery, bottom=supply_wind + supply_pv, label='Battery Discharge Supply', color='cyan')
    
    # 绘制需求堆积柱状图（负值）
    bar4 = plt.bar(x, -demand_electrolyzer_AE, label='Electrolyzer Demand AE', color='purple')
    bar5 = plt.bar(x, -demand_electrolyzer_PEM, bottom=-demand_electrolyzer_AE, label='Electrolyzer Demand PEM', color='yellow')
    bar6 = plt.bar(x, -demand_battery, bottom=-(demand_electrolyzer_AE+demand_electrolyzer_PEM), label='Battery Charge Demand', color='red')
    bar7 = plt.bar(x, -demand_hydrogen, bottom=-(demand_electrolyzer_AE+demand_electrolyzer_PEM + demand_battery), label='Hydrogen Charge Demand', color='green')
    bar8 = plt.bar(x, -demand_surplus, bottom=-(demand_electrolyzer_AE+demand_electrolyzer_PEM + demand_battery + demand_hydrogen), label='Surplus', color='brown')

    plt.axhline(0, color='black', linewidth=0.8)
    plt.xlabel('Hour')
    plt.ylabel('Power (kW)')
    plt.title('Stacked Power Supply (Positive) & Demand (Negative) - Sample Week')
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.show()

    # ---------------------------
    # 按月堆积柱状图
    # ---------------------------
    # 近似每月小时数划分（可根据实际情况调整）
    hours_in_month = [0, 744, 1416, 2160, 2880, 3624, 4344, 5088, 5832, 6552, 7296, 8016, 8760]
    month_names = ['January', 'February', 'March', 'April', 'May', 'June', 
                   'July', 'August', 'September', 'October', 'November', 'December']

    for i in range(1):
        start = hours_in_month[i]
        end = hours_in_month[i + 1]
        x_month = np.arange(end - start)
        
        # 供给侧数据
        supply_wind_month = np.array([wind_output[t] * wind_capacity.x for t in range(start, end)])
        supply_pv_month = np.array([pv_output[t] * pv_capacity.x for t in range(start, end)])
        supply_battery_month = np.array([battery_discharge[t].x for t in range(start, end)])
        
        # 需求侧数据
        demand_electrolyzer_month_AE = np.array([electrolyzer_power_AE[t].x for t in range(start, end)])
        demand_electrolyzer_month_PEM = np.array([electrolyzer_power_PEM[t].x for t in range(start, end)])
        demand_battery_month = np.array([battery_charge[t].x for t in range(start, end)])
        demand_hydrogen_month = np.array([hydrogen_charge[t].x * hydrogen_storage_elec for t in range(start, end)])
        demand_surplus_month = np.array([surplus[t].x  for t in range(start, end)])
        
        plt.figure(figsize=(15, 8))
        plt.bar(x_month, supply_wind_month, label='Wind Supply', color='blue')
        plt.bar(x_month, supply_pv_month, bottom=supply_wind_month, label='PV Supply', color='orange')
        plt.bar(x_month, supply_battery_month, bottom=supply_wind_month + supply_pv_month, label='Battery Discharge Supply', color='cyan')
        
        plt.bar(x_month, -demand_electrolyzer_month_AE, label='Electrolyzer Demand AE', color='purple')
        plt.bar(x_month, -demand_electrolyzer_month_PEM, bottom=-demand_electrolyzer_month_AE, label='Electrolyzer Demand PEM', color='yellow')
        plt.bar(x_month, -demand_battery_month, bottom=-(demand_electrolyzer_month_AE + demand_electrolyzer_month_PEM), label='Battery Charge Demand', color='red')
        plt.bar(x_month, -demand_hydrogen_month, bottom=-(demand_electrolyzer_month_AE + demand_electrolyzer_month_PEM + demand_battery_month), label='Hydrogen Charge Demand', color='green')
        plt.bar(x_month, -demand_surplus_month, bottom=-(demand_electrolyzer_month_AE + demand_electrolyzer_month_PEM + demand_battery_month + demand_hydrogen_month), label='Surplus', color='brown')
        
        plt.axhline(0, color='black', linewidth=0.8)
        plt.xlabel('Hour')
        plt.ylabel('Power (kW)')
        plt.title(f'Stacked Power Supply & Demand ({month_names[i]})')
        plt.legend(loc='upper right')
        plt.grid(True)
        plt.show()
else:
    print("No feasible solution found.")


In [ ]:
if model.status == GRB.OPTIMAL:
    # ---------------------------
    # 样本周（168小时）氢能供需堆积柱状图
    # ---------------------------
    week_start = 168 * 10   # 例如取第11周
    week_end = week_start + 168
    x = np.arange(week_end - week_start)
    
    # 供给侧：
    # 1. 氢气生产：由AE和PEM电解槽产生的氢气（单位：kg/h）
    supply_h2_production = np.array([
        (electrolyzer_power_AE[t].x + electrolyzer_power_PEM[t].x) * conversion_factor
        for t in range(week_start, week_end)
    ])
    # 2. 氢气放电：氢气储存释放的氢气（单位：kg/h）
    supply_h2_discharge = np.array([hydrogen_discharge[t].x for t in range(week_start, week_end)])
    
    plt.figure(figsize=(15, 8))
    # 绘制供给侧堆积柱状图（正值）
    bar1 = plt.bar(x, supply_h2_production, label='Hydrogen Production', color='blue')
    bar2 = plt.bar(x, supply_h2_discharge, bottom=supply_h2_production, label='Hydrogen Discharge', color='cyan')
    
    
    plt.axhline(0, color='black', linewidth=0.8)
    plt.xlabel('Hour')
    plt.ylabel('Hydrogen Flow (kg/h)')
    plt.title('Stacked Hydrogen Supply (Positive) & Demand (Negative) - Sample Week')
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.show()
    
    # ---------------------------
    # 按月氢能供需堆积柱状图
    # ---------------------------
    # 近似每月的小时数划分（可根据实际情况调整）
    hours_in_month = [0, 744, 1416, 2160, 2880, 3624, 4344, 5088, 5832, 6552, 7296, 8016, 8760]
    month_names = ['January', 'February', 'March', 'April', 'May', 'June', 
                   'July', 'August', 'September', 'October', 'November', 'December']
    
    for i in range(1):
        start = hours_in_month[i]
        end = hours_in_month[i+1]
        x_month = np.arange(end - start)
        
        # 供给侧数据
        supply_h2_production_month = np.array([
            (electrolyzer_power_AE[t].x + electrolyzer_power_PEM[t].x) * conversion_factor
            for t in range(start, end)
        ])
        supply_h2_discharge_month = np.array([hydrogen_discharge[t].x for t in range(start, end)])
        
        
        plt.figure(figsize=(15, 8))
        plt.bar(x_month, supply_h2_production_month, label='Hydrogen Production', color='blue')
        plt.bar(x_month, supply_h2_discharge_month, bottom=supply_h2_production_month, label='Hydrogen Discharge', color='cyan')
        
        plt.axhline(0, color='black', linewidth=0.8)
        plt.xlabel('Hour')
        plt.ylabel('Hydrogen Flow (kg/h)')
        plt.title(f'Stacked Hydrogen Supply & Demand ({month_names[i]})')
        plt.legend(loc='upper right')
        plt.grid(True)
        plt.show()
else:
    print("No feasible solution found.")


In [ ]:
demand_h2_base